## 1. Downloading and Automatically Cleaning Train/Validation Images

This section downloads and automatically cleans the Cat, Dog, Bird, and Unknown images intended for the training and validation pools.

Raw source archives and extracted datasets are stored under:

`Model Building/Datasets/Original/Train and Validation`

Automatically cleaned images are stored under:

`Model Building/Datasets/Auto Cleaned/Train and Validation`

Automatically rejected images are copied into source- and reason-based folders under:

`Model Building/Datasets/Rejected/Automatic/Train and Validation`

The automatic-cleaning process rejects broken or unreadable files, invalid image dimensions, images with a width or height below 64 pixels, completely blank images, exact duplicates, and images that cannot be processed or copied correctly.

Accepted images are copied unchanged while preserving their original filenames, extensions, and relative source-folder structure. Content-based decisions are left for the later manual-cleaning stage.


### 1.1 Downloading and Automatically Cleaning Cat Train/Validation Images

This section downloads and automatically cleans the Cat images intended for the training and validation pool.

Raw Cat train/validation source archives and extracted datasets are stored in source-based folders under:

`Model Building/Datasets/Original/Train and Validation`

Automatically cleaned Cat train/validation images are stored in source-based folders under:

`Model Building/Datasets/Auto Cleaned/Train and Validation/Cat`

Automatically rejected Cat images are copied into source- and reason-based folders under:

`Model Building/Datasets/Rejected/Automatic/Train and Validation/Cat`

Cat is the main constraint because there is less available Cat data than Dog, Bird, and Unknown data. Dog and Bird have larger source pools, while Unknown can be built from many different non-target datasets. Because of this, the project first downloads, automatically cleans, and counts the Cat training and validation sources before the final dataset size is confirmed. The cleaned Cat count is then used to guide the matching Dog and Bird counts, while Unknown remains larger because it represents a wider range of non-target images.

The Cat train/validation sources are:

```text
Kaggle Cats and Dogs Dataset:
https://download.microsoft.com/download/3/e/1/3e1c3f21-ecdb-4869-8368-6deba77b919f/kagglecatsanddogs_5340.zip

The Oxford-IIIT Pet Dataset:
https://thor.robots.ox.ac.uk/~vgg/data/pets/images.tar.gz
https://thor.robots.ox.ac.uk/~vgg/data/pets/annotations.tar.gz

Animal Faces-HQ Dataset (AFHQ-v2):
https://www.dropbox.com/s/vkzjokiwof5h8w6/afhq_v2.zip?dl=1

CAT Dataset:
https://archive.org/download/CAT_DATASET/CAT_DATASET_01.zip
https://archive.org/download/CAT_DATASET/CAT_DATASET_02.zip
```

These datasets were chosen because they provide Cat images from different dataset styles, including general pet images, breed-based pet images, animal face images, and a dedicated Cat dataset. Using multiple sources improves image diversity and makes the Cat class less dependent on one dataset style.

The same universal automatic-cleaning rules are applied across all Cat sources.


In [1]:
from pathlib import Path
import subprocess
import sys
import urllib.request
import zipfile
import tarfile
import shutil
import hashlib
from collections import defaultdict

PROJECT_NAME = "Image Classifier Project"
RESET_OUTPUT_FOLDERS = True

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
}

MIN_IMAGE_WIDTH = 64
MIN_IMAGE_HEIGHT = 64


def install_package(package_name):
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", package_name],
        check=True,
    )


install_package("pillow")
install_package("tqdm")

from PIL import Image, ImageOps
from tqdm.auto import tqdm


def find_project_root(start_path):
    current_path = Path(start_path).resolve()

    # Search upward from the notebook directory.
    for path in [current_path, *current_path.parents]:
        has_project_structure = (
            (path / "README.md").is_file()
            and (path / "Model Building").is_dir()
        )

        if path.name == PROJECT_NAME or has_project_structure:
            return path

    raise FileNotFoundError(
        f"Project folder not found: {PROJECT_NAME}"
    )


def show_path(path):
    # Hide the full computer path in notebook outputs.
    path = Path(path).resolve()

    return (
        Path(PROJECT_ROOT.name)
        / path.relative_to(PROJECT_ROOT)
    )


def reset_folder(folder):
    # Only reset generated outputs, not raw downloads.
    if folder.exists():
        shutil.rmtree(folder)

    folder.mkdir(parents=True, exist_ok=True)


def download_file(url, output_path):
    output_path.parent.mkdir(parents=True, exist_ok=True)

    # Skip archives that were already downloaded.
    if output_path.exists() and output_path.stat().st_size > 0:
        print(f"Already downloaded: {output_path.name}")
        return

    if output_path.exists():
        output_path.unlink()

    print(f"Downloading: {output_path.name}")

    progress_bar = None

    def update_progress(block_number, block_size, total_size):
        nonlocal progress_bar

        if progress_bar is None:
            progress_bar = tqdm(
                total=total_size if total_size > 0 else None,
                unit="B",
                unit_scale=True,
                desc=output_path.name,
            )

        downloaded = block_number * block_size

        progress_bar.update(
            max(0, downloaded - progress_bar.n)
        )

    try:
        urllib.request.urlretrieve(
            url,
            output_path,
            reporthook=update_progress,
        )

    except Exception:
        if output_path.exists():
            output_path.unlink()

        raise

    finally:
        if progress_bar is not None:
            progress_bar.close()

    print(f"Saved: {show_path(output_path)}")


def extract_zip(zip_path, output_dir):
    marker_path = output_dir / f".{zip_path.name}_extracted"

    # Avoid extracting the same archive more than once.
    if marker_path.exists():
        print(f"Already extracted: {zip_path.name}")
        return

    if not zipfile.is_zipfile(zip_path):
        print(
            "Invalid ZIP file found and removed: "
            f"{show_path(zip_path)}"
        )

        zip_path.unlink()

        raise zipfile.BadZipFile(
            "Invalid ZIP file removed. Rerun the cell to "
            f"download it again: {zip_path.name}"
        )

    output_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(output_dir)

    marker_path.write_text(
        "extracted",
        encoding="utf-8",
    )

    print(f"Extracted: {zip_path.name}")


def extract_tar(tar_path, output_dir):
    marker_path = output_dir / f".{tar_path.name}_extracted"

    # Avoid extracting the same archive more than once.
    if marker_path.exists():
        print(f"Already extracted: {tar_path.name}")
        return

    if not tarfile.is_tarfile(tar_path):
        print(
            "Invalid TAR file found and removed: "
            f"{show_path(tar_path)}"
        )

        tar_path.unlink()

        raise tarfile.TarError(
            "Invalid TAR file removed. Rerun the cell to "
            f"download it again: {tar_path.name}"
        )

    output_dir.mkdir(parents=True, exist_ok=True)

    with tarfile.open(tar_path, "r:*") as tar_ref:
        try:
            tar_ref.extractall(
                output_dir,
                filter="data",
            )

        except TypeError:
            tar_ref.extractall(output_dir)

    marker_path.write_text(
        "extracted",
        encoding="utf-8",
    )

    print(f"Extracted: {tar_path.name}")


def get_image_files(folder):
    if not folder.exists():
        return []

    return sorted(
        path
        for path in folder.rglob("*")
        if (
            path.is_file()
            and path.suffix.lower() in IMAGE_EXTENSIONS
        )
    )


def get_microsoft_cat_images(source_dir):
    # Microsoft Cats and Dogs stores Cat images in Cat folders.
    return [
        path
        for path in get_image_files(source_dir)
        if any(
            part.lower() == "cat"
            for part in path.parts
        )
    ]


def get_oxford_cat_images(source_dir):
    images_dir = source_dir / "images"

    # Oxford Cat breed filenames begin with uppercase letters.
    return [
        path
        for path in get_image_files(images_dir)
        if path.name and path.name[0].isupper()
    ]


def get_afhq_cat_images(source_dir):
    # AFHQ-v2 stores Cat images in Cat folders.
    return [
        path
        for path in get_image_files(source_dir)
        if any(
            part.lower() == "cat"
            for part in path.parts
        )
    ]


def get_cat_dataset_images(source_dir):
    # CAT Dataset is already Cat-focused.
    return get_image_files(source_dir)


def check_image(
    image_path,
    min_width=MIN_IMAGE_WIDTH,
    min_height=MIN_IMAGE_HEIGHT,
):
    try:
        with Image.open(image_path) as image:
            image.verify()

        # Reopen after verify() before reading image data.
        with Image.open(image_path) as image:
            image = ImageOps.exif_transpose(image)

            width, height = image.size

            if width <= 0 or height <= 0:
                return False, "invalid_dimensions"

            if width < min_width or height < min_height:
                return False, "low_resolution"

            rgb_image = image.convert("RGB")
            channel_extrema = rgb_image.getextrema()

            if all(
                minimum_value == maximum_value
                for minimum_value, maximum_value in channel_extrema
            ):
                return False, "blank_image"

        return True, "usable"

    except Exception:
        return False, "broken_or_unreadable"


# Hash decoded pixels so metadata differences do not hide duplicates.
def get_exact_image_hash(image_path):
    with Image.open(image_path) as image:
        image = ImageOps.exif_transpose(image)
        image = image.convert("RGB")

        width, height = image.size

        image_hasher = hashlib.sha256()

        image_hasher.update(
            width.to_bytes(
                4,
                byteorder="big",
                signed=False,
            )
        )

        image_hasher.update(
            height.to_bytes(
                4,
                byteorder="big",
                signed=False,
            )
        )

        image_hasher.update(image.tobytes())

        return image_hasher.hexdigest()


def copy_preserving_structure(
    source_path,
    source_root,
    output_root,
):
    relative_path = source_path.relative_to(source_root)
    output_path = output_root / relative_path

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    shutil.copy2(
        source_path,
        output_path,
    )

    return output_path


def copy_rejected_image(
    source_path,
    source_root,
    output_root,
):
    try:
        return copy_preserving_structure(
            source_path,
            source_root,
            output_root,
        )

    except Exception:
        return None


def copy_duplicate_group(
    duplicate_path,
    duplicate_source_root,
    duplicate_source_name,
    kept_record,
    group_dir,
):
    # Save the kept image beside the rejected duplicate for inspection.
    kept_output_root = (
        group_dir
        / "kept"
        / kept_record["source_name"]
    )

    duplicate_output_root = (
        group_dir
        / "duplicate"
        / duplicate_source_name
    )

    kept_marker = group_dir / ".kept_copied"

    if not kept_marker.exists():
        copy_rejected_image(
            kept_record["original_path"],
            kept_record["source_root"],
            kept_output_root,
        )

        kept_marker.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        kept_marker.write_text(
            "copied",
            encoding="utf-8",
        )

    copy_rejected_image(
        duplicate_path,
        duplicate_source_root,
        duplicate_output_root,
    )


PROJECT_ROOT = find_project_root(Path.cwd())

datasets_dir = (
    PROJECT_ROOT
    / "Model Building"
    / "Datasets"
)

original_train_val_dir = (
    datasets_dir
    / "Original"
    / "Train and Validation"
)

cat_auto_cleaned_dir = (
    datasets_dir
    / "Auto Cleaned"
    / "Train and Validation"
    / "Cat"
)

cat_rejected_dir = (
    datasets_dir
    / "Rejected"
    / "Automatic"
    / "Train and Validation"
    / "Cat"
)


cat_sources = {
    "microsoft_cats_and_dogs": {
        "folder_name": "Microsoft Cats and Dogs",
        "selector": get_microsoft_cat_images,
        "downloads": [
            (
                "https://download.microsoft.com/download/"
                "3/e/1/3e1c3f21-ecdb-4869-8368-6deba77b919f/"
                "kagglecatsanddogs_5340.zip",
                "kagglecatsanddogs_5340.zip",
            ),
        ],
    },

    "oxford_iiit_pet": {
        "folder_name": "Oxford-IIIT Pet",
        "selector": get_oxford_cat_images,
        "downloads": [
            (
                "https://thor.robots.ox.ac.uk/"
                "~vgg/data/pets/images.tar.gz",
                "images.tar.gz",
            ),
            (
                "https://thor.robots.ox.ac.uk/"
                "~vgg/data/pets/annotations.tar.gz",
                "annotations.tar.gz",
            ),
        ],
    },

    "afhq_v2": {
        "folder_name": "AFHQ-v2",
        "output_folder_name": "AFHQ-v2 Cat",
        "selector": get_afhq_cat_images,
        "downloads": [
            (
                "https://www.dropbox.com/s/"
                "vkzjokiwof5h8w6/afhq_v2.zip?dl=1",
                "afhq_v2.zip",
            ),
        ],
    },

    "cat_dataset": {
        "folder_name": "CAT Dataset",
        "selector": get_cat_dataset_images,
        "downloads": [
            (
                "https://archive.org/download/"
                "CAT_DATASET/CAT_DATASET_01.zip",
                "CAT_DATASET_01.zip",
            ),
            (
                "https://archive.org/download/"
                "CAT_DATASET/CAT_DATASET_02.zip",
                "CAT_DATASET_02.zip",
            ),
        ],
    },
}


for source in cat_sources.values():
    folder_name = source["folder_name"]

    output_folder_name = source.get(
        "output_folder_name",
        folder_name,
    )

    source["original_dir"] = (
        original_train_val_dir
        / folder_name
    )

    source["cleaned_dir"] = (
        cat_auto_cleaned_dir
        / output_folder_name
    )

    source["rejected_dir"] = (
        cat_rejected_dir
        / output_folder_name
    )


if RESET_OUTPUT_FOLDERS:
    reset_folder(cat_auto_cleaned_dir)
    reset_folder(cat_rejected_dir)


for folder in [
    original_train_val_dir,
    cat_auto_cleaned_dir,
    cat_rejected_dir,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )


for source in cat_sources.values():
    source["original_dir"].mkdir(
        parents=True,
        exist_ok=True,
    )

    source["cleaned_dir"].mkdir(
        parents=True,
        exist_ok=True,
    )

    source["rejected_dir"].mkdir(
        parents=True,
        exist_ok=True,
    )


print("Cat train/validation paths are ready.")

print(
    "Original dataset directory:       "
    f"{show_path(original_train_val_dir)}"
)

print(
    "Auto Cleaned Cat directory:       "
    f"{show_path(cat_auto_cleaned_dir)}"
)

print(
    "Automatically rejected directory: "
    f"{show_path(cat_rejected_dir)}"
)

print()


print("Downloading Cat train/validation source archives...")

for source in cat_sources.values():
    source_dir = source["original_dir"]

    for url, file_name in source["downloads"]:
        download_file(
            url,
            source_dir / file_name,
        )

print()
print("Cat train/validation source downloads are complete.")
print()


print("Extracting Cat train/validation source archives...")

for source in cat_sources.values():
    source_dir = source["original_dir"]
    display_name = source["folder_name"]

    print(f"Checking archives in {display_name}...")

    for zip_path in sorted(
        source_dir.rglob("*.zip")
    ):
        extract_zip(
            zip_path,
            zip_path.parent,
        )

    tar_archives = sorted(
        {
            *source_dir.rglob("*.tar"),
            *source_dir.rglob("*.tar.gz"),
            *source_dir.rglob("*.tgz"),
        }
    )

    for tar_path in tar_archives:
        extract_tar(
            tar_path,
            tar_path.parent,
        )

print()
print("Archive extraction complete.")
print()


cat_source_image_paths = {}

for source_name, source in cat_sources.items():
    selector = source["selector"]
    source_dir = source["original_dir"]

    cat_source_image_paths[source_name] = selector(
        source_dir
    )


print("Candidate Cat images found:")

for source_name, image_paths in cat_source_image_paths.items():
    display_name = cat_sources[source_name].get(
        "output_folder_name",
        cat_sources[source_name]["folder_name"],
    )

    print(
        f"{display_name}: "
        f"{len(image_paths):,}"
    )

print()


# Share hashes across sources to catch cross-dataset duplicates.
seen_exact_hashes = {}

duplicate_group_numbers = {}

cleaned_counts = defaultdict(int)
rejected_counts = defaultdict(int)

rejection_reasons = defaultdict(
    lambda: defaultdict(int)
)


for source_name, image_paths in cat_source_image_paths.items():
    source = cat_sources[source_name]

    display_name = source.get(
        "output_folder_name",
        source["folder_name"],
    )

    source_root = source["original_dir"]
    cleaned_output_root = source["cleaned_dir"]
    rejected_output_root = source["rejected_dir"]

    print(f"Automatically cleaning {display_name}...")

    for image_path in tqdm(
        image_paths,
        desc=display_name,
        unit="image",
    ):
        is_valid, reason = check_image(image_path)

        if not is_valid:
            rejected_counts[source_name] += 1
            rejection_reasons[source_name][reason] += 1

            copy_rejected_image(
                image_path,
                source_root,
                rejected_output_root / reason,
            )

            continue

        try:
            exact_hash = get_exact_image_hash(
                image_path
            )

        except Exception:
            reason = "hash_failed"

            rejected_counts[source_name] += 1
            rejection_reasons[source_name][reason] += 1

            copy_rejected_image(
                image_path,
                source_root,
                rejected_output_root / reason,
            )

            continue

        if exact_hash in seen_exact_hashes:
            kept_record = seen_exact_hashes[exact_hash]

            rejected_counts[source_name] += 1
            rejection_reasons[source_name][
                "exact_duplicate"
            ] += 1

            if exact_hash not in duplicate_group_numbers:
                duplicate_group_numbers[exact_hash] = (
                    len(duplicate_group_numbers) + 1
                )

            group_number = duplicate_group_numbers[
                exact_hash
            ]

            group_dir = (
                rejected_output_root
                / "exact_duplicate"
                / f"duplicate_group_{group_number:06d}"
            )

            copy_duplicate_group(
                duplicate_path=image_path,
                duplicate_source_root=source_root,
                duplicate_source_name=display_name,
                kept_record=kept_record,
                group_dir=group_dir,
            )

            continue

        try:
            cleaned_path = copy_preserving_structure(
                image_path,
                source_root,
                cleaned_output_root,
            )

        except Exception:
            reason = "copy_failed"

            rejected_counts[source_name] += 1
            rejection_reasons[source_name][reason] += 1

            copy_rejected_image(
                image_path,
                source_root,
                rejected_output_root / reason,
            )

            continue

        seen_exact_hashes[exact_hash] = {
            "source_name": display_name,
            "source_root": source_root,
            "original_path": image_path,
            "cleaned_path": cleaned_path,
        }

        cleaned_counts[source_name] += 1


print()
print("Cat train/validation automatic cleaning is complete.")
print()


total_candidates = 0
total_cleaned = 0
total_rejected = 0


for source_name, image_paths in cat_source_image_paths.items():
    display_name = cat_sources[source_name].get(
        "output_folder_name",
        cat_sources[source_name]["folder_name"],
    )

    candidates = len(image_paths)
    cleaned = cleaned_counts[source_name]
    rejected = rejected_counts[source_name]

    total_candidates += candidates
    total_cleaned += cleaned
    total_rejected += rejected

    print(display_name)
    print(f"  Candidate images:       {candidates:,}")
    print(f"  Auto Cleaned images:    {cleaned:,}")
    print(f"  Automatically rejected: {rejected:,}")

    if rejection_reasons[source_name]:
        print("  Rejection reasons:")

        for reason, count in sorted(
            rejection_reasons[source_name].items()
        ):
            print(f"    {reason}: {count:,}")

    print()


print("Total")
print(f"  Candidate images:       {total_candidates:,}")
print(f"  Auto Cleaned images:    {total_cleaned:,}")
print(f"  Automatically rejected: {total_rejected:,}")
print()

print(
    "Auto Cleaned Cat images saved in: "
    f"{show_path(cat_auto_cleaned_dir)}"
)

print(
    "Automatically rejected Cat images saved in: "
    f"{show_path(cat_rejected_dir)}"
)

c:\Users\uushu\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cat train/validation paths are ready.
Original dataset directory:       Image Classifier Project\Model Building\Datasets\Original\Train and Validation
Auto Cleaned Cat directory:       Image Classifier Project\Model Building\Datasets\Auto Cleaned\Train and Validation\Cat
Automatically rejected directory: Image Classifier Project\Model Building\Datasets\Rejected\Automatic\Train and Validation\Cat

Already downloaded: kagglecatsanddogs_5340.zip
Already downloaded: images.tar.gz
Already downloaded: annotations.tar.gz
Already downloaded: afhq_v2.zip
Already downloaded: CAT_DATASET_01.zip
Already downloaded: CAT_DATASET_02.zip

Cat train/validation source downloads are complete.

Extracting Cat train/validation source archives...
Checking archives in Microsoft Cats and Dogs...
Already extracted: kagglecatsanddogs_5340.zip
Checking archives in Oxford-IIIT Pet...
Already extracted: annotations.tar.gz
Already extracted: images.tar.gz
Checking archives in AFHQ-v2...
Already extracted: afhq_v2.z

Microsoft Cats and Dogs: 100%|██████████| 12500/12500 [00:52<00:00, 238.61image/s]


Automatically cleaning Oxford-IIIT Pet...


Oxford-IIIT Pet: 100%|██████████| 2400/2400 [00:16<00:00, 144.76image/s]


Automatically cleaning AFHQ-v2 Cat...


AFHQ-v2 Cat: 100%|██████████| 5558/5558 [02:18<00:00, 40.18image/s]


Automatically cleaning CAT Dataset...


CAT Dataset: 100%|██████████| 9997/9997 [01:56<00:00, 85.96image/s] 


Cat train/validation automatic cleaning is complete.

Microsoft Cats and Dogs
  Candidate images:       12,500
  Auto Cleaned images:    12,465
  Automatically rejected: 35
  Rejection reasons:
    broken_or_unreadable: 1
    exact_duplicate: 17
    low_resolution: 17

Oxford-IIIT Pet
  Candidate images:       2,400
  Auto Cleaned images:    2,380
  Automatically rejected: 20
  Rejection reasons:
    exact_duplicate: 20

AFHQ-v2 Cat
  Candidate images:       5,558
  Auto Cleaned images:    5,558
  Automatically rejected: 0

CAT Dataset
  Candidate images:       9,997
  Auto Cleaned images:    9,936
  Automatically rejected: 61
  Rejection reasons:
    exact_duplicate: 61

Total
  Candidate images:       30,455
  Auto Cleaned images:    30,339
  Automatically rejected: 116

Auto Cleaned Cat images saved in: Image Classifier Project\Model Building\Datasets\Auto Cleaned\Train and Validation\Cat
Automatically rejected Cat images saved in: Image Classifier Project\Model Building\Datasets\R

### 1.2 Downloading and Automatically Cleaning Dog Train/Validation Images

This section downloads and automatically cleans the Dog images intended for the training and validation pool.

Raw Dog train/validation source archives and extracted datasets are stored in source-based folders under:

`Model Building/Datasets/Original/Train and Validation`

Automatically cleaned Dog train/validation images are stored in source-based folders under:

`Model Building/Datasets/Auto Cleaned/Train and Validation/Dog`

Automatically rejected Dog images are copied into source- and reason-based folders under:

`Model Building/Datasets/Rejected/Automatic/Train and Validation/Dog`

Dog has more available source data than Cat. The Dog sources are therefore processed in a controlled order, with Stanford Dogs used last because it contains the largest source pool.

The automatic-cleaning process stops once approximately **31,000 accepted Dog images** have been saved. This provides a small buffer for the later manual-cleaning stage. After manual review, the Cat, Dog, and Bird pools will be adjusted to the same final size.

The Dog train/validation sources are processed in this order:

```text
1. Kaggle Cats and Dogs Dataset
2. The Oxford-IIIT Pet Dataset
3. Animal Faces-HQ Dataset (AFHQ-v2)
4. Stanford Dogs Dataset
```

The Dog train/validation source links are:

```text
Kaggle Cats and Dogs Dataset:
https://download.microsoft.com/download/3/e/1/3e1c3f21-ecdb-4869-8368-6deba77b919f/kagglecatsanddogs_5340.zip

The Oxford-IIIT Pet Dataset:
https://thor.robots.ox.ac.uk/~vgg/data/pets/images.tar.gz
https://thor.robots.ox.ac.uk/~vgg/data/pets/annotations.tar.gz

Animal Faces-HQ Dataset (AFHQ-v2):
https://www.dropbox.com/s/vkzjokiwof5h8w6/afhq_v2.zip?dl=1

Stanford Dogs Dataset:
http://vision.stanford.edu/aditya86/ImageNetDogs/images.tar
http://vision.stanford.edu/aditya86/ImageNetDogs/annotation.tar
```

These datasets provide Dog images from different dataset styles, including general pet images, breed-based pet images, animal face images, and a large dedicated Dog dataset. Using multiple sources improves image diversity and makes the Dog class less dependent on one dataset style.

The same universal automatic-cleaning rules used for Cat are applied to the Dog sources.


In [2]:
import time


def download_file(url, output_path, max_retries=5):
    output_path.parent.mkdir(parents=True, exist_ok=True)

    # Skip archives that were already downloaded.
    if output_path.exists() and output_path.stat().st_size > 0:
        print(f"Already downloaded: {output_path.name}")
        return

    if output_path.exists():
        output_path.unlink()

    for attempt in range(1, max_retries + 1):
        print(
            f"Downloading: {output_path.name} "
            f"attempt {attempt}/{max_retries}"
        )

        progress_bar = None

        def update_progress(block_number, block_size, total_size):
            nonlocal progress_bar

            if progress_bar is None:
                progress_bar = tqdm(
                    total=total_size if total_size > 0 else None,
                    unit="B",
                    unit_scale=True,
                    desc=output_path.name,
                )

            downloaded = block_number * block_size

            progress_bar.update(
                max(0, downloaded - progress_bar.n)
            )

        try:
            urllib.request.urlretrieve(
                url,
                output_path,
                reporthook=update_progress,
            )

            if progress_bar is not None:
                progress_bar.close()

            print(f"Saved: {show_path(output_path)}")
            return

        except Exception as error:
            if progress_bar is not None:
                progress_bar.close()

            if output_path.exists():
                output_path.unlink()

            print(f"Download failed: {output_path.name}")
            print(f"Reason: {error}")

            if attempt == max_retries:
                raise

            print("Retrying...")
            time.sleep(3)


DOG_TRAIN_VAL_TARGET = 31_000

dog_auto_cleaned_dir = (
    datasets_dir
    / "Auto Cleaned"
    / "Train and Validation"
    / "Dog"
)

dog_rejected_dir = (
    datasets_dir
    / "Rejected"
    / "Automatic"
    / "Train and Validation"
    / "Dog"
)


def get_microsoft_dog_images(source_dir):
    # Microsoft Cats and Dogs stores Dog images in Dog folders.
    return [
        path
        for path in get_image_files(source_dir)
        if any(
            part.lower() == "dog"
            for part in path.parts
        )
    ]


def get_oxford_dog_images(source_dir):
    images_dir = source_dir / "images"

    # Oxford Dog breed filenames begin with lowercase letters.
    return [
        path
        for path in get_image_files(images_dir)
        if path.name and path.name[0].islower()
    ]


def get_afhq_dog_images(source_dir):
    # AFHQ-v2 stores Dog images in Dog folders.
    return [
        path
        for path in get_image_files(source_dir)
        if any(
            part.lower() == "dog"
            for part in path.parts
        )
    ]


def get_stanford_dog_images(source_dir):
    # Stanford Dogs is already Dog-focused.
    return get_image_files(source_dir)


dog_sources = {
    "microsoft_cats_and_dogs": {
        "folder_name": "Microsoft Cats and Dogs",
        "selector": get_microsoft_dog_images,
        "downloads": [
            (
                "https://download.microsoft.com/download/"
                "3/e/1/3e1c3f21-ecdb-4869-8368-6deba77b919f/"
                "kagglecatsanddogs_5340.zip",
                "kagglecatsanddogs_5340.zip",
            ),
        ],
    },

    "oxford_iiit_pet": {
        "folder_name": "Oxford-IIIT Pet",
        "selector": get_oxford_dog_images,
        "downloads": [
            (
                "https://thor.robots.ox.ac.uk/"
                "~vgg/data/pets/images.tar.gz",
                "images.tar.gz",
            ),
            (
                "https://thor.robots.ox.ac.uk/"
                "~vgg/data/pets/annotations.tar.gz",
                "annotations.tar.gz",
            ),
        ],
    },

    "afhq_v2": {
        "folder_name": "AFHQ-v2",
        "output_folder_name": "AFHQ-v2 Dog",
        "selector": get_afhq_dog_images,
        "downloads": [
            (
                "https://www.dropbox.com/s/"
                "vkzjokiwof5h8w6/afhq_v2.zip?dl=1",
                "afhq_v2.zip",
            ),
        ],
    },

    "stanford_dogs": {
        "folder_name": "Stanford Dogs",
        "selector": get_stanford_dog_images,
        "downloads": [
            (
                "http://vision.stanford.edu/"
                "aditya86/ImageNetDogs/images.tar",
                "images.tar",
            ),
            (
                "http://vision.stanford.edu/"
                "aditya86/ImageNetDogs/annotation.tar",
                "annotation.tar",
            ),
        ],
    },
}


for source in dog_sources.values():
    folder_name = source["folder_name"]

    output_folder_name = source.get(
        "output_folder_name",
        folder_name,
    )

    source["original_dir"] = (
        original_train_val_dir
        / folder_name
    )

    source["cleaned_dir"] = (
        dog_auto_cleaned_dir
        / output_folder_name
    )

    source["rejected_dir"] = (
        dog_rejected_dir
        / output_folder_name
    )


if RESET_OUTPUT_FOLDERS:
    reset_folder(dog_auto_cleaned_dir)
    reset_folder(dog_rejected_dir)


for folder in [
    original_train_val_dir,
    dog_auto_cleaned_dir,
    dog_rejected_dir,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )


for source in dog_sources.values():
    source["original_dir"].mkdir(
        parents=True,
        exist_ok=True,
    )

    source["cleaned_dir"].mkdir(
        parents=True,
        exist_ok=True,
    )

    source["rejected_dir"].mkdir(
        parents=True,
        exist_ok=True,
    )


print("Dog train/validation paths are ready.")

print(
    "Original dataset directory:       "
    f"{show_path(original_train_val_dir)}"
)

print(
    "Auto Cleaned Dog directory:       "
    f"{show_path(dog_auto_cleaned_dir)}"
)

print(
    "Automatically rejected directory: "
    f"{show_path(dog_rejected_dir)}"
)

print(
    "Dog train/validation target:      "
    f"{DOG_TRAIN_VAL_TARGET:,}"
)

print()


print("Downloading Dog train/validation source archives...")

for source in dog_sources.values():
    source_dir = source["original_dir"]

    for url, file_name in source["downloads"]:
        download_file(
            url,
            source_dir / file_name,
        )

print()
print("Dog train/validation source downloads are complete.")
print()


print("Extracting Dog train/validation source archives...")

for source in dog_sources.values():
    source_dir = source["original_dir"]
    display_name = source["folder_name"]

    print(f"Checking archives in {display_name}...")

    for zip_path in sorted(
        source_dir.rglob("*.zip")
    ):
        extract_zip(
            zip_path,
            zip_path.parent,
        )

    tar_paths = sorted(
        {
            *source_dir.rglob("*.tar"),
            *source_dir.rglob("*.tar.gz"),
            *source_dir.rglob("*.tgz"),
        }
    )

    for tar_path in tar_paths:
        extract_tar(
            tar_path,
            tar_path.parent,
        )

print()
print("Archive extraction complete.")
print()


dog_source_image_paths = {}

for source_name, source in dog_sources.items():
    selector = source["selector"]
    source_dir = source["original_dir"]

    dog_source_image_paths[source_name] = selector(
        source_dir
    )


print("Candidate Dog images found:")

for source_name, image_paths in dog_source_image_paths.items():
    display_name = dog_sources[source_name].get(
        "output_folder_name",
        dog_sources[source_name]["folder_name"],
    )

    print(
        f"{display_name}: "
        f"{len(image_paths):,}"
    )

print()


# Share hashes across sources to catch cross-dataset duplicates.
seen_exact_hashes = {}

duplicate_group_numbers = {}

processed_counts = defaultdict(int)
cleaned_counts = defaultdict(int)
rejected_counts = defaultdict(int)

rejection_reasons = defaultdict(
    lambda: defaultdict(int)
)

accepted_count = 0


for source_name, image_paths in dog_source_image_paths.items():
    if accepted_count >= DOG_TRAIN_VAL_TARGET:
        break

    source = dog_sources[source_name]

    display_name = source.get(
        "output_folder_name",
        source["folder_name"],
    )

    source_root = source["original_dir"]
    cleaned_output_root = source["cleaned_dir"]
    rejected_output_root = source["rejected_dir"]

    print(f"Automatically cleaning {display_name}...")

    for image_path in tqdm(
        image_paths,
        desc=display_name,
        unit="image",
    ):
        if accepted_count >= DOG_TRAIN_VAL_TARGET:
            break

        processed_counts[source_name] += 1

        is_valid, reason = check_image(image_path)

        if not is_valid:
            rejected_counts[source_name] += 1
            rejection_reasons[source_name][reason] += 1

            copy_rejected_image(
                image_path,
                source_root,
                rejected_output_root / reason,
            )

            continue

        try:
            exact_hash = get_exact_image_hash(
                image_path
            )

        except Exception:
            reason = "hash_failed"

            rejected_counts[source_name] += 1
            rejection_reasons[source_name][reason] += 1

            copy_rejected_image(
                image_path,
                source_root,
                rejected_output_root / reason,
            )

            continue

        if exact_hash in seen_exact_hashes:
            kept_record = seen_exact_hashes[exact_hash]

            rejected_counts[source_name] += 1
            rejection_reasons[source_name][
                "exact_duplicate"
            ] += 1

            if exact_hash not in duplicate_group_numbers:
                duplicate_group_numbers[exact_hash] = (
                    len(duplicate_group_numbers) + 1
                )

            group_number = duplicate_group_numbers[
                exact_hash
            ]

            group_dir = (
                rejected_output_root
                / "exact_duplicate"
                / f"duplicate_group_{group_number:06d}"
            )

            copy_duplicate_group(
                duplicate_path=image_path,
                duplicate_source_root=source_root,
                duplicate_source_name=display_name,
                kept_record=kept_record,
                group_dir=group_dir,
            )

            continue

        try:
            cleaned_path = copy_preserving_structure(
                image_path,
                source_root,
                cleaned_output_root,
            )

        except Exception:
            reason = "copy_failed"

            rejected_counts[source_name] += 1
            rejection_reasons[source_name][reason] += 1

            copy_rejected_image(
                image_path,
                source_root,
                rejected_output_root / reason,
            )

            continue

        seen_exact_hashes[exact_hash] = {
            "source_name": display_name,
            "source_root": source_root,
            "original_path": image_path,
            "cleaned_path": cleaned_path,
        }

        cleaned_counts[source_name] += 1
        accepted_count += 1


print()
print("Dog train/validation automatic cleaning is complete.")
print()


total_candidates = 0
total_processed = 0
total_cleaned = 0
total_rejected = 0


for source_name, image_paths in dog_source_image_paths.items():
    display_name = dog_sources[source_name].get(
        "output_folder_name",
        dog_sources[source_name]["folder_name"],
    )

    candidates = len(image_paths)
    processed = processed_counts[source_name]
    cleaned = cleaned_counts[source_name]
    rejected = rejected_counts[source_name]

    total_candidates += candidates
    total_processed += processed
    total_cleaned += cleaned
    total_rejected += rejected

    print(display_name)
    print(f"  Available candidates:   {candidates:,}")
    print(f"  Processed images:       {processed:,}")
    print(f"  Auto Cleaned images:    {cleaned:,}")
    print(f"  Automatically rejected: {rejected:,}")

    if rejection_reasons[source_name]:
        print("  Rejection reasons:")

        for reason, count in sorted(
            rejection_reasons[source_name].items()
        ):
            print(f"    {reason}: {count:,}")

    print()


print("Total")
print(f"  Available candidates:   {total_candidates:,}")
print(f"  Processed images:       {total_processed:,}")
print(f"  Auto Cleaned images:    {total_cleaned:,}")
print(f"  Automatically rejected: {total_rejected:,}")
print(f"  Target images:          {DOG_TRAIN_VAL_TARGET:,}")
print()

if total_cleaned >= DOG_TRAIN_VAL_TARGET:
    print("The Dog train/validation target was reached.")
else:
    print(
        "The Dog train/validation target was not reached. "
        f"Additional images needed: "
        f"{DOG_TRAIN_VAL_TARGET - total_cleaned:,}"
    )

print()

print(
    "Auto Cleaned Dog images saved in: "
    f"{show_path(dog_auto_cleaned_dir)}"
)

print(
    "Automatically rejected Dog images saved in: "
    f"{show_path(dog_rejected_dir)}"
)

Dog train/validation paths are ready.
Original dataset directory:       Image Classifier Project\Model Building\Datasets\Original\Train and Validation
Auto Cleaned Dog directory:       Image Classifier Project\Model Building\Datasets\Auto Cleaned\Train and Validation\Dog
Automatically rejected directory: Image Classifier Project\Model Building\Datasets\Rejected\Automatic\Train and Validation\Dog
Dog train/validation target:      31,000

Already downloaded: kagglecatsanddogs_5340.zip
Already downloaded: images.tar.gz
Already downloaded: annotations.tar.gz
Already downloaded: afhq_v2.zip
Already downloaded: images.tar
Already downloaded: annotation.tar

Dog train/validation source downloads are complete.

Extracting Dog train/validation source archives...
Checking archives in Microsoft Cats and Dogs...
Already extracted: kagglecatsanddogs_5340.zip
Checking archives in Oxford-IIIT Pet...
Already extracted: annotations.tar.gz
Already extracted: images.tar.gz
Checking archives in AFHQ-v2...

Microsoft Cats and Dogs:  91%|█████████▏| 11418/12500 [00:52<00:04, 222.73image/s]c:\Users\uushu\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\PIL\TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
Microsoft Cats and Dogs: 100%|██████████| 12500/12500 [00:57<00:00, 218.52image/s]


Automatically cleaning Oxford-IIIT Pet...


Oxford-IIIT Pet: 100%|██████████| 4990/4990 [00:30<00:00, 164.07image/s]


Automatically cleaning AFHQ-v2 Dog...


AFHQ-v2 Dog: 100%|██████████| 5169/5169 [02:04<00:00, 41.41image/s]


Automatically cleaning Stanford Dogs...


Stanford Dogs:  41%|████      | 8422/20580 [00:39<00:56, 214.49image/s]


Dog train/validation automatic cleaning is complete.

Microsoft Cats and Dogs
  Available candidates:   12,500
  Processed images:       12,500
  Auto Cleaned images:    12,462
  Automatically rejected: 38
  Rejection reasons:
    broken_or_unreadable: 1
    exact_duplicate: 11
    low_resolution: 26

Oxford-IIIT Pet
  Available candidates:   4,990
  Processed images:       4,990
  Auto Cleaned images:    4,980
  Automatically rejected: 10
  Rejection reasons:
    exact_duplicate: 10

AFHQ-v2 Dog
  Available candidates:   5,169
  Processed images:       5,169
  Auto Cleaned images:    5,169
  Automatically rejected: 0

Stanford Dogs
  Available candidates:   20,580
  Processed images:       8,422
  Auto Cleaned images:    8,389
  Automatically rejected: 33
  Rejection reasons:
    exact_duplicate: 33

Total
  Available candidates:   43,239
  Processed images:       31,081
  Auto Cleaned images:    31,000
  Automatically rejected: 81
  Target images:          31,000

The Dog train/vali

### 1.3 Downloading and Automatically Cleaning Bird Train/Validation Images

This section downloads and automatically cleans the Bird images intended for the training and validation pool.

Raw Bird train/validation source archives and extracted datasets are stored in source-based folders under:

`Model Building/Datasets/Original/Train and Validation`

Automatically cleaned Bird train/validation images are stored in source-based folders under:

`Model Building/Datasets/Auto Cleaned/Train and Validation/Bird`

Automatically rejected Bird images are copied into source- and reason-based folders under:

`Model Building/Datasets/Rejected/Automatic/Train and Validation/Bird`

CUB-200-2011 is used first because it is the smallest Bird source. All usable CUB-200-2011 images are retained. The remaining images needed to reach approximately **31,000 accepted Bird images** are then divided as evenly as possible between NABirds and Birdsnap.

This provides a small buffer for the later manual-cleaning stage while preventing either of the two larger Bird datasets from dominating the class. After manual review, the Cat, Dog, and Bird pools will be adjusted to the same final size.

The Bird train/validation sources are processed in this order:

```text
1. CUB-200-2011 Dataset
2. NABirds Dataset
3. Birdsnap Dataset
```

The planned source contribution is approximately:

```text
CUB-200-2011: All usable images
NABirds:      Half of the remaining accepted-image target
Birdsnap:     Half of the remaining accepted-image target
Total:        Approximately 31,000 accepted images
```

The Bird train/validation source links are:

```text
CUB-200-2011 Dataset:
https://www.vision.caltech.edu/datasets/cub_200_2011/
https://data.caltech.edu/records/65de6-vp158

NABirds Dataset:
https://dl.allaboutbirds.org/nabirds

Birdsnap Dataset:
https://huggingface.co/datasets/sasha/birdsnap
```

These datasets provide Bird images across many species, environments, poses, and dataset styles. Using multiple sources improves image diversity and makes the Bird class less dependent on one dataset.

The same universal automatic-cleaning rules used for Cat and Dog are applied to the Bird sources.


In [3]:
import hashlib
import io
import random
import re
import shutil
import subprocess
import tarfile
import time
from collections import defaultdict
from pathlib import Path
from urllib.parse import quote, unquote, urlparse

install_package("requests")
install_package("pyarrow")

import pyarrow.parquet as pq
import requests
from PIL import Image
from tqdm.auto import tqdm


BIRD_TRAIN_VAL_TARGET = 31_000
BIRDSNAP_CANDIDATE_TARGET = 12_400
DOWNLOAD_ATTEMPTS = 5
RANDOM_SEED = 42
RESET_OUTPUT_FOLDERS = True

CUB_URL = (
    "https://data.caltech.edu/records/65de6-vp158/files/"
    "CUB_200_2011.tgz?download=1"
)

CUB_EXPECTED_MD5 = "97eceeb196236b17998738112f37df78"

NABIRDS_URL = (
    "https://www.dropbox.com/s/nf78cbxq6bxpcfc/"
    "nabirds.tar.gz?dl=1"
)

NABIRDS_EXPECTED_SIZE = 9_884_285_075

BIRDSNAP_REPOSITORY = "sasha/birdsnap"


bird_auto_cleaned_dir = (
    datasets_dir
    / "Auto Cleaned"
    / "Train and Validation"
    / "Bird"
)

bird_rejected_dir = (
    datasets_dir
    / "Rejected"
    / "Automatic"
    / "Train and Validation"
    / "Bird"
)


bird_sources = {
    "cub_200_2011": {
        "folder_name": "CUB-200-2011",
    },
    "nabirds": {
        "folder_name": "NABirds",
    },
    "birdsnap": {
        "folder_name": "Birdsnap",
    },
}


for source in bird_sources.values():
    source["original_dir"] = (
        original_train_val_dir
        / source["folder_name"]
    )

    source["cleaned_dir"] = (
        bird_auto_cleaned_dir
        / source["folder_name"]
    )

    source["rejected_dir"] = (
        bird_rejected_dir
        / source["folder_name"]
    )

    source["original_dir"].mkdir(
        parents=True,
        exist_ok=True,
    )


cub_source = bird_sources["cub_200_2011"]
nabirds_source = bird_sources["nabirds"]
birdsnap_source = bird_sources["birdsnap"]


cub_archive_path = (
    cub_source["original_dir"]
    / "CUB_200_2011.tgz"
)

cub_images_dir = (
    cub_source["original_dir"]
    / "CUB_200_2011"
    / "images"
)


nabirds_archive_path = (
    nabirds_source["original_dir"]
    / "nabirds.tar.gz"
)


birdsnap_parquet_dir = (
    birdsnap_source["original_dir"]
    / "parquet"
)

birdsnap_images_dir = (
    birdsnap_source["original_dir"]
    / "images"
)


print("Bird train/validation paths are ready.")

print(
    "Original dataset directory:       "
    f"{show_path(original_train_val_dir)}"
)

print(
    "Auto Cleaned Bird directory:      "
    f"{show_path(bird_auto_cleaned_dir)}"
)

print(
    "Automatically rejected directory: "
    f"{show_path(bird_rejected_dir)}"
)

print(
    "Bird train/validation target:     "
    f"{BIRD_TRAIN_VAL_TARGET:,}"
)

print()


def file_md5(file_path):
    digest = hashlib.md5()

    with file_path.open("rb") as file_ref:
        for block in iter(
            lambda: file_ref.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def valid_cub_archive(file_path):
    return (
        file_path.exists()
        and file_md5(file_path)
        == CUB_EXPECTED_MD5
    )


def valid_nabirds_archive(file_path):
    return (
        file_path.exists()
        and file_path.stat().st_size
        == NABIRDS_EXPECTED_SIZE
    )


def valid_parquet(file_path):
    if not file_path.exists():
        return False

    try:
        parquet_file = pq.ParquetFile(
            str(file_path)
        )

        return (
            parquet_file.metadata.num_rows
            > 0
        )

    except Exception:
        return False


def download_with_curl(
    url,
    output_path,
    validator,
):
    if validator(output_path):
        print(
            f"Already downloaded: "
            f"{output_path.name}"
        )
        return

    curl_command = (
        shutil.which("curl.exe")
        or shutil.which("curl")
    )

    if curl_command is None:
        raise RuntimeError(
            "curl was not found on this computer."
        )

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    partial_path = output_path.with_name(
        output_path.name + ".part"
    )

    if validator(partial_path):
        partial_path.replace(output_path)

        print(
            "Recovered completed download: "
            f"{output_path.name}"
        )

        return

    if output_path.exists():
        invalid_path = output_path.with_name(
            output_path.name + ".invalid"
        )

        invalid_path.unlink(
            missing_ok=True
        )

        output_path.replace(
            invalid_path
        )

    print(
        f"Downloading: {output_path.name}"
    )

    command = [
        curl_command,
        "-L",
        "--fail",
        "--show-error",
        "--progress-bar",
        "--retry",
        str(DOWNLOAD_ATTEMPTS),
        "--retry-delay",
        "3",
        "--connect-timeout",
        "30",
        "--speed-time",
        "120",
        "--speed-limit",
        "1024",
        "-C",
        "-",
        "-o",
        str(partial_path),
        url,
    ]

    result = subprocess.run(
        command,
        check=False,
    )

    if result.returncode != 0:
        raise RuntimeError(
            f"Download failed: "
            f"{output_path.name}"
        )

    partial_path.replace(
        output_path
    )

    if not validator(output_path):
        invalid_path = output_path.with_name(
            output_path.name + ".invalid"
        )

        invalid_path.unlink(
            missing_ok=True
        )

        output_path.replace(
            invalid_path
        )

        raise RuntimeError(
            "Downloaded file failed validation: "
            f"{output_path.name}"
        )

    print(
        f"Downloaded: {output_path.name}"
    )


def extract_tar_if_needed(
    archive_path,
    output_dir,
    image_search_dir,
    minimum_images,
    description,
):
    existing_images = get_image_files(
        image_search_dir
    )

    if len(existing_images) >= minimum_images:
        print(
            f"Already extracted: {description}"
        )
        return

    print(
        f"Extracting {description}..."
    )

    with tarfile.open(
        archive_path,
        "r:*",
    ) as tar_ref:
        members = tar_ref.getmembers()

        for member in tqdm(
            members,
            desc=description,
            unit="file",
        ):
            tar_ref.extract(
                member,
                output_dir,
                filter="data",
            )

    extracted_images = get_image_files(
        image_search_dir
    )

    if len(extracted_images) < minimum_images:
        raise RuntimeError(
            f"{description} extraction "
            "appears incomplete."
        )

    print(
        f"Extracted: {description}"
    )


def prepare_cub():
    existing_images = get_image_files(
        cub_images_dir
    )

    if len(existing_images) >= 11_700:
        print(
            "CUB-200-2011 is already "
            "downloaded and extracted."
        )
        return

    download_with_curl(
        CUB_URL,
        cub_archive_path,
        valid_cub_archive,
    )

    extract_tar_if_needed(
        cub_archive_path,
        cub_source["original_dir"],
        cub_images_dir,
        11_700,
        "CUB-200-2011",
    )


def prepare_nabirds():
    existing_images = get_image_files(
        nabirds_source["original_dir"]
    )

    if len(existing_images) >= 48_000:
        print(
            "NABirds is already "
            "downloaded and extracted."
        )
        return

    download_with_curl(
        NABIRDS_URL,
        nabirds_archive_path,
        valid_nabirds_archive,
    )

    extract_tar_if_needed(
        nabirds_archive_path,
        nabirds_source["original_dir"],
        nabirds_source["original_dir"],
        48_000,
        "NABirds",
    )


def request_json(url):
    headers = {
        "User-Agent":
        "Image-Classifier-Project/1.0"
    }

    for attempt in range(
        1,
        DOWNLOAD_ATTEMPTS + 1,
    ):
        try:
            response = requests.get(
                url,
                headers=headers,
                timeout=(30, 120),
            )

            response.raise_for_status()

            return response.json()

        except Exception as error:
            if attempt == DOWNLOAD_ATTEMPTS:
                raise RuntimeError(
                    "Birdsnap file list could "
                    "not be retrieved."
                ) from error

            print(
                "Birdsnap file-list attempt "
                f"{attempt}/{DOWNLOAD_ATTEMPTS} "
                "failed. Retrying..."
            )

            time.sleep(5)


def get_birdsnap_parquet_files():
    api_url = (
        "https://huggingface.co/api/"
        "datasets/sasha/birdsnap/"
        "tree/main/data"
        "?recursive=false"
        "&expand=false"
        "&limit=1000"
    )

    records = request_json(
        api_url
    )

    parquet_paths = sorted(
        record.get("path", "")
        for record in records
        if record.get("path", "")
        .lower()
        .endswith(".parquet")
    )

    if not parquet_paths:
        raise RuntimeError(
            "No Birdsnap Parquet files "
            "were found."
        )

    return parquet_paths


def birdsnap_file_url(
    relative_path,
):
    encoded_path = quote(
        relative_path,
        safe="/",
    )

    return (
        "https://huggingface.co/datasets/"
        f"{BIRDSNAP_REPOSITORY}/"
        "resolve/main/"
        f"{encoded_path}?download=true"
    )


def download_birdsnap_subset():
    parquet_paths = (
        get_birdsnap_parquet_files()
    )

    random.Random(
        RANDOM_SEED
    ).shuffle(
        parquet_paths
    )

    selected_files = []
    candidate_rows = 0

    print(
        "Downloading the required "
        "Birdsnap subset..."
    )

    for index, relative_path in enumerate(
        parquet_paths,
        start=1,
    ):
        local_path = (
            birdsnap_parquet_dir
            / relative_path
        )

        print(
            "Birdsnap file "
            f"{index}/{len(parquet_paths)}"
        )

        download_with_curl(
            birdsnap_file_url(
                relative_path
            ),
            local_path,
            valid_parquet,
        )

        parquet_file = pq.ParquetFile(
            str(local_path)
        )

        row_count = (
            parquet_file.metadata.num_rows
        )

        selected_files.append(
            local_path
        )

        candidate_rows += row_count

        print(
            "Birdsnap candidate rows: "
            f"{candidate_rows:,}/"
            f"{BIRDSNAP_CANDIDATE_TARGET:,}"
        )

        if (
            candidate_rows
            >= BIRDSNAP_CANDIDATE_TARGET
        ):
            break

    if (
        candidate_rows
        < BIRDSNAP_CANDIDATE_TARGET
    ):
        raise RuntimeError(
            "Not enough Birdsnap candidate "
            "rows were downloaded."
        )

    return selected_files


def detect_image_column(row):
    for column_name, value in row.items():
        if isinstance(value, dict):
            if (
                "bytes" in value
                or "path" in value
            ):
                return column_name

    raise ValueError(
        "No image column was found "
        "in the Birdsnap file."
    )


def image_bytes_and_path(
    image_record,
):
    image_bytes = image_record.get(
        "bytes"
    )

    original_path = image_record.get(
        "path"
    )

    if isinstance(
        image_bytes,
        memoryview,
    ):
        image_bytes = (
            image_bytes.tobytes()
        )

    elif isinstance(
        image_bytes,
        bytearray,
    ):
        image_bytes = bytes(
            image_bytes
        )

    if image_bytes is None:
        raise ValueError(
            "Birdsnap image bytes "
            "are missing."
        )

    return image_bytes, original_path


def image_extension(
    image_bytes,
    original_path,
):
    if original_path:
        extension = Path(
            urlparse(
                str(original_path)
            ).path
        ).suffix.lower()

        if extension in IMAGE_EXTENSIONS:
            return extension

    with Image.open(
        io.BytesIO(image_bytes)
    ) as image:
        format_to_extension = {
            "JPEG": ".jpg",
            "PNG": ".png",
            "BMP": ".bmp",
            "WEBP": ".webp",
            "TIFF": ".tif",
        }

        if (
            image.format
            not in format_to_extension
        ):
            raise ValueError(
                "Unsupported image format: "
                f"{image.format}"
            )

        return format_to_extension[
            image.format
        ]


def safe_relative_path(
    original_path,
    row_number,
    extension,
):
    if original_path:
        raw_path = unquote(
            urlparse(
                str(original_path)
            ).path
        ).replace("\\", "/")

        path_parts = [
            re.sub(
                r'[<>:"|?*]',
                "_",
                part,
            )
            for part in raw_path.split("/")
            if part not in {
                "",
                ".",
                "..",
            }
        ]

        if "images" in path_parts:
            images_index = (
                path_parts.index("images")
            )

            path_parts = path_parts[
                images_index + 1:
            ]

        if path_parts:
            relative_path = Path(
                *path_parts
            )

            if (
                relative_path.suffix.lower()
                not in IMAGE_EXTENSIONS
            ):
                relative_path = (
                    relative_path.with_suffix(
                        extension
                    )
                )

            return relative_path

    return Path(
        f"image_{row_number:07d}"
        f"{extension}"
    )


def export_birdsnap_file(
    parquet_path,
):
    output_dir = (
        birdsnap_images_dir
        / parquet_path.stem
    )

    marker_path = (
        output_dir
        / ".export_complete.txt"
    )

    if marker_path.exists():
        print(
            "Already exported: "
            f"{parquet_path.name}"
        )
        return

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    parquet_file = pq.ParquetFile(
        str(parquet_path)
    )

    total_rows = (
        parquet_file.metadata.num_rows
    )

    image_column = None
    row_number = 0
    saved_count = 0
    failed_count = 0
    used_paths = set()

    progress = tqdm(
        total=total_rows,
        desc=(
            f"Exporting "
            f"{parquet_path.stem}"
        ),
        unit="image",
    )

    try:
        for batch in parquet_file.iter_batches(
            batch_size=16
        ):
            for row in batch.to_pylist():
                row_number += 1

                try:
                    if image_column is None:
                        image_column = (
                            detect_image_column(
                                row
                            )
                        )

                    (
                        image_bytes,
                        original_path,
                    ) = image_bytes_and_path(
                        row[image_column]
                    )

                    extension = image_extension(
                        image_bytes,
                        original_path,
                    )

                    relative_path = (
                        safe_relative_path(
                            original_path,
                            row_number,
                            extension,
                        )
                    )

                    relative_key = str(
                        relative_path
                    ).lower()

                    if (
                        relative_key
                        in used_paths
                    ):
                        relative_path = (
                            relative_path.parent
                            / (
                                f"{relative_path.stem}_"
                                f"{row_number:07d}"
                                f"{relative_path.suffix}"
                            )
                        )

                    used_paths.add(
                        str(
                            relative_path
                        ).lower()
                    )

                    output_path = (
                        output_dir
                        / relative_path
                    )

                    output_path.parent.mkdir(
                        parents=True,
                        exist_ok=True,
                    )

                    output_path.write_bytes(
                        image_bytes
                    )

                    saved_count += 1

                except Exception:
                    failed_count += 1

                progress.update(1)

    finally:
        progress.close()

    marker_path.write_text(
        (
            f"rows={total_rows}\n"
            f"saved={saved_count}\n"
            f"failed={failed_count}\n"
        ),
        encoding="utf-8",
    )

    print(
        f"Exported {saved_count:,} images "
        f"from {parquet_path.name}."
    )

    if failed_count:
        print(
            f"Skipped {failed_count:,} "
            "unreadable rows."
        )


def prepare_birdsnap():
    existing_images = get_image_files(
        birdsnap_images_dir
    )

    if (
        len(existing_images)
        >= BIRDSNAP_CANDIDATE_TARGET
    ):
        print(
            "The required Birdsnap subset "
            "is already prepared."
        )
        return

    selected_files = (
        download_birdsnap_subset()
    )

    print(
        "Exporting Birdsnap images..."
    )

    for parquet_path in selected_files:
        export_birdsnap_file(
            parquet_path
        )

    exported_images = get_image_files(
        birdsnap_images_dir
    )

    if (
        len(exported_images)
        < BIRDSNAP_CANDIDATE_TARGET
    ):
        raise RuntimeError(
            "The exported Birdsnap subset "
            "is too small."
        )

    print(
        "Birdsnap preparation is complete."
    )


def shuffled_paths(
    image_paths,
    seed,
):
    image_paths = list(
        image_paths
    )

    random.Random(seed).shuffle(
        image_paths
    )

    return image_paths


print("Preparing CUB-200-2011...")

prepare_cub()

print()


print("Preparing NABirds...")

prepare_nabirds()

print()


print("Preparing Birdsnap...")

prepare_birdsnap()

print()


if RESET_OUTPUT_FOLDERS:
    print(
        "Resetting previous Bird "
        "automatic-cleaning outputs..."
    )

    reset_folder(
        bird_auto_cleaned_dir
    )

    reset_folder(
        bird_rejected_dir
    )

    print(
        "Previous Bird automatic-cleaning "
        "outputs were reset."
    )


for source in bird_sources.values():
    source["cleaned_dir"].mkdir(
        parents=True,
        exist_ok=True,
    )

    source["rejected_dir"].mkdir(
        parents=True,
        exist_ok=True,
    )


print()


bird_source_image_paths = {
    "cub_200_2011": get_image_files(
        cub_images_dir
    ),
    "nabirds": shuffled_paths(
        get_image_files(
            nabirds_source["original_dir"]
        ),
        RANDOM_SEED,
    ),
    "birdsnap": shuffled_paths(
        get_image_files(
            birdsnap_images_dir
        ),
        RANDOM_SEED + 1,
    ),
}


print("Candidate Bird images found:")

for source_name, image_paths in (
    bird_source_image_paths.items()
):
    display_name = bird_sources[
        source_name
    ]["folder_name"]

    print(
        f"{display_name}: "
        f"{len(image_paths):,}"
    )

print()


seen_exact_hashes = {}
duplicate_group_numbers = {}

processed_counts = defaultdict(int)
cleaned_counts = defaultdict(int)
rejected_counts = defaultdict(int)

rejection_reasons = defaultdict(
    lambda: defaultdict(int)
)


def clean_bird_source(
    source_name,
    accepted_target=None,
):
    source = bird_sources[
        source_name
    ]

    image_paths = (
        bird_source_image_paths[
            source_name
        ]
    )

    display_name = source[
        "folder_name"
    ]

    source_root = source[
        "original_dir"
    ]

    source_accepted_count = 0

    print(
        f"Automatically cleaning "
        f"{display_name}..."
    )

    for image_path in tqdm(
        image_paths,
        desc=display_name,
        unit="image",
    ):
        if (
            accepted_target is not None
            and source_accepted_count
            >= accepted_target
        ):
            break

        processed_counts[
            source_name
        ] += 1

        is_valid, reason = check_image(
            image_path
        )

        if not is_valid:
            rejected_counts[
                source_name
            ] += 1

            rejection_reasons[
                source_name
            ][reason] += 1

            copy_rejected_image(
                image_path,
                source_root,
                source["rejected_dir"]
                / reason,
            )

            continue

        try:
            exact_hash = (
                get_exact_image_hash(
                    image_path
                )
            )

        except Exception:
            reason = "hash_failed"

            rejected_counts[
                source_name
            ] += 1

            rejection_reasons[
                source_name
            ][reason] += 1

            copy_rejected_image(
                image_path,
                source_root,
                source["rejected_dir"]
                / reason,
            )

            continue

        if exact_hash in seen_exact_hashes:
            rejected_counts[
                source_name
            ] += 1

            rejection_reasons[
                source_name
            ]["exact_duplicate"] += 1

            if (
                exact_hash
                not in duplicate_group_numbers
            ):
                duplicate_group_numbers[
                    exact_hash
                ] = (
                    len(
                        duplicate_group_numbers
                    )
                    + 1
                )

            group_number = (
                duplicate_group_numbers[
                    exact_hash
                ]
            )

            group_dir = (
                source["rejected_dir"]
                / "exact_duplicate"
                / (
                    "duplicate_group_"
                    f"{group_number:06d}"
                )
            )

            copy_duplicate_group(
                duplicate_path=image_path,
                duplicate_source_root=(
                    source_root
                ),
                duplicate_source_name=(
                    display_name
                ),
                kept_record=(
                    seen_exact_hashes[
                        exact_hash
                    ]
                ),
                group_dir=group_dir,
            )

            continue

        try:
            cleaned_path = (
                copy_preserving_structure(
                    image_path,
                    source_root,
                    source["cleaned_dir"],
                )
            )

        except Exception:
            reason = "copy_failed"

            rejected_counts[
                source_name
            ] += 1

            rejection_reasons[
                source_name
            ][reason] += 1

            copy_rejected_image(
                image_path,
                source_root,
                source["rejected_dir"]
                / reason,
            )

            continue

        seen_exact_hashes[
            exact_hash
        ] = {
            "source_name": display_name,
            "source_root": source_root,
            "original_path": image_path,
            "cleaned_path": cleaned_path,
        }

        cleaned_counts[
            source_name
        ] += 1

        source_accepted_count += 1


clean_bird_source(
    "cub_200_2011"
)


cub_accepted_count = cleaned_counts[
    "cub_200_2011"
]

remaining_target = max(
    0,
    BIRD_TRAIN_VAL_TARGET
    - cub_accepted_count,
)

nabirds_target = (
    remaining_target // 2
)

birdsnap_planned_target = (
    remaining_target
    - nabirds_target
)


print()

print(
    "Planned accepted-image "
    "contribution:"
)

print(
    f"  CUB-200-2011: "
    f"{cub_accepted_count:,}"
)

print(
    f"  NABirds:      "
    f"{nabirds_target:,}"
)

print(
    f"  Birdsnap:     "
    f"{birdsnap_planned_target:,}"
)

print()


clean_bird_source(
    "nabirds",
    accepted_target=nabirds_target,
)


birdsnap_target = max(
    0,
    BIRD_TRAIN_VAL_TARGET
    - sum(
        cleaned_counts.values()
    ),
)


if (
    len(
        bird_source_image_paths[
            "birdsnap"
        ]
    )
    < birdsnap_target
):
    raise RuntimeError(
        "The Birdsnap subset does not "
        "contain enough images."
    )


clean_bird_source(
    "birdsnap",
    accepted_target=birdsnap_target,
)


print()

print(
    "Bird train/validation automatic "
    "cleaning is complete."
)

print()


for source_name, image_paths in (
    bird_source_image_paths.items()
):
    display_name = bird_sources[
        source_name
    ]["folder_name"]

    print(display_name)

    print(
        "  Available candidates:    "
        f"{len(image_paths):,}"
    )

    print(
        "  Processed images:        "
        f"{processed_counts[source_name]:,}"
    )

    print(
        "  Auto Cleaned images:     "
        f"{cleaned_counts[source_name]:,}"
    )

    print(
        "  Automatically rejected:  "
        f"{rejected_counts[source_name]:,}"
    )

    if rejection_reasons[source_name]:
        print("  Rejection reasons:")

        for reason, count in sorted(
            rejection_reasons[
                source_name
            ].items()
        ):
            print(
                f"    {reason}: "
                f"{count:,}"
            )

    print()


total_candidates = sum(
    len(image_paths)
    for image_paths
    in bird_source_image_paths.values()
)

total_processed = sum(
    processed_counts.values()
)

total_cleaned = sum(
    cleaned_counts.values()
)

total_rejected = sum(
    rejected_counts.values()
)


print("Total")

print(
    "  Available candidates:    "
    f"{total_candidates:,}"
)

print(
    "  Processed images:        "
    f"{total_processed:,}"
)

print(
    "  Auto Cleaned images:     "
    f"{total_cleaned:,}"
)

print(
    "  Automatically rejected:  "
    f"{total_rejected:,}"
)

print(
    "  Target images:           "
    f"{BIRD_TRAIN_VAL_TARGET:,}"
)

print()


if total_cleaned >= BIRD_TRAIN_VAL_TARGET:
    print(
        "The Bird train/validation "
        "target was reached."
    )

else:
    print(
        "The Bird train/validation "
        "target was not reached. "
        "Additional images needed: "
        f"{BIRD_TRAIN_VAL_TARGET - total_cleaned:,}"
    )


print()

print(
    "Auto Cleaned Bird images saved in: "
    f"{show_path(bird_auto_cleaned_dir)}"
)

print(
    "Automatically rejected Bird images "
    "saved in: "
    f"{show_path(bird_rejected_dir)}"
)

Bird train/validation paths are ready.
Original dataset directory:       Image Classifier Project\Model Building\Datasets\Original\Train and Validation
Auto Cleaned Bird directory:      Image Classifier Project\Model Building\Datasets\Auto Cleaned\Train and Validation\Bird
Automatically rejected directory: Image Classifier Project\Model Building\Datasets\Rejected\Automatic\Train and Validation\Bird
Bird train/validation target:     31,000

Preparing CUB-200-2011...
CUB-200-2011 is already downloaded and extracted.

Preparing NABirds...
NABirds is already downloaded and extracted.

Preparing Birdsnap...
The required Birdsnap subset is already prepared.

Resetting previous Bird automatic-cleaning outputs...
Previous Bird automatic-cleaning outputs were reset.

Candidate Bird images found:
CUB-200-2011: 11,788
NABirds: 48,562
Birdsnap: 12,473

Automatically cleaning CUB-200-2011...


CUB-200-2011: 100%|██████████| 11788/11788 [01:13<00:00, 160.09image/s]



Planned accepted-image contribution:
  CUB-200-2011: 11,787
  NABirds:      9,606
  Birdsnap:     9,607

Automatically cleaning NABirds...


NABirds:  20%|█▉        | 9606/48562 [02:43<11:01, 58.89image/s]


Automatically cleaning Birdsnap...


Birdsnap:  77%|███████▋  | 9620/12473 [17:59<05:20,  8.91image/s] 


Bird train/validation automatic cleaning is complete.

CUB-200-2011
  Available candidates:    11,788
  Processed images:        11,788
  Auto Cleaned images:     11,787
  Automatically rejected:  1
  Rejection reasons:
    exact_duplicate: 1

NABirds
  Available candidates:    48,562
  Processed images:        9,606
  Auto Cleaned images:     9,606
  Automatically rejected:  0

Birdsnap
  Available candidates:    12,473
  Processed images:        9,620
  Auto Cleaned images:     9,607
  Automatically rejected:  13
  Rejection reasons:
    exact_duplicate: 13

Total
  Available candidates:    72,823
  Processed images:        31,014
  Auto Cleaned images:     31,000
  Automatically rejected:  14
  Target images:           31,000

The Bird train/validation target was reached.

Auto Cleaned Bird images saved in: Image Classifier Project\Model Building\Datasets\Auto Cleaned\Train and Validation\Bird
Automatically rejected Bird images saved in: Image Classifier Project\Model Building\Data

### 1.4 Downloading and Automatically Cleaning Unknown Train/Validation Images

This section downloads and automatically cleans the Unknown images intended for the training and validation pool.

Raw Unknown train/validation source archives and extracted datasets are stored in source-based folders under:

`Model Building/Datasets/Original/Train and Validation`

Automatically cleaned Unknown train/validation images are stored in source-based folders under:

`Model Building/Datasets/Auto Cleaned/Train and Validation/Unknown`

Automatically rejected Unknown images are copied into source- and reason-based folders under:

`Model Building/Datasets/Rejected/Automatic/Train and Validation/Unknown`

The Unknown class contains images that do **not** belong to Cat, Dog, or Bird. This helps the model reject unrelated uploads instead of forcing every image into one of the three target classes.

The Unknown train/validation pool uses several external datasets plus one derived Unknown image source:

```text
iNaturalist Non-Target Taxa:
Links:
https://www.inaturalist.org/
https://api.inaturalist.org/v1/docs/

Non-target animals such as fish, frogs, reptiles, insects, bats, rodents, livestock, primates, marine animals, plants, and other wildlife.

AFHQ-v2 Wild:
Link:
https://github.com/clovaai/stargan-v2

Wild-animal images from the AFHQ-v2 Wild category that do not belong to the Cat, Dog, or Bird classes.

NINCO:
Links:
https://zenodo.org/records/8013288
https://zenodo.org/records/8013288/files/NINCO_all.tar.gz?download=1

Natural out-of-distribution images containing non-target animals, objects, food, plants, and other content outside the target classes.

ImageNet-O:
Links:
https://github.com/hendrycks/natural-adv-examples
https://people.eecs.berkeley.edu/~hendrycks/imagenet-o.tar

Unusual real-world out-of-distribution images used to improve the variety of difficult Unknown examples.

COCO filtered negatives:
Links:
https://cocodataset.org/
http://images.cocodataset.org/zips/train2017.zip
http://images.cocodataset.org/annotations/annotations_trainval2017.zip

Everyday objects, humans, vehicles, furniture, household content, and other real-world images. Images annotated as Cat, Dog, or Bird are excluded using the COCO annotations.

Places365:
Links:
https://places2.csail.mit.edu/download.html
http://data.csail.mit.edu/places/places365/val_256.tar

Scene images such as streets, rooms, offices, beaches, buildings, landscapes, and other indoor or outdoor environments.

Food-101:
Links:
https://data.vision.ee.ethz.ch/cvl/datasets_extra/food-101/
http://data.vision.ee.ethz.ch/cvl/food-101.tar.gz

Food images representing meals and other common non-animal uploads.

Derived Unknown Corruptions:
No external download link.

These images are generated locally from accepted Unknown images using realistic blur, compression, darkening, noise, and crop-resize transformations.

In [9]:
import json
import math
import random
import re
import time
from collections import defaultdict
from pathlib import Path
from urllib.parse import urlparse

import numpy as np
import requests
from PIL import Image, ImageEnhance, ImageFilter, ImageOps
from tqdm.auto import tqdm


UNKNOWN_FINAL_TARGET = 50000
UNKNOWN_AUTO_CLEANED_TARGET = 55000
UNKNOWN_BUFFER_SIZE = (
    UNKNOWN_AUTO_CLEANED_TARGET
    - UNKNOWN_FINAL_TARGET
)

RANDOM_SEED = 42

RESET_UNKNOWN_TRAIN_VAL_OUTPUT = True
CHECK_TARGET_CLASS_DUPLICATES = True


UNKNOWN_SOURCE_TARGETS = {
    "iNaturalist Non-Target Taxa": 10350,
    "AFHQ-v2 Wild": 4500,
    "NINCO": 3850,
    "ImageNet-O": 1650,
    "COCO Humans": 8800,
    "COCO Other Filtered Negatives": 4400,
    "Places365": 7700,
    "Food-101": 8250,
    "Derived Unknown Corruptions": 5500,
}


if (
    sum(UNKNOWN_SOURCE_TARGETS.values())
    != UNKNOWN_AUTO_CLEANED_TARGET
):
    raise ValueError(
        "Unknown source targets do not add up "
        "to the automatic-cleaning target."
    )


unknown_original_dirs = {
    "iNaturalist Non-Target Taxa": (
        original_train_val_dir
        / "iNaturalist Non-Target Taxa"
    ),
    "AFHQ-v2 Wild": (
        original_train_val_dir
        / "AFHQ-v2"
    ),
    "NINCO": (
        original_train_val_dir
        / "NINCO"
    ),
    "ImageNet-O": (
        original_train_val_dir
        / "ImageNet-O"
    ),
    "COCO 2017": (
        original_train_val_dir
        / "COCO 2017"
    ),
    "Places365": (
        original_train_val_dir
        / "Places365"
    ),
    "Food-101": (
        original_train_val_dir
        / "Food-101"
    ),
    "Derived Unknown Corruptions": (
        original_train_val_dir
        / "Derived Unknown Corruptions"
    ),
}


unknown_auto_cleaned_dir = (
    auto_cleaned_train_val_dir
    / "Unknown"
)

unknown_rejected_dir = (
    rejected_automatic_train_val_dir
    / "Unknown"
)


unknown_output_dirs = {
    "iNaturalist Non-Target Taxa": (
        unknown_auto_cleaned_dir
        / "iNaturalist Non-Target Taxa"
    ),
    "AFHQ-v2 Wild": (
        unknown_auto_cleaned_dir
        / "AFHQ-v2 Wild"
    ),
    "NINCO": (
        unknown_auto_cleaned_dir
        / "NINCO"
    ),
    "ImageNet-O": (
        unknown_auto_cleaned_dir
        / "ImageNet-O"
    ),
    "COCO Humans": (
        unknown_auto_cleaned_dir
        / "COCO Filtered Negatives"
        / "Humans"
    ),
    "COCO Other Filtered Negatives": (
        unknown_auto_cleaned_dir
        / "COCO Filtered Negatives"
        / "Other"
    ),
    "Places365": (
        unknown_auto_cleaned_dir
        / "Places365"
    ),
    "Food-101": (
        unknown_auto_cleaned_dir
        / "Food-101"
    ),
    "Derived Unknown Corruptions": (
        unknown_auto_cleaned_dir
        / "Derived Unknown Corruptions"
    ),
}


unknown_rejected_source_dirs = {
    source_name: (
        unknown_rejected_dir
        / source_name
    )
    for source_name in UNKNOWN_SOURCE_TARGETS
}


if RESET_UNKNOWN_TRAIN_VAL_OUTPUT:
    reset_folder(
        unknown_auto_cleaned_dir
    )

    reset_folder(
        unknown_rejected_dir
    )


for folder in [
    *unknown_original_dirs.values(),
    *unknown_output_dirs.values(),
    *unknown_rejected_source_dirs.values(),
]:
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )


UNKNOWN_ARCHIVES = [
    {
        "source": "NINCO",
        "url": (
            "https://zenodo.org/records/8013288/"
            "files/NINCO_all.tar.gz?download=1"
        ),
        "path": (
            unknown_original_dirs["NINCO"]
            / "NINCO_all.tar.gz"
        ),
        "type": "tar",
    },
    {
        "source": "ImageNet-O",
        "url": (
            "https://people.eecs.berkeley.edu/"
            "~hendrycks/imagenet-o.tar"
        ),
        "path": (
            unknown_original_dirs["ImageNet-O"]
            / "imagenet-o.tar"
        ),
        "type": "tar",
    },
    {
        "source": "COCO 2017 Images",
        "url": (
            "https://s3.amazonaws.com/"
            "images.cocodataset.org/"
            "zips/train2017.zip"
        ),
        "path": (
            unknown_original_dirs["COCO 2017"]
            / "train2017.zip"
        ),
        "type": "zip",
    },
    {
        "source": "COCO 2017 Annotations",
        "url": (
            "https://s3.amazonaws.com/"
            "images.cocodataset.org/"
            "annotations/annotations_trainval2017.zip"
        ),
        "path": (
            unknown_original_dirs["COCO 2017"]
            / "annotations_trainval2017.zip"
        ),
        "type": "zip",
    },
    {
        "source": "Places365",
        "url": (
            "http://data.csail.mit.edu/"
            "places/places365/val_256.tar"
        ),
        "path": (
            unknown_original_dirs["Places365"]
            / "val_256.tar"
        ),
        "type": "tar",
    },
    {
        "source": "Food-101",
        "url": (
            "http://data.vision.ee.ethz.ch/"
            "cvl/food-101.tar.gz"
        ),
        "path": (
            unknown_original_dirs["Food-101"]
            / "food-101.tar.gz"
        ),
        "type": "tar",
    },
]


def unknown_safe_name(value):
    return re.sub(
        r'[<>:"/\\|?*]+',
        "_",
        str(value),
    ).strip()


def unknown_find_file(
    folder,
    file_name,
):
    matches = sorted(
        path
        for path in Path(folder).rglob(file_name)
        if path.is_file()
    )

    if matches:
        return matches[0]

    return None


def unknown_find_folder(
    folder,
    folder_name,
):
    folder_name = folder_name.lower()

    matches = sorted(
        path
        for path in Path(folder).rglob("*")
        if (
            path.is_dir()
            and path.name.lower() == folder_name
        )
    )

    if matches:
        return matches[0]

    return None


def unknown_shuffled_paths(
    image_paths,
    seed,
):
    image_paths = list(
        image_paths
    )

    random.Random(seed).shuffle(
        image_paths
    )

    return image_paths


print(
    "Unknown train/validation paths are ready."
)

print(
    f"Original directory:     "
    f"{show_path(original_train_val_dir)}"
)

print(
    f"Auto Cleaned directory: "
    f"{show_path(unknown_auto_cleaned_dir)}"
)

print(
    f"Rejected directory:     "
    f"{show_path(unknown_rejected_dir)}"
)

print()

print(
    "Unknown source targets:"
)

for source_name, source_target in (
    UNKNOWN_SOURCE_TARGETS.items()
):
    print(
        f"  {source_name}: "
        f"{source_target:,}"
    )

print()

print(
    f"Automatically cleaned target: "
    f"{UNKNOWN_AUTO_CLEANED_TARGET:,}"
)

print(
    f"Final target:                 "
    f"{UNKNOWN_FINAL_TARGET:,}"
)

print(
    f"Manual-cleaning buffer:       "
    f"{UNKNOWN_BUFFER_SIZE:,}"
)

print()


print(
    "Downloading Unknown source archives..."
)

for archive in UNKNOWN_ARCHIVES:
    print(
        f"{archive['source']}:"
    )

    download_file(
        archive["url"],
        archive["path"],
    )

print()

print(
    "Extracting Unknown source archives..."
)

for archive in UNKNOWN_ARCHIVES:
    if archive["type"] == "zip":
        extract_zip(
            archive["path"],
            archive["path"].parent,
        )

    else:
        extract_tar(
            archive["path"],
            archive["path"].parent,
        )

print()


# AFHQ-v2 was downloaded earlier for Cat and Dog.
afhq_source_dir = (
    unknown_original_dirs["AFHQ-v2 Wild"]
)

for afhq_archive in sorted(
    afhq_source_dir.glob("*.zip")
):
    extract_zip(
        afhq_archive,
        afhq_source_dir,
    )


INATURALIST_API_URL = (
    "https://api.inaturalist.org/v1"
)


# More candidates are collected than the accepted target.
INATURALIST_CANDIDATE_TARGETS = {
    "Actinopterygii": 1500,
    "Amphibia": 900,
    "Reptilia": 1000,
    "Insecta": 1400,
    "Arachnida": 700,
    "Chiroptera": 550,
    "Rodentia": 900,
    "Primates": 500,
    "Cetacea": 450,
    "Mollusca": 800,
    "Crustacea": 650,
    "Equus caballus": 400,
    "Bos taurus": 400,
    "Ovis aries": 400,
    "Capra hircus": 400,
    "Sus scrofa": 400,
    "Mustelidae": 450,
    "Echinodermata": 700,
}


if (
    sum(INATURALIST_CANDIDATE_TARGETS.values())
    != 12500
):
    raise ValueError(
        "The iNaturalist candidate targets "
        "should add up to 12,500."
    )


inat_session = requests.Session()

inat_session.headers.update(
    {
        "User-Agent": (
            "Image-Classifier-Project/1.0"
        )
    }
)


def inat_request_json(
    endpoint,
    params,
    attempts=5,
):
    url = (
        f"{INATURALIST_API_URL}"
        f"/{endpoint}"
    )

    for attempt in range(
        1,
        attempts + 1,
    ):
        try:
            response = inat_session.get(
                url,
                params=params,
                timeout=(30, 120),
            )

            if response.status_code == 429:
                wait_time = (
                    10 * attempt
                )

                print(
                    "iNaturalist rate limit reached. "
                    f"Waiting {wait_time} seconds..."
                )

                time.sleep(
                    wait_time
                )

                continue

            response.raise_for_status()

            return response.json()

        except Exception as error:
            if attempt == attempts:
                raise RuntimeError(
                    "iNaturalist request failed: "
                    f"{endpoint}"
                ) from error

            time.sleep(
                3 * attempt
            )


def resolve_inaturalist_taxon_id(
    taxon_name,
):
    payload = inat_request_json(
        "taxa/autocomplete",
        {
            "q": taxon_name,
            "per_page": 30,
        },
    )

    results = payload.get(
        "results",
        [],
    )

    for result in results:
        if (
            result.get(
                "name",
                "",
            ).lower()
            == taxon_name.lower()
        ):
            return result["id"]

    if results:
        return results[0]["id"]

    raise RuntimeError(
        "iNaturalist taxon was not found: "
        f"{taxon_name}"
    )


def inat_large_photo_url(
    photo_url,
):
    return re.sub(
        r"/square\.",
        "/large.",
        photo_url,
    )


def download_inaturalist_photo(
    photo_url,
    output_path,
    attempts=4,
):
    output_path = Path(
        output_path
    )

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    partial_path = output_path.with_name(
        output_path.name + ".part"
    )

    for attempt in range(
        1,
        attempts + 1,
    ):
        try:
            with inat_session.get(
                photo_url,
                stream=True,
                timeout=(30, 120),
            ) as response:
                response.raise_for_status()

                content_type = (
                    response.headers.get(
                        "content-type",
                        "",
                    ).lower()
                )

                if (
                    content_type
                    and "image" not in content_type
                ):
                    raise ValueError(
                        "The downloaded response "
                        "was not an image."
                    )

                with partial_path.open(
                    "wb"
                ) as output_file:
                    for block in (
                        response.iter_content(
                            chunk_size=1024 * 1024
                        )
                    ):
                        if block:
                            output_file.write(
                                block
                            )

            partial_path.replace(
                output_path
            )

            return True

        except Exception:
            if partial_path.exists():
                partial_path.unlink()

            if attempt < attempts:
                time.sleep(
                    attempt * 2
                )

    return False


def download_inaturalist_candidates():
    source_root = (
        unknown_original_dirs[
            "iNaturalist Non-Target Taxa"
        ]
    )

    print(
        "Collecting iNaturalist "
        "non-target animal candidates..."
    )

    for taxon_index, (
        taxon_name,
        candidate_target,
    ) in enumerate(
        INATURALIST_CANDIDATE_TARGETS.items(),
        start=1,
    ):
        taxon_folder = (
            source_root
            / (
                f"{taxon_index:02d}_"
                f"{unknown_safe_name(taxon_name)}"
            )
        )

        taxon_folder.mkdir(
            parents=True,
            exist_ok=True,
        )

        existing_count = len(
            get_image_files(
                taxon_folder
            )
        )

        if existing_count >= candidate_target:
            print(
                f"{taxon_name}: "
                f"{existing_count:,} already available."
            )

            continue

        taxon_id = (
            resolve_inaturalist_taxon_id(
                taxon_name
            )
        )

        base_params = {
            "taxon_id": taxon_id,
            "quality_grade": "research",
            "photos": "true",
            "captive": "false",
            "per_page": 200,
            "order_by": "id",
            "order": "desc",
        }

        first_page_payload = (
            inat_request_json(
                "observations",
                {
                    **base_params,
                    "page": 1,
                },
            )
        )

        total_results = int(
            first_page_payload.get(
                "total_results",
                0,
            )
        )

        available_window = min(
            total_results,
            10000,
        )

        page_count = max(
            1,
            math.ceil(
                available_window / 200
            ),
        )

        page_numbers = list(
            range(
                1,
                page_count + 1,
            )
        )

        page_rng = random.Random(
            RANDOM_SEED
            + taxon_index
        )

        page_rng.shuffle(
            page_numbers
        )

        page_cache = {
            1: first_page_payload
        }

        needed_count = (
            candidate_target
            - existing_count
        )

        downloaded_count = 0
        seen_observation_ids = set()

        with tqdm(
            total=needed_count,
            desc=taxon_name,
            unit="image",
        ) as progress_bar:
            for page_number in page_numbers:
                if (
                    downloaded_count
                    >= needed_count
                ):
                    break

                if page_number in page_cache:
                    payload = (
                        page_cache[
                            page_number
                        ]
                    )

                else:
                    payload = (
                        inat_request_json(
                            "observations",
                            {
                                **base_params,
                                "page": page_number,
                            },
                        )
                    )

                    time.sleep(
                        1
                    )

                observations = list(
                    payload.get(
                        "results",
                        [],
                    )
                )

                page_rng.shuffle(
                    observations
                )

                for observation in observations:
                    if (
                        downloaded_count
                        >= needed_count
                    ):
                        break

                    observation_id = (
                        observation.get(
                            "id"
                        )
                    )

                    if (
                        observation_id
                        in seen_observation_ids
                    ):
                        continue

                    seen_observation_ids.add(
                        observation_id
                    )

                    observation_taxon = (
                        observation.get(
                            "taxon"
                        )
                        or {}
                    )

                    if (
                        observation_taxon.get(
                            "name"
                        )
                        == "Homo sapiens"
                    ):
                        continue

                    photos = observation.get(
                        "photos",
                        [],
                    )

                    if not photos:
                        continue

                    photo = page_rng.choice(
                        photos
                    )

                    photo_url = photo.get(
                        "url"
                    )

                    if not photo_url:
                        continue

                    photo_url = (
                        inat_large_photo_url(
                            photo_url
                        )
                    )

                    photo_id = photo.get(
                        "id",
                        "unknown",
                    )

                    suffix = Path(
                        urlparse(
                            photo_url
                        ).path
                    ).suffix.lower()

                    if (
                        suffix
                        not in IMAGE_EXTENSIONS
                    ):
                        suffix = ".jpg"

                    output_name = (
                        f"observation_"
                        f"{observation_id}_"
                        f"photo_{photo_id}"
                        f"{suffix}"
                    )

                    output_path = (
                        taxon_folder
                        / output_name
                    )

                    if output_path.exists():
                        continue

                    downloaded = (
                        download_inaturalist_photo(
                            photo_url,
                            output_path,
                        )
                    )

                    if downloaded:
                        downloaded_count += 1

                        progress_bar.update(
                            1
                        )

        final_count = len(
            get_image_files(
                taxon_folder
            )
        )

        print(
            f"{taxon_name}: "
            f"{final_count:,} candidates."
        )

        if final_count < candidate_target:
            raise RuntimeError(
                f"Only {final_count:,} "
                f"{taxon_name} images were "
                f"downloaded. The target was "
                f"{candidate_target:,}."
            )

    print()


download_inaturalist_candidates()


def get_afhq_wild_images():
    source_dir = (
        unknown_original_dirs[
            "AFHQ-v2 Wild"
        ]
    )

    image_paths = get_image_files(
        source_dir
    )

    wild_images = [
        image_path
        for image_path in image_paths
        if any(
            part.lower() == "wild"
            for part in image_path.parts
        )
    ]

    if (
        len(wild_images)
        < UNKNOWN_SOURCE_TARGETS[
            "AFHQ-v2 Wild"
        ]
    ):
        raise RuntimeError(
            "AFHQ-v2 does not contain "
            "enough Wild images."
        )

    return (
        source_dir,
        wild_images,
    )


def get_ninco_images():
    source_dir = (
        unknown_original_dirs[
            "NINCO"
        ]
    )

    preferred_names = {
        "ninco_ood_classes",
        "ninco",
    }

    candidate_folders = [
        path
        for path in source_dir.rglob("*")
        if (
            path.is_dir()
            and path.name.lower()
            in preferred_names
        )
    ]

    candidate_folders.sort(
        key=lambda folder: len(
            get_image_files(
                folder
            )
        ),
        reverse=True,
    )

    for candidate_folder in (
        candidate_folders
    ):
        candidate_images = (
            get_image_files(
                candidate_folder
            )
        )

        if (
            len(candidate_images)
            >= UNKNOWN_SOURCE_TARGETS[
                "NINCO"
            ]
        ):
            return (
                candidate_folder,
                candidate_images,
            )

    all_images = get_image_files(
        source_dir
    )

    if (
        len(all_images)
        < UNKNOWN_SOURCE_TARGETS[
            "NINCO"
        ]
    ):
        raise RuntimeError(
            "NINCO does not contain "
            "enough images."
        )

    return (
        source_dir,
        all_images,
    )


def get_imagenet_o_images():
    source_dir = (
        unknown_original_dirs[
            "ImageNet-O"
        ]
    )

    image_paths = get_image_files(
        source_dir
    )

    if (
        len(image_paths)
        < UNKNOWN_SOURCE_TARGETS[
            "ImageNet-O"
        ]
    ):
        raise RuntimeError(
            "ImageNet-O does not contain "
            "enough images."
        )

    return (
        source_dir,
        image_paths,
    )


def get_places365_images():
    source_dir = (
        unknown_original_dirs[
            "Places365"
        ]
    )

    image_paths = get_image_files(
        source_dir
    )

    if (
        len(image_paths)
        < UNKNOWN_SOURCE_TARGETS[
            "Places365"
        ]
    ):
        raise RuntimeError(
            "Places365 does not contain "
            "enough images."
        )

    return (
        source_dir,
        image_paths,
    )


def get_food101_images():
    source_dir = (
        unknown_original_dirs[
            "Food-101"
        ]
    )

    image_folder = None

    for candidate_folder in (
        source_dir.rglob("images")
    ):
        if (
            candidate_folder.is_dir()
            and "food-101"
            in str(
                candidate_folder
            ).lower()
        ):
            image_folder = (
                candidate_folder
            )

            break

    if image_folder is None:
        image_folder = source_dir

    image_paths = get_image_files(
        image_folder
    )

    if (
        len(image_paths)
        < UNKNOWN_SOURCE_TARGETS[
            "Food-101"
        ]
    ):
        raise RuntimeError(
            "Food-101 does not contain "
            "enough images."
        )

    return (
        image_folder,
        image_paths,
    )


def get_coco_filtered_images():
    source_dir = (
        unknown_original_dirs[
            "COCO 2017"
        ]
    )

    annotation_path = (
        unknown_find_file(
            source_dir,
            "instances_train2017.json",
        )
    )

    image_folder = (
        unknown_find_folder(
            source_dir,
            "train2017",
        )
    )

    if annotation_path is None:
        raise FileNotFoundError(
            "COCO instances_train2017.json "
            "was not found."
        )

    if image_folder is None:
        raise FileNotFoundError(
            "COCO train2017 image folder "
            "was not found."
        )

    print(
        "Reading COCO annotations..."
    )

    with annotation_path.open(
        "r",
        encoding="utf-8",
    ) as annotation_file:
        coco_data = json.load(
            annotation_file
        )

    category_ids = {
        category["name"]:
        category["id"]
        for category
        in coco_data["categories"]
    }

    required_categories = {
        "person",
        "bird",
        "cat",
        "dog",
    }

    missing_categories = (
        required_categories
        - set(category_ids)
    )

    if missing_categories:
        raise RuntimeError(
            "Missing COCO categories: "
            + ", ".join(
                sorted(
                    missing_categories
                )
            )
        )

    target_category_ids = {
        category_ids["bird"],
        category_ids["cat"],
        category_ids["dog"],
    }

    person_category_id = (
        category_ids["person"]
    )

    categories_by_image = (
        defaultdict(set)
    )

    for annotation in tqdm(
        coco_data["annotations"],
        desc="Reading COCO labels",
        unit="annotation",
    ):
        categories_by_image[
            annotation["image_id"]
        ].add(
            annotation["category_id"]
        )

    human_file_names = set()
    other_file_names = set()

    for image_info in (
        coco_data["images"]
    ):
        image_id = (
            image_info["id"]
        )

        image_categories = (
            categories_by_image[
                image_id
            ]
        )

        # Remove images annotated as Cat, Dog, or Bird.
        if (
            image_categories
            & target_category_ids
        ):
            continue

        if (
            person_category_id
            in image_categories
        ):
            human_file_names.add(
                image_info["file_name"]
            )

        else:
            other_file_names.add(
                image_info["file_name"]
            )

    del coco_data
    del categories_by_image

    all_coco_images = (
        get_image_files(
            image_folder
        )
    )

    human_images = [
        image_path
        for image_path
        in all_coco_images
        if (
            image_path.name
            in human_file_names
        )
    ]

    other_images = [
        image_path
        for image_path
        in all_coco_images
        if (
            image_path.name
            in other_file_names
        )
    ]

    if (
        len(human_images)
        < UNKNOWN_SOURCE_TARGETS[
            "COCO Humans"
        ]
    ):
        raise RuntimeError(
            "COCO does not contain enough "
            "filtered human images."
        )

    if (
        len(other_images)
        < UNKNOWN_SOURCE_TARGETS[
            "COCO Other Filtered Negatives"
        ]
    ):
        raise RuntimeError(
            "COCO does not contain enough "
            "other filtered negative images."
        )

    return (
        image_folder,
        human_images,
        other_images,
    )


print(
    "Locating Unknown source images..."
)

inat_source_root = (
    unknown_original_dirs[
        "iNaturalist Non-Target Taxa"
    ]
)

inat_images = get_image_files(
    inat_source_root
)

(
    afhq_source_root,
    afhq_images,
) = get_afhq_wild_images()

(
    ninco_source_root,
    ninco_images,
) = get_ninco_images()

(
    imagenet_o_source_root,
    imagenet_o_images,
) = get_imagenet_o_images()

(
    coco_source_root,
    coco_human_images,
    coco_other_images,
) = get_coco_filtered_images()

(
    places_source_root,
    places_images,
) = get_places365_images()

(
    food_source_root,
    food_images,
) = get_food101_images()


unknown_source_data = {
    "iNaturalist Non-Target Taxa": {
        "source_root": (
            inat_source_root
        ),
        "images": inat_images,
    },
    "AFHQ-v2 Wild": {
        "source_root": (
            afhq_source_root
        ),
        "images": afhq_images,
    },
    "NINCO": {
        "source_root": (
            ninco_source_root
        ),
        "images": ninco_images,
    },
    "ImageNet-O": {
        "source_root": (
            imagenet_o_source_root
        ),
        "images": imagenet_o_images,
    },
    "COCO Humans": {
        "source_root": (
            coco_source_root
        ),
        "images": (
            coco_human_images
        ),
    },
    "COCO Other Filtered Negatives": {
        "source_root": (
            coco_source_root
        ),
        "images": (
            coco_other_images
        ),
    },
    "Places365": {
        "source_root": (
            places_source_root
        ),
        "images": places_images,
    },
    "Food-101": {
        "source_root": (
            food_source_root
        ),
        "images": food_images,
    },
}


print()
print(
    "Candidate Unknown images found:"
)

for source_name, source_data in (
    unknown_source_data.items()
):
    print(
        f"  {source_name}: "
        f"{len(source_data['images']):,}"
    )

print()


seen_hashes = {}


def load_seen_hashes(
    folder,
    source_name,
):
    folder = Path(
        folder
    )

    image_paths = get_image_files(
        folder
    )

    for image_path in tqdm(
        image_paths,
        desc=f"Hashing {source_name}",
        unit="image",
    ):
        try:
            image_hash = (
                get_exact_image_hash(
                    image_path
                )
            )

            if image_hash not in seen_hashes:
                seen_hashes[
                    image_hash
                ] = {
                    "original_path": (
                        image_path
                    ),
                    "source_root": (
                        folder
                    ),
                    "source_name": (
                        source_name
                    ),
                }

        except Exception:
            continue


if CHECK_TARGET_CLASS_DUPLICATES:
    print(
        "Loading Cat, Dog, and Bird hashes..."
    )

    for target_class in [
        "Cat",
        "Dog",
        "Bird",
    ]:
        class_folder = (
            auto_cleaned_train_val_dir
            / target_class
        )

        if not class_folder.exists():
            raise FileNotFoundError(
                f"{target_class} Auto Cleaned "
                "Train and Validation folder "
                "was not found."
            )

        load_seen_hashes(
            class_folder,
            (
                f"{target_class} "
                "Train and Validation"
            ),
        )

print()


unknown_statistics = {
    source_name: {
        "candidates": 0,
        "accepted": 0,
        "rejected": 0,
        "reasons": (
            defaultdict(int)
        ),
    }
    for source_name
    in UNKNOWN_SOURCE_TARGETS
}


def reject_unknown_image(
    source_name,
    image_path,
    source_root,
    reason,
):
    rejection_root = (
        unknown_rejected_source_dirs[
            source_name
        ]
        / reason
    )

    try:
        copy_rejected_image(
            image_path,
            source_root,
            rejection_root,
        )

    except Exception:
        pass

    unknown_statistics[
        source_name
    ]["rejected"] += 1

    unknown_statistics[
        source_name
    ]["reasons"][reason] += 1


def process_unknown_image(
    source_name,
    image_path,
    source_root,
):
    is_valid, reason = check_image(
        image_path
    )

    if not is_valid:
        reject_unknown_image(
            source_name,
            image_path,
            source_root,
            reason,
        )

        return False

    try:
        image_hash = (
            get_exact_image_hash(
                image_path
            )
        )

    except Exception:
        reject_unknown_image(
            source_name,
            image_path,
            source_root,
            "hash_failed",
        )

        return False

    if image_hash in seen_hashes:
        kept_record = (
            seen_hashes[
                image_hash
            ]
        )

        group_dir = (
            unknown_rejected_source_dirs[
                source_name
            ]
            / "duplicate"
            / (
                "duplicate_"
                f"{image_hash[:16]}"
            )
        )

        try:
            copy_duplicate_group(
                image_path,
                source_root,
                source_name,
                kept_record,
                group_dir,
            )

        except Exception:
            reject_unknown_image(
                source_name,
                image_path,
                source_root,
                "copy_failed",
            )

            return False

        unknown_statistics[
            source_name
        ]["rejected"] += 1

        unknown_statistics[
            source_name
        ]["reasons"]["duplicate"] += 1

        return False

    try:
        copy_preserving_structure(
            image_path,
            source_root,
            unknown_output_dirs[
                source_name
            ],
        )

    except Exception:
        reject_unknown_image(
            source_name,
            image_path,
            source_root,
            "copy_failed",
        )

        return False

    seen_hashes[
        image_hash
    ] = {
        "original_path": (
            image_path
        ),
        "source_root": (
            source_root
        ),
        "source_name": (
            source_name
        ),
    }

    unknown_statistics[
        source_name
    ]["accepted"] += 1

    return True


print(
    "Automatically cleaning external "
    "Unknown sources..."
)


external_source_names = [
    "iNaturalist Non-Target Taxa",
    "AFHQ-v2 Wild",
    "NINCO",
    "ImageNet-O",
    "COCO Humans",
    "COCO Other Filtered Negatives",
    "Places365",
    "Food-101",
]


for source_index, source_name in enumerate(
    external_source_names,
    start=1,
):
    source_target = (
        UNKNOWN_SOURCE_TARGETS[
            source_name
        ]
    )

    source_root = (
        unknown_source_data[
            source_name
        ]["source_root"]
    )

    source_images = (
        unknown_shuffled_paths(
            unknown_source_data[
                source_name
            ]["images"],
            (
                RANDOM_SEED
                + source_index
            ),
        )
    )

    unknown_statistics[
        source_name
    ]["candidates"] = len(
        source_images
    )

    print()
    print(
        f"Cleaning {source_name}..."
    )

    print(
        f"Target: "
        f"{source_target:,}"
    )

    accepted_count = 0

    for image_path in tqdm(
        source_images,
        desc=source_name,
        unit="image",
    ):
        if (
            accepted_count
            >= source_target
        ):
            break

        accepted = (
            process_unknown_image(
                source_name,
                image_path,
                source_root,
            )
        )

        if accepted:
            accepted_count += 1

    if accepted_count < source_target:
        raise RuntimeError(
            f"Only {accepted_count:,} "
            f"{source_name} images were "
            f"accepted. The target was "
            f"{source_target:,}."
        )


def save_unknown_corruption(
    source_path,
    output_path,
    corruption_type,
    seed,
):
    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    rng = random.Random(
        seed
    )

    np_rng = (
        np.random.default_rng(
            seed
        )
    )

    with Image.open(
        source_path
    ) as image:
        image = (
            ImageOps.exif_transpose(
                image
            )
        )

        image = image.convert(
            "RGB"
        )

        if corruption_type == "blur":
            image = image.filter(
                ImageFilter.GaussianBlur(
                    radius=rng.uniform(
                        1.2,
                        3.0,
                    )
                )
            )

            image.save(
                output_path,
                format="JPEG",
                quality=90,
            )

        elif corruption_type == "darken":
            image = (
                ImageEnhance.Brightness(
                    image
                ).enhance(
                    rng.uniform(
                        0.30,
                        0.65,
                    )
                )
            )

            image.save(
                output_path,
                format="JPEG",
                quality=90,
            )

        elif corruption_type == "compression":
            image.save(
                output_path,
                format="JPEG",
                quality=rng.randint(
                    20,
                    45,
                ),
            )

        elif corruption_type == "noise":
            image_array = np.asarray(
                image,
                dtype=np.float32,
            )

            noise = np_rng.normal(
                loc=0.0,
                scale=rng.uniform(
                    12.0,
                    28.0,
                ),
                size=image_array.shape,
            )

            noisy_array = np.clip(
                image_array + noise,
                0,
                255,
            ).astype(
                np.uint8
            )

            noisy_image = (
                Image.fromarray(
                    noisy_array
                )
            )

            noisy_image.save(
                output_path,
                format="JPEG",
                quality=90,
            )

        elif (
            corruption_type
            == "crop_resize"
        ):
            width, height = (
                image.size
            )

            crop_scale = rng.uniform(
                0.65,
                0.90,
            )

            crop_width = min(
                width,
                max(
                    64,
                    int(
                        width
                        * crop_scale
                    ),
                ),
            )

            crop_height = min(
                height,
                max(
                    64,
                    int(
                        height
                        * crop_scale
                    ),
                ),
            )

            left = rng.randint(
                0,
                max(
                    0,
                    width - crop_width,
                ),
            )

            top = rng.randint(
                0,
                max(
                    0,
                    height - crop_height,
                ),
            )

            image = image.crop(
                (
                    left,
                    top,
                    left + crop_width,
                    top + crop_height,
                )
            )

            image = image.resize(
                (
                    width,
                    height,
                ),
                Image.Resampling.BICUBIC,
            )

            image.save(
                output_path,
                format="JPEG",
                quality=90,
            )

        else:
            raise ValueError(
                "Unknown corruption type: "
                f"{corruption_type}"
            )


derived_source_name = (
    "Derived Unknown Corruptions"
)

derived_source_root = (
    unknown_original_dirs[
        derived_source_name
    ]
)

derived_target = (
    UNKNOWN_SOURCE_TARGETS[
        derived_source_name
    ]
)


external_unknown_images = []

for source_name in (
    external_source_names
):
    external_unknown_images.extend(
        get_image_files(
            unknown_output_dirs[
                source_name
            ]
        )
    )


external_unknown_images = (
    unknown_shuffled_paths(
        external_unknown_images,
        RANDOM_SEED + 100,
    )
)


if not external_unknown_images:
    raise RuntimeError(
        "No accepted external Unknown images "
        "are available for derived corruptions."
    )


corruption_types = [
    "blur",
    "darken",
    "compression",
    "noise",
    "crop_resize",
]


print()
print(
    "Creating Derived Unknown Corruptions..."
)

print(
    f"Base Unknown images available: "
    f"{len(external_unknown_images):,}"
)

print(
    f"Derived target: "
    f"{derived_target:,}"
)


derived_accepted_count = 0
derived_attempt_count = 0

maximum_derived_attempts = (
    derived_target * 4
)


with tqdm(
    total=derived_target,
    desc=derived_source_name,
    unit="image",
) as progress_bar:
    while (
        derived_accepted_count
        < derived_target
        and derived_attempt_count
        < maximum_derived_attempts
    ):
        base_image = (
            external_unknown_images[
                derived_attempt_count
                % len(
                    external_unknown_images
                )
            ]
        )

        corruption_type = (
            corruption_types[
                derived_attempt_count
                % len(
                    corruption_types
                )
            ]
        )

        derived_number = (
            derived_attempt_count + 1
        )

        derived_name = (
            f"derived_"
            f"{derived_number:06d}_"
            f"{corruption_type}.jpg"
        )

        derived_path = (
            derived_source_root
            / corruption_type
            / derived_name
        )

        if not derived_path.exists():
            try:
                save_unknown_corruption(
                    base_image,
                    derived_path,
                    corruption_type,
                    (
                        RANDOM_SEED
                        + 1000
                        + derived_attempt_count
                    ),
                )

            except Exception:
                unknown_statistics[
                    derived_source_name
                ]["rejected"] += 1

                unknown_statistics[
                    derived_source_name
                ]["reasons"][
                    "generation_failed"
                ] += 1

                derived_attempt_count += 1

                continue

        unknown_statistics[
            derived_source_name
        ]["candidates"] += 1

        accepted = (
            process_unknown_image(
                derived_source_name,
                derived_path,
                derived_source_root,
            )
        )

        if accepted:
            derived_accepted_count += 1

            progress_bar.update(
                1
            )

        derived_attempt_count += 1


if (
    derived_accepted_count
    < derived_target
):
    raise RuntimeError(
        f"Only {derived_accepted_count:,} "
        "Derived Unknown Corruptions were "
        f"accepted. The target was "
        f"{derived_target:,}."
    )


print()
print(
    "Unknown train/validation automatic "
    "cleaning is complete."
)
print()


total_candidates = 0
total_accepted = 0
total_rejected = 0


for source_name in (
    UNKNOWN_SOURCE_TARGETS
):
    source_stats = (
        unknown_statistics[
            source_name
        ]
    )

    output_count = len(
        get_image_files(
            unknown_output_dirs[
                source_name
            ]
        )
    )

    total_candidates += (
        source_stats[
            "candidates"
        ]
    )

    total_accepted += (
        output_count
    )

    total_rejected += (
        source_stats[
            "rejected"
        ]
    )

    print(
        source_name
    )

    print(
        f"  Candidate images: "
        f"{source_stats['candidates']:,}"
    )

    print(
        f"  Accepted images:  "
        f"{output_count:,}"
    )

    print(
        f"  Target images:    "
        f"{UNKNOWN_SOURCE_TARGETS[source_name]:,}"
    )

    print(
        f"  Rejected images:  "
        f"{source_stats['rejected']:,}"
    )

    if source_stats["reasons"]:
        print(
            "  Rejection reasons:"
        )

        for reason, count in sorted(
            source_stats[
                "reasons"
            ].items()
        ):
            print(
                f"    {reason}: "
                f"{count:,}"
            )

    print()


print(
    "Total"
)

print(
    f"  Candidate images: "
    f"{total_candidates:,}"
)

print(
    f"  Accepted images:  "
    f"{total_accepted:,}"
)

print(
    f"  Rejected images:  "
    f"{total_rejected:,}"
)

print(
    f"  Automatic target: "
    f"{UNKNOWN_AUTO_CLEANED_TARGET:,}"
)

print(
    f"  Final target:     "
    f"{UNKNOWN_FINAL_TARGET:,}"
)

print(
    f"  Buffer:           "
    f"{UNKNOWN_BUFFER_SIZE:,}"
)

print()


if (
    total_accepted
    != UNKNOWN_AUTO_CLEANED_TARGET
):
    raise RuntimeError(
        "The automatically cleaned Unknown pool "
        f"contains {total_accepted:,} images "
        f"instead of "
        f"{UNKNOWN_AUTO_CLEANED_TARGET:,}."
    )


print(
    "Auto Cleaned Unknown images saved in:"
)

print(
    show_path(
        unknown_auto_cleaned_dir
    )
)

print()

print(
    "Automatically rejected Unknown "
    "images saved in:"
)

print(
    show_path(
        unknown_rejected_dir
    )
)

print()

print(
    f"After manual cleaning, retain the best "
    f"{UNKNOWN_FINAL_TARGET:,} Unknown images."
)

Unknown train/validation paths are ready.
Original directory:     Model Building\Datasets\Original\Train and Validation
Auto Cleaned directory: Model Building\Datasets\Auto Cleaned\Train and Validation\Unknown
Rejected directory:     Model Building\Datasets\Rejected\Automatic\Train and Validation\Unknown

Unknown source targets:
  iNaturalist Non-Target Taxa: 10,350
  AFHQ-v2 Wild: 4,500
  NINCO: 3,850
  ImageNet-O: 1,650
  COCO Humans: 8,800
  COCO Other Filtered Negatives: 4,400
  Places365: 7,700
  Food-101: 8,250
  Derived Unknown Corruptions: 5,500

Automatically cleaned target: 55,000
Final target:                 50,000
Manual-cleaning buffer:       5,000

NINCO:
Already downloaded: NINCO_all.tar.gz
ImageNet-O:
Already downloaded: imagenet-o.tar
COCO 2017 Images:
Already downloaded: train2017.zip
COCO 2017 Annotations:
Already downloaded: annotations_trainval2017.zip
Places365:
Already downloaded: val_256.tar
Food-101:
Already downloaded: food-101.tar.gz

Extracting Unknown sour

Mustelidae: 100%|██████████| 276/276 [06:00<00:00,  1.31s/image]


Mustelidae: 450 candidates.


Echinodermata: 100%|██████████| 700/700 [17:51<00:00,  1.53s/image]


Echinodermata: 700 candidates.

Locating Unknown source images...
Reading COCO annotations...


...


Cleaning Food-101...
Target: 8,250


Food-101:   8%|▊         | 8260/101000 [01:20<15:04, 102.52image/s]



Creating Derived Unknown Corruptions...
Base Unknown images available: 49,500
Derived target: 5,500


Derived Unknown Corruptions: 100%|██████████| 5500/5500 [03:12<00:00, 28.58image/s]



Unknown train/validation automatic cleaning is complete.

iNaturalist Non-Target Taxa
  Candidate images: 12,500
  Accepted images:  10,350
  Target images:    10,350
  Rejected images:  9
  Rejection reasons:
    duplicate: 9

AFHQ-v2 Wild
  Candidate images: 5,076
  Accepted images:  4,500
  Target images:    4,500
  Rejected images:  1
  Rejection reasons:
    duplicate: 1

NINCO
  Candidate images: 15,393
  Accepted images:  3,850
  Target images:    3,850
  Rejected images:  458
  Rejection reasons:
    blank_image: 450
    duplicate: 7
    low_resolution: 1

ImageNet-O
  Candidate images: 2,000
  Accepted images:  1,650
  Target images:    1,650
  Rejected images:  92
  Rejection reasons:
    duplicate: 88
    low_resolution: 4

COCO Humans
  Candidate images: 60,669
  Accepted images:  8,800
  Target images:    8,800
  Rejected images:  1
  Rejection reasons:
    low_resolution: 1

COCO Other Filtered Negatives
  Candidate images: 46,223
  Accepted images:  4,400
  Target image

## 2. Downloading and Automatically Cleaning Test Images

This section downloads and automatically cleans the Cat, Dog, Bird, and Unknown images intended for the official Test set.

Raw Open Images V7 images and metadata are stored under:

`Model Building/Datasets/Original/Test/Open Images V7`

Automatically cleaned Test images are stored under:

`Model Building/Datasets/Auto Cleaned/Test`

Automatically rejected Test images are copied into class- and reason-based folders under:

`Model Building/Datasets/Rejected/Automatic/Test`

Open Images V7 is used as the source for the official Test images. Candidates are collected from the validation split first, followed by the test split and then the train split where additional images are needed.

Open Images is not used in the rebuilt Train/Validation pools, so accepted images from these splits remain separate from the images used for model training and validation.

The automatic-cleaning process rejects failed downloads, broken or unreadable files, invalid image dimensions, images with a width or height below 64 pixels, completely blank images, exact duplicates, and images that cannot be processed or copied correctly.

Exact duplicates are also checked against the automatically cleaned Train/Validation images.

Accepted images are copied unchanged while preserving their original filenames and extensions. Content-based decisions are left for the later manual-cleaning stage.


### 2.1 Downloading and Automatically Cleaning Cat Test Images

This section downloads and automatically cleans the Cat images intended for the official Test set.

Raw Cat Test images from Open Images V7 are stored under:

`Model Building/Datasets/Original/Test/Open Images V7/Cat`

Automatically cleaned Cat Test images are stored under:

`Model Building/Datasets/Auto Cleaned/Test/Cat`

Automatically rejected Cat images are copied into reason-based folders under:

`Model Building/Datasets/Rejected/Automatic/Test/Cat`

Open Images V7 is used as the source for the official Cat Test images. Cat candidates are identified using human-verified image labels, Cat bounding-box annotations, and suitable machine-generated image labels.

Candidates are collected from the Open Images validation split first, followed by the test split and then the train split until the automatic-cleaning target is reached.

The Cat Test source link is:

```text
Open Images V7:
https://storage.googleapis.com/openimages/web/index.html
```

The automatic-cleaning process retains up to **3,000 Cat images**. This provides a 500-image buffer for the later manual-cleaning stage, after which the best **2,500 Cat images** will be retained.

The same universal automatic-cleaning rules used for the Train/Validation images are applied to the Cat Test images.


In [4]:
import csv
import pickle
import random
import time

try:
    import boto3

    from botocore import UNSIGNED
    from botocore.config import Config

except ModuleNotFoundError:
    install_package("boto3")

    import boto3

    from botocore import UNSIGNED
    from botocore.config import Config


CAT_TEST_FINAL_TARGET = 2_500
CAT_TEST_AUTOMATIC_TARGET = 3_000
CAT_TEST_BUFFER = (
    CAT_TEST_AUTOMATIC_TARGET
    - CAT_TEST_FINAL_TARGET
)

OPEN_IMAGES_RANDOM_SEED = 42
DOWNLOAD_RETRIES = 3

RESET_CAT_TEST_OUTPUTS = False


original_test_dir = (
    datasets_dir
    / "Original"
    / "Test"
)

open_images_original_dir = (
    original_test_dir
    / "Open Images V7"
)

open_images_metadata_dir = (
    open_images_original_dir
    / "Metadata"
)

cat_original_test_dirs = {
    "test": (
        open_images_original_dir
        / "Test"
        / "Cat"
    ),
    "train": (
        open_images_original_dir
        / "Train"
        / "Cat"
    ),
}

cat_auto_cleaned_test_dir = (
    datasets_dir
    / "Auto Cleaned"
    / "Test"
    / "Cat"
)

cat_rejected_test_dir = (
    datasets_dir
    / "Rejected"
    / "Automatic"
    / "Test"
    / "Cat"
)

auto_cleaned_train_val_dir = (
    datasets_dir
    / "Auto Cleaned"
    / "Train and Validation"
)

train_val_hash_cache_path = (
    auto_cleaned_train_val_dir
    / ".exact_hash_cache.pkl"
)

old_validation_metadata_dir = (
    original_test_dir
    / "Open Images V7 Validation"
    / "Metadata"
)

class_descriptions_path = (
    open_images_metadata_dir
    / "oidv7-class-descriptions.csv"
)

cat_candidate_manifest_path = (
    open_images_metadata_dir
    / "cat_test_candidate_manifest.csv"
)


split_metadata = {
    "test": {
        "display_name": "Test",
        "url": (
            "https://storage.googleapis.com/openimages/v5/"
            "test-annotations-human-imagelabels-boxable.csv"
        ),
        "path": (
            open_images_metadata_dir
            / "test-annotations-human-imagelabels-boxable.csv"
        ),
        "random_seed": OPEN_IMAGES_RANDOM_SEED + 1,
    },
    "train": {
        "display_name": "Train",
        "url": (
            "https://storage.googleapis.com/openimages/v5/"
            "train-annotations-human-imagelabels-boxable.csv"
        ),
        "path": (
            open_images_metadata_dir
            / "train-annotations-human-imagelabels-boxable.csv"
        ),
        "random_seed": OPEN_IMAGES_RANDOM_SEED + 2,
    },
}


if RESET_CAT_TEST_OUTPUTS:
    reset_folder(cat_auto_cleaned_test_dir)
    reset_folder(cat_rejected_test_dir)


for folder in [
    original_test_dir,
    open_images_original_dir,
    open_images_metadata_dir,
    cat_auto_cleaned_test_dir,
    cat_rejected_test_dir,
    *cat_original_test_dirs.values(),
]:
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )


old_class_descriptions_path = (
    old_validation_metadata_dir
    / "oidv7-class-descriptions.csv"
)

if (
    not class_descriptions_path.exists()
    and old_class_descriptions_path.exists()
):
    shutil.copy2(
        old_class_descriptions_path,
        class_descriptions_path,
    )


if not class_descriptions_path.exists():
    download_file(
        (
            "https://storage.googleapis.com/openimages/v7/"
            "oidv7-class-descriptions.csv"
        ),
        class_descriptions_path,
    )


print("Cat Test continuation paths are ready.")

print(
    "Original Open Images directory:   "
    f"{show_path(open_images_original_dir)}"
)

print(
    "Auto Cleaned Cat Test directory:  "
    f"{show_path(cat_auto_cleaned_test_dir)}"
)

print(
    "Automatically rejected directory: "
    f"{show_path(cat_rejected_test_dir)}"
)

print(
    "Cat Test automatic target:        "
    f"{CAT_TEST_AUTOMATIC_TARGET:,}"
)

print(
    "Cat Test final target:            "
    f"{CAT_TEST_FINAL_TARGET:,}"
)

print(
    "Manual-cleaning buffer:           "
    f"{CAT_TEST_BUFFER:,}"
)

print()


def find_open_images_label_id(
    class_descriptions_file,
    class_name,
):
    with class_descriptions_file.open(
        "r",
        encoding="utf-8-sig",
        newline="",
    ) as file:
        reader = csv.reader(file)

        for row in reader:
            if len(row) < 2:
                continue

            label_id = row[0].strip()
            display_name = row[1].strip()

            if (
                display_name.casefold()
                == class_name.casefold()
            ):
                return label_id

    raise ValueError(
        f"Open Images label was not found: "
        f"{class_name}"
    )


cat_label_id = find_open_images_label_id(
    class_descriptions_path,
    "Cat",
)

print(
    f"Open Images Cat label: "
    f"{cat_label_id}"
)

print()


if not train_val_hash_cache_path.exists():
    raise FileNotFoundError(
        "The Train/Validation exact-hash cache was not found:\n"
        f"{show_path(train_val_hash_cache_path)}\n\n"
        "Run the previous Cat Test hashing code once before "
        "running this continuation cell."
    )


with train_val_hash_cache_path.open(
    "rb"
) as file:
    cached_data = pickle.load(file)


train_val_hash_records = cached_data.get(
    "hash_records"
)

if not isinstance(
    train_val_hash_records,
    dict,
):
    raise RuntimeError(
        "The Train/Validation exact-hash cache "
        "does not contain valid hash records."
    )


print(
    "Loaded cached Train/Validation exact hashes."
)

print(
    "Train/Validation unique hashes loaded: "
    f"{len(train_val_hash_records):,}"
)

print()


existing_cat_test_images = get_image_files(
    cat_auto_cleaned_test_dir
)

seen_cat_test_hashes = {}
accepted_image_ids = set()


print(
    "Loading existing Auto Cleaned Cat Test hashes..."
)

for image_path in tqdm(
    existing_cat_test_images,
    desc="Hashing existing Cat Test",
    unit="image",
):
    try:
        exact_hash = get_exact_image_hash(
            image_path
        )

    except Exception:
        continue

    if exact_hash not in seen_cat_test_hashes:
        seen_cat_test_hashes[exact_hash] = {
            "source_name": "Cat Test",
            "source_root": (
                cat_auto_cleaned_test_dir
            ),
            "original_path": image_path,
            "cleaned_path": image_path,
        }

    accepted_image_ids.add(
        image_path.stem
    )


existing_cleaned_count = len(
    existing_cat_test_images
)

print(
    "Existing Auto Cleaned Cat Test images: "
    f"{existing_cleaned_count:,}"
)

print(
    "Existing unique Cat Test hashes:       "
    f"{len(seen_cat_test_hashes):,}"
)

print()


s3_resource = boto3.resource(
    "s3",
    config=Config(
        signature_version=UNSIGNED,
        retries={
            "max_attempts": 5,
            "mode": "standard",
        },
    ),
)

open_images_bucket = s3_resource.Bucket(
    "open-images-dataset"
)


def download_open_images_image(
    split_name,
    image_id,
    output_path,
):
    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if (
        output_path.exists()
        and output_path.stat().st_size > 0
    ):
        return True, False, None

    if output_path.exists():
        output_path.unlink()

    temporary_path = output_path.with_suffix(
        output_path.suffix + ".part"
    )

    if temporary_path.exists():
        temporary_path.unlink()

    last_error = None

    for attempt in range(
        1,
        DOWNLOAD_RETRIES + 1,
    ):
        try:
            open_images_bucket.download_file(
                f"{split_name}/{image_id}.jpg",
                str(temporary_path),
            )

            temporary_path.replace(
                output_path
            )

            return True, True, None

        except Exception as error:
            last_error = error

            if temporary_path.exists():
                temporary_path.unlink()

            if attempt < DOWNLOAD_RETRIES:
                time.sleep(attempt)

    return False, False, str(last_error)


def save_download_failure(
    split_name,
    image_id,
    error_message,
):
    failure_dir = (
        cat_rejected_test_dir
        / "download_failed"
        / split_name.title()
    )

    failure_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    failure_path = (
        failure_dir
        / f"{image_id}.txt"
    )

    failure_path.write_text(
        (
            f"Image ID: {image_id}\n"
            f"Split: {split_name}\n"
            f"Error: {error_message}\n"
        ),
        encoding="utf-8",
    )


def load_positive_cat_ids(
    annotation_path,
    display_name,
):
    positive_ids = set()

    with annotation_path.open(
        "r",
        encoding="utf-8-sig",
        newline="",
    ) as file:
        reader = csv.DictReader(file)

        for row in tqdm(
            reader,
            desc=(
                f"Reading {display_name} "
                "Cat labels"
            ),
            unit="annotation",
        ):
            if (
                row.get("LabelName")
                != cat_label_id
            ):
                continue

            try:
                confidence = float(
                    row.get(
                        "Confidence",
                        0,
                    )
                )

            except (
                TypeError,
                ValueError,
            ):
                continue

            if confidence < 1.0:
                continue

            image_id = row.get(
                "ImageID",
                "",
            ).strip()

            if image_id:
                positive_ids.add(
                    image_id
                )

    return sorted(
        positive_ids
    )


def save_rejected_test_image(
    image_path,
    split_root,
    split_name,
    reason,
):
    return copy_rejected_image(
        image_path,
        split_root,
        (
            cat_rejected_test_dir
            / reason
            / split_name.title()
        ),
    )


def save_train_val_duplicate(
    image_path,
    split_root,
    split_name,
    exact_hash,
):
    kept_relative_path = Path(
        train_val_hash_records[
            exact_hash
        ]
    )

    kept_path = (
        auto_cleaned_train_val_dir
        / kept_relative_path
    )

    kept_record = {
        "source_name": (
            "Train and Validation"
        ),
        "source_root": (
            auto_cleaned_train_val_dir
        ),
        "original_path": kept_path,
        "cleaned_path": kept_path,
    }

    group_dir = (
        cat_rejected_test_dir
        / "exact_duplicate_train_validation"
        / split_name.title()
        / (
            "duplicate_group_"
            f"{exact_hash[:16]}"
        )
    )

    copy_duplicate_group(
        duplicate_path=image_path,
        duplicate_source_root=split_root,
        duplicate_source_name=(
            f"Open Images {split_name.title()}"
        ),
        kept_record=kept_record,
        group_dir=group_dir,
    )


def save_cat_test_duplicate(
    image_path,
    split_root,
    split_name,
    exact_hash,
):
    group_dir = (
        cat_rejected_test_dir
        / "exact_duplicate"
        / split_name.title()
        / (
            "duplicate_group_"
            f"{exact_hash[:16]}"
        )
    )

    copy_duplicate_group(
        duplicate_path=image_path,
        duplicate_source_root=split_root,
        duplicate_source_name=(
            f"Open Images {split_name.title()}"
        ),
        kept_record=(
            seen_cat_test_hashes[
                exact_hash
            ]
        ),
        group_dir=group_dir,
    )


def process_cat_split(
    split_name,
    candidate_ids,
):
    split_info = split_metadata[
        split_name
    ]

    display_name = split_info[
        "display_name"
    ]

    source_root = (
        cat_original_test_dirs[
            split_name
        ]
    )

    random_generator = random.Random(
        split_info["random_seed"]
    )

    candidate_ids = list(
        candidate_ids
    )

    random_generator.shuffle(
        candidate_ids
    )

    stats = {
        "candidate_ids": len(
            candidate_ids
        ),
        "processed": 0,
        "newly_downloaded": 0,
        "accepted": 0,
        "rejected": 0,
        "already_accepted": 0,
        "rejection_reasons": (
            defaultdict(int)
        ),
    }

    print(
        "Downloading and automatically cleaning "
        f"Open Images {display_name} Cat images..."
    )

    for image_id in tqdm(
        candidate_ids,
        desc=(
            f"Open Images {display_name} Cat"
        ),
        unit="image",
    ):
        current_cleaned_count = len(
            get_image_files(
                cat_auto_cleaned_test_dir
            )
        )

        if (
            current_cleaned_count
            >= CAT_TEST_AUTOMATIC_TARGET
        ):
            break

        if image_id in accepted_image_ids:
            stats["already_accepted"] += 1
            continue

        image_path = (
            source_root
            / f"{image_id}.jpg"
        )

        (
            download_successful,
            newly_downloaded,
            download_error,
        ) = download_open_images_image(
            split_name,
            image_id,
            image_path,
        )

        stats["processed"] += 1

        if newly_downloaded:
            stats[
                "newly_downloaded"
            ] += 1

        if not download_successful:
            reason = "download_failed"

            stats["rejected"] += 1
            stats[
                "rejection_reasons"
            ][reason] += 1

            save_download_failure(
                split_name,
                image_id,
                download_error,
            )

            continue

        is_valid, reason = check_image(
            image_path
        )

        if not is_valid:
            stats["rejected"] += 1
            stats[
                "rejection_reasons"
            ][reason] += 1

            save_rejected_test_image(
                image_path,
                source_root,
                split_name,
                reason,
            )

            continue

        try:
            exact_hash = get_exact_image_hash(
                image_path
            )

        except Exception:
            reason = "hash_failed"

            stats["rejected"] += 1
            stats[
                "rejection_reasons"
            ][reason] += 1

            save_rejected_test_image(
                image_path,
                source_root,
                split_name,
                reason,
            )

            continue

        if (
            exact_hash
            in train_val_hash_records
        ):
            reason = (
                "exact_duplicate_"
                "train_validation"
            )

            stats["rejected"] += 1
            stats[
                "rejection_reasons"
            ][reason] += 1

            save_train_val_duplicate(
                image_path,
                source_root,
                split_name,
                exact_hash,
            )

            continue

        if exact_hash in seen_cat_test_hashes:
            reason = "exact_duplicate"

            stats["rejected"] += 1
            stats[
                "rejection_reasons"
            ][reason] += 1

            save_cat_test_duplicate(
                image_path,
                source_root,
                split_name,
                exact_hash,
            )

            continue

        try:
            cleaned_path = (
                copy_preserving_structure(
                    image_path,
                    source_root,
                    cat_auto_cleaned_test_dir,
                )
            )

        except Exception:
            reason = "copy_failed"

            stats["rejected"] += 1
            stats[
                "rejection_reasons"
            ][reason] += 1

            save_rejected_test_image(
                image_path,
                source_root,
                split_name,
                reason,
            )

            continue

        seen_cat_test_hashes[
            exact_hash
        ] = {
            "source_name": (
                f"Open Images "
                f"{display_name} Cat"
            ),
            "source_root": source_root,
            "original_path": image_path,
            "cleaned_path": cleaned_path,
        }

        accepted_image_ids.add(
            image_id
        )

        stats["accepted"] += 1

    return stats


candidate_manifest_rows = []

for image_path in existing_cat_test_images:
    candidate_manifest_rows.append(
        {
            "split": "validation",
            "image_id": image_path.stem,
            "candidate_source": (
                "existing_auto_cleaned"
            ),
            "confidence": 1.0,
        }
    )


split_results = {}


for split_name in [
    "test",
    "train",
]:
    current_cleaned_count = len(
        get_image_files(
            cat_auto_cleaned_test_dir
        )
    )

    if (
        current_cleaned_count
        >= CAT_TEST_AUTOMATIC_TARGET
    ):
        break

    split_info = split_metadata[
        split_name
    ]

    print(
        "Preparing Open Images "
        f"{split_info['display_name']} "
        "Cat candidates..."
    )

    download_file(
        split_info["url"],
        split_info["path"],
    )

    candidate_ids = load_positive_cat_ids(
        split_info["path"],
        split_info["display_name"],
    )

    print()

    print(
        "Human-verified positive Cat "
        f"images in {split_info['display_name']}: "
        f"{len(candidate_ids):,}"
    )

    print()

    for image_id in candidate_ids:
        candidate_manifest_rows.append(
            {
                "split": split_name,
                "image_id": image_id,
                "candidate_source": (
                    "human_verified"
                ),
                "confidence": 1.0,
            }
        )

    split_results[split_name] = (
        process_cat_split(
            split_name,
            candidate_ids,
        )
    )

    print()


with cat_candidate_manifest_path.open(
    "w",
    encoding="utf-8",
    newline="",
) as file:
    writer = csv.DictWriter(
        file,
        fieldnames=[
            "split",
            "image_id",
            "candidate_source",
            "confidence",
        ],
    )

    writer.writeheader()
    writer.writerows(
        candidate_manifest_rows
    )


final_cleaned_count = len(
    get_image_files(
        cat_auto_cleaned_test_dir
    )
)

newly_accepted_total = sum(
    result["accepted"]
    for result in split_results.values()
)

newly_rejected_total = sum(
    result["rejected"]
    for result in split_results.values()
)


print(
    "Cat Test automatic cleaning is complete."
)

print()


print("Existing Validation results")

print(
    "  Auto Cleaned images before continuation: "
    f"{existing_cleaned_count:,}"
)

print()


for split_name, result in (
    split_results.items()
):
    display_name = split_metadata[
        split_name
    ]["display_name"]

    print(
        f"Open Images {display_name}"
    )

    print(
        "  Candidate image IDs:    "
        f"{result['candidate_ids']:,}"
    )

    print(
        "  Processed images:       "
        f"{result['processed']:,}"
    )

    print(
        "  Newly downloaded:       "
        f"{result['newly_downloaded']:,}"
    )

    print(
        "  Auto Cleaned images:    "
        f"{result['accepted']:,}"
    )

    print(
        "  Automatically rejected: "
        f"{result['rejected']:,}"
    )

    if result["rejection_reasons"]:
        print("  Rejection reasons:")

        for reason, count in sorted(
            result[
                "rejection_reasons"
            ].items()
        ):
            print(
                f"    {reason}: "
                f"{count:,}"
            )

    print()


print("Total")

print(
    "  Existing Auto Cleaned images: "
    f"{existing_cleaned_count:,}"
)

print(
    "  Newly Auto Cleaned images:    "
    f"{newly_accepted_total:,}"
)

print(
    "  Automatically rejected:       "
    f"{newly_rejected_total:,}"
)

print(
    "  Current Auto Cleaned images:   "
    f"{final_cleaned_count:,}"
)

print(
    "  Automatic target:             "
    f"{CAT_TEST_AUTOMATIC_TARGET:,}"
)

print(
    "  Final target:                 "
    f"{CAT_TEST_FINAL_TARGET:,}"
)

print(
    "  Manual-cleaning buffer:       "
    f"{CAT_TEST_BUFFER:,}"
)

print()


if (
    final_cleaned_count
    >= CAT_TEST_AUTOMATIC_TARGET
):
    print(
        "The Cat Test automatic-cleaning "
        "target was reached."
    )

else:
    print(
        "The Cat Test automatic-cleaning "
        "target was not reached."
    )

    print(
        "Automatically cleaned images "
        "still needed: "
        f"{CAT_TEST_AUTOMATIC_TARGET - final_cleaned_count:,}"
    )


print()

print(
    "Auto Cleaned Cat Test images saved in: "
    f"{show_path(cat_auto_cleaned_test_dir)}"
)

print(
    "Automatically rejected Cat Test images "
    "saved in: "
    f"{show_path(cat_rejected_test_dir)}"
)

print(
    "Cat Test candidate manifest saved in: "
    f"{show_path(cat_candidate_manifest_path)}"
)

Cat Test continuation paths are ready.
Original Open Images directory:   Image Classifier Project\Model Building\Datasets\Original\Test\Open Images V7
Auto Cleaned Cat Test directory:  Image Classifier Project\Model Building\Datasets\Auto Cleaned\Test\Cat
Automatically rejected directory: Image Classifier Project\Model Building\Datasets\Rejected\Automatic\Test\Cat
Cat Test automatic target:        3,000
Cat Test final target:            2,500
Manual-cleaning buffer:           500

Open Images Cat label: /m/01yrx

Loaded cached Train/Validation exact hashes.
Train/Validation unique hashes loaded: 147,337

Loading existing Auto Cleaned Cat Test hashes...


Hashing existing Cat Test: 100%|██████████| 350/350 [00:07<00:00, 49.90image/s]


Existing Auto Cleaned Cat Test images: 350
Existing unique Cat Test hashes:       350

Preparing Open Images Test Cat candidates...
Downloading: test-annotations-human-imagelabels-boxable.csv


test-annotations-human-imagelabels-boxable.csv: 32.1MB [00:18, 1.72MB/s]                            


Saved: Image Classifier Project\Model Building\Datasets\Original\Test\Open Images V7\Metadata\test-annotations-human-imagelabels-boxable.csv


...

Saved: Image Classifier Project\Model Building\Datasets\Original\Test\Open Images V7\Metadata\train-annotations-human-imagelabels-boxable.csv


Reading Train Cat labels: 8996795annotation [00:09, 970236.39annotation/s] 



Human-verified positive Cat images in Train: 13,936



Open Images Train Cat:  12%|█▏        | 1638/13936 [56:58<7:07:48,  2.09s/image] 


Cat Test automatic cleaning is complete.

Existing Validation results
  Auto Cleaned images before continuation: 350

Open Images Test
  Candidate image IDs:    1,012
  Processed images:       1,012
  Newly downloaded:       1,012
  Auto Cleaned images:    1,012
  Automatically rejected: 0

Open Images Train
  Candidate image IDs:    13,936
  Processed images:       1,638
  Newly downloaded:       1,638
  Auto Cleaned images:    1,638
  Automatically rejected: 0

Total
  Existing Auto Cleaned images: 350
  Newly Auto Cleaned images:    2,650
  Automatically rejected:       0
  Current Auto Cleaned images:   3,000
  Automatic target:             3,000
  Final target:                 2,500
  Manual-cleaning buffer:       500

The Cat Test automatic-cleaning target was reached.

Auto Cleaned Cat Test images saved in: Image Classifier Project\Model Building\Datasets\Auto Cleaned\Test\Cat
Automatically rejected Cat Test images saved in: Image Classifier Project\Model Building\Datasets\Reje

### 2.2 Downloading and Automatically Cleaning Dog Test Images

This section downloads and automatically cleans the Dog images intended for the official Test set.

Raw Dog Test images from Open Images V7 are stored under:

`Model Building/Datasets/Original/Test/Open Images V7/Dog`

Automatically cleaned Dog Test images are stored under:

`Model Building/Datasets/Auto Cleaned/Test/Dog`

Automatically rejected Dog images are copied into reason-based folders under:

`Model Building/Datasets/Rejected/Automatic/Test/Dog`

Open Images V7 is used as the source for the official Dog Test images. Dog candidates are identified using human-verified image labels and suitable bounding-box annotations.

Candidates are collected from the Open Images validation split first, followed by the test split and then the train split until the automatic-cleaning target is reached.

The Dog Test source link is:

```text
Open Images V7:
https://storage.googleapis.com/openimages/web/index.html
```

The automatic-cleaning process retains up to **3,000 Dog images**. This provides a 500-image buffer for the later manual-cleaning stage, after which the best **2,500 Dog images** will be retained.

The same universal automatic-cleaning rules used for the Train/Validation images are applied to the Dog Test images.


In [5]:
import csv
import pickle
import random
import time

try:
    import boto3

    from botocore import UNSIGNED
    from botocore.config import Config

except ModuleNotFoundError:
    install_package("boto3")

    import boto3

    from botocore import UNSIGNED
    from botocore.config import Config


DOG_TEST_FINAL_TARGET = 2_500
DOG_TEST_AUTOMATIC_TARGET = 3_000
DOG_TEST_BUFFER = (
    DOG_TEST_AUTOMATIC_TARGET
    - DOG_TEST_FINAL_TARGET
)

OPEN_IMAGES_RANDOM_SEED = 42
DOWNLOAD_RETRIES = 3

RESET_DOG_TEST_OUTPUTS = False


original_test_dir = (
    datasets_dir
    / "Original"
    / "Test"
)

open_images_original_dir = (
    original_test_dir
    / "Open Images V7"
)

open_images_metadata_dir = (
    open_images_original_dir
    / "Metadata"
)

dog_original_test_dirs = {
    "validation": (
        open_images_original_dir
        / "Validation"
        / "Dog"
    ),
    "test": (
        open_images_original_dir
        / "Test"
        / "Dog"
    ),
    "train": (
        open_images_original_dir
        / "Train"
        / "Dog"
    ),
}

dog_auto_cleaned_test_dir = (
    datasets_dir
    / "Auto Cleaned"
    / "Test"
    / "Dog"
)

cat_auto_cleaned_test_dir = (
    datasets_dir
    / "Auto Cleaned"
    / "Test"
    / "Cat"
)

dog_rejected_test_dir = (
    datasets_dir
    / "Rejected"
    / "Automatic"
    / "Test"
    / "Dog"
)

auto_cleaned_train_val_dir = (
    datasets_dir
    / "Auto Cleaned"
    / "Train and Validation"
)

train_val_hash_cache_path = (
    auto_cleaned_train_val_dir
    / ".exact_hash_cache.pkl"
)

class_descriptions_path = (
    open_images_metadata_dir
    / "oidv7-class-descriptions.csv"
)

validation_labels_path = (
    open_images_metadata_dir
    / "oidv7-val-annotations-human-imagelabels.csv"
)

test_labels_path = (
    open_images_metadata_dir
    / "test-annotations-human-imagelabels-boxable.csv"
)

train_labels_path = (
    open_images_metadata_dir
    / "train-annotations-human-imagelabels-boxable.csv"
)

dog_candidate_manifest_path = (
    open_images_metadata_dir
    / "dog_test_candidate_manifest.csv"
)


old_validation_metadata_dir = (
    original_test_dir
    / "Open Images V7 Validation"
    / "Metadata"
)

old_class_descriptions_path = (
    old_validation_metadata_dir
    / "oidv7-class-descriptions.csv"
)

old_validation_labels_path = (
    old_validation_metadata_dir
    / "oidv7-val-annotations-human-imagelabels.csv"
)


split_metadata = {
    "validation": {
        "display_name": "Validation",
        "url": (
            "https://storage.googleapis.com/openimages/v7/"
            "oidv7-val-annotations-human-imagelabels.csv"
        ),
        "path": validation_labels_path,
        "random_seed": OPEN_IMAGES_RANDOM_SEED,
    },
    "test": {
        "display_name": "Test",
        "url": (
            "https://storage.googleapis.com/openimages/v5/"
            "test-annotations-human-imagelabels-boxable.csv"
        ),
        "path": test_labels_path,
        "random_seed": OPEN_IMAGES_RANDOM_SEED + 1,
    },
    "train": {
        "display_name": "Train",
        "url": (
            "https://storage.googleapis.com/openimages/v5/"
            "train-annotations-human-imagelabels-boxable.csv"
        ),
        "path": train_labels_path,
        "random_seed": OPEN_IMAGES_RANDOM_SEED + 2,
    },
}


if RESET_DOG_TEST_OUTPUTS:
    reset_folder(dog_auto_cleaned_test_dir)
    reset_folder(dog_rejected_test_dir)


for folder in [
    original_test_dir,
    open_images_original_dir,
    open_images_metadata_dir,
    dog_auto_cleaned_test_dir,
    dog_rejected_test_dir,
    *dog_original_test_dirs.values(),
]:
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )


if (
    not class_descriptions_path.exists()
    and old_class_descriptions_path.exists()
):
    shutil.copy2(
        old_class_descriptions_path,
        class_descriptions_path,
    )


if (
    not validation_labels_path.exists()
    and old_validation_labels_path.exists()
):
    shutil.copy2(
        old_validation_labels_path,
        validation_labels_path,
    )


if not class_descriptions_path.exists():
    download_file(
        (
            "https://storage.googleapis.com/openimages/v7/"
            "oidv7-class-descriptions.csv"
        ),
        class_descriptions_path,
    )


print("Dog Test paths are ready.")

print(
    "Original Open Images directory:   "
    f"{show_path(open_images_original_dir)}"
)

print(
    "Auto Cleaned Dog Test directory:  "
    f"{show_path(dog_auto_cleaned_test_dir)}"
)

print(
    "Automatically rejected directory: "
    f"{show_path(dog_rejected_test_dir)}"
)

print(
    "Dog Test automatic target:        "
    f"{DOG_TEST_AUTOMATIC_TARGET:,}"
)

print(
    "Dog Test final target:            "
    f"{DOG_TEST_FINAL_TARGET:,}"
)

print(
    "Manual-cleaning buffer:           "
    f"{DOG_TEST_BUFFER:,}"
)

print()


def find_open_images_label_id(
    class_descriptions_file,
    class_name,
):
    with class_descriptions_file.open(
        "r",
        encoding="utf-8-sig",
        newline="",
    ) as file:
        reader = csv.reader(file)

        for row in reader:
            if len(row) < 2:
                continue

            label_id = row[0].strip()
            display_name = row[1].strip()

            if (
                display_name.casefold()
                == class_name.casefold()
            ):
                return label_id

    raise ValueError(
        f"Open Images label was not found: "
        f"{class_name}"
    )


dog_label_id = find_open_images_label_id(
    class_descriptions_path,
    "Dog",
)

print(
    f"Open Images Dog label: "
    f"{dog_label_id}"
)

print()


if not train_val_hash_cache_path.exists():
    raise FileNotFoundError(
        "The Train/Validation exact-hash cache was not found:\n"
        f"{show_path(train_val_hash_cache_path)}"
    )


with train_val_hash_cache_path.open(
    "rb"
) as file:
    cached_data = pickle.load(file)


train_val_hash_records = cached_data.get(
    "hash_records"
)

if not isinstance(
    train_val_hash_records,
    dict,
):
    raise RuntimeError(
        "The Train/Validation exact-hash cache "
        "does not contain valid hash records."
    )


print(
    "Loaded cached Train/Validation exact hashes."
)

print(
    "Train/Validation unique hashes loaded: "
    f"{len(train_val_hash_records):,}"
)

print()


existing_cat_test_hashes = {}


print(
    "Loading existing Auto Cleaned Cat Test hashes..."
)

for image_path in tqdm(
    get_image_files(cat_auto_cleaned_test_dir),
    desc="Hashing existing Cat Test",
    unit="image",
):
    try:
        exact_hash = get_exact_image_hash(
            image_path
        )

    except Exception:
        continue

    if exact_hash not in existing_cat_test_hashes:
        existing_cat_test_hashes[exact_hash] = {
            "source_name": "Cat Test",
            "source_root": cat_auto_cleaned_test_dir,
            "original_path": image_path,
            "cleaned_path": image_path,
        }


print(
    "Existing unique Cat Test hashes: "
    f"{len(existing_cat_test_hashes):,}"
)

print()


existing_dog_test_images = get_image_files(
    dog_auto_cleaned_test_dir
)

seen_dog_test_hashes = {}
accepted_image_ids = set()


print(
    "Loading existing Auto Cleaned Dog Test hashes..."
)

for image_path in tqdm(
    existing_dog_test_images,
    desc="Hashing existing Dog Test",
    unit="image",
):
    try:
        exact_hash = get_exact_image_hash(
            image_path
        )

    except Exception:
        continue

    if exact_hash not in seen_dog_test_hashes:
        seen_dog_test_hashes[exact_hash] = {
            "source_name": "Dog Test",
            "source_root": dog_auto_cleaned_test_dir,
            "original_path": image_path,
            "cleaned_path": image_path,
        }

    accepted_image_ids.add(
        image_path.stem
    )


existing_cleaned_count = len(
    existing_dog_test_images
)

cleaning_state = {
    "cleaned_count": existing_cleaned_count
}


print(
    "Existing Auto Cleaned Dog Test images: "
    f"{existing_cleaned_count:,}"
)

print(
    "Existing unique Dog Test hashes:       "
    f"{len(seen_dog_test_hashes):,}"
)

print()


s3_resource = boto3.resource(
    "s3",
    config=Config(
        signature_version=UNSIGNED,
        retries={
            "max_attempts": 5,
            "mode": "standard",
        },
    ),
)

open_images_bucket = s3_resource.Bucket(
    "open-images-dataset"
)


def download_open_images_image(
    split_name,
    image_id,
    output_path,
):
    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if (
        output_path.exists()
        and output_path.stat().st_size > 0
    ):
        return True, False, None

    if output_path.exists():
        output_path.unlink()

    temporary_path = output_path.with_suffix(
        output_path.suffix + ".part"
    )

    if temporary_path.exists():
        temporary_path.unlink()

    last_error = None

    for attempt in range(
        1,
        DOWNLOAD_RETRIES + 1,
    ):
        try:
            open_images_bucket.download_file(
                f"{split_name}/{image_id}.jpg",
                str(temporary_path),
            )

            temporary_path.replace(
                output_path
            )

            return True, True, None

        except Exception as error:
            last_error = error

            if temporary_path.exists():
                temporary_path.unlink()

            if attempt < DOWNLOAD_RETRIES:
                time.sleep(attempt)

    return False, False, str(last_error)


def save_download_failure(
    split_name,
    image_id,
    error_message,
):
    failure_dir = (
        dog_rejected_test_dir
        / "download_failed"
        / split_name.title()
    )

    failure_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    failure_path = (
        failure_dir
        / f"{image_id}.txt"
    )

    failure_path.write_text(
        (
            f"Image ID: {image_id}\n"
            f"Split: {split_name}\n"
            f"Error: {error_message}\n"
        ),
        encoding="utf-8",
    )


def load_positive_dog_ids(
    annotation_path,
    display_name,
):
    positive_ids = set()

    with annotation_path.open(
        "r",
        encoding="utf-8-sig",
        newline="",
    ) as file:
        reader = csv.DictReader(file)

        for row in tqdm(
            reader,
            desc=(
                f"Reading {display_name} "
                "Dog labels"
            ),
            unit="annotation",
        ):
            if (
                row.get("LabelName")
                != dog_label_id
            ):
                continue

            try:
                confidence = float(
                    row.get(
                        "Confidence",
                        0,
                    )
                )

            except (
                TypeError,
                ValueError,
            ):
                continue

            if confidence < 1.0:
                continue

            image_id = row.get(
                "ImageID",
                "",
            ).strip()

            if image_id:
                positive_ids.add(
                    image_id
                )

    return sorted(
        positive_ids
    )


def save_rejected_dog_image(
    image_path,
    split_root,
    split_name,
    reason,
):
    return copy_rejected_image(
        image_path,
        split_root,
        (
            dog_rejected_test_dir
            / reason
            / split_name.title()
        ),
    )


def save_duplicate_group_record(
    image_path,
    split_root,
    split_name,
    reason,
    exact_hash,
    kept_record,
):
    group_dir = (
        dog_rejected_test_dir
        / reason
        / split_name.title()
        / (
            "duplicate_group_"
            f"{exact_hash[:16]}"
        )
    )

    copy_duplicate_group(
        duplicate_path=image_path,
        duplicate_source_root=split_root,
        duplicate_source_name=(
            f"Open Images {split_name.title()} Dog"
        ),
        kept_record=kept_record,
        group_dir=group_dir,
    )


def process_dog_split(
    split_name,
    candidate_ids,
):
    split_info = split_metadata[
        split_name
    ]

    display_name = split_info[
        "display_name"
    ]

    source_root = (
        dog_original_test_dirs[
            split_name
        ]
    )

    random_generator = random.Random(
        split_info["random_seed"]
    )

    candidate_ids = list(
        candidate_ids
    )

    random_generator.shuffle(
        candidate_ids
    )

    stats = {
        "candidate_ids": len(candidate_ids),
        "processed": 0,
        "newly_downloaded": 0,
        "accepted": 0,
        "rejected": 0,
        "already_accepted": 0,
        "rejection_reasons": defaultdict(int),
    }

    print(
        "Downloading and automatically cleaning "
        f"Open Images {display_name} Dog images..."
    )

    for image_id in tqdm(
        candidate_ids,
        desc=(
            f"Open Images {display_name} Dog"
        ),
        unit="image",
    ):
        if (
            cleaning_state["cleaned_count"]
            >= DOG_TEST_AUTOMATIC_TARGET
        ):
            break

        if image_id in accepted_image_ids:
            stats["already_accepted"] += 1
            continue

        image_path = (
            source_root
            / f"{image_id}.jpg"
        )

        (
            download_successful,
            newly_downloaded,
            download_error,
        ) = download_open_images_image(
            split_name,
            image_id,
            image_path,
        )

        stats["processed"] += 1

        if newly_downloaded:
            stats["newly_downloaded"] += 1

        if not download_successful:
            reason = "download_failed"

            stats["rejected"] += 1
            stats["rejection_reasons"][reason] += 1

            save_download_failure(
                split_name,
                image_id,
                download_error,
            )

            continue

        is_valid, reason = check_image(
            image_path
        )

        if not is_valid:
            stats["rejected"] += 1
            stats["rejection_reasons"][reason] += 1

            save_rejected_dog_image(
                image_path,
                source_root,
                split_name,
                reason,
            )

            continue

        try:
            exact_hash = get_exact_image_hash(
                image_path
            )

        except Exception:
            reason = "hash_failed"

            stats["rejected"] += 1
            stats["rejection_reasons"][reason] += 1

            save_rejected_dog_image(
                image_path,
                source_root,
                split_name,
                reason,
            )

            continue

        if exact_hash in train_val_hash_records:
            reason = (
                "exact_duplicate_train_validation"
            )

            kept_relative_path = Path(
                train_val_hash_records[
                    exact_hash
                ]
            )

            kept_path = (
                auto_cleaned_train_val_dir
                / kept_relative_path
            )

            kept_record = {
                "source_name": "Train and Validation",
                "source_root": auto_cleaned_train_val_dir,
                "original_path": kept_path,
                "cleaned_path": kept_path,
            }

            stats["rejected"] += 1
            stats["rejection_reasons"][reason] += 1

            save_duplicate_group_record(
                image_path,
                source_root,
                split_name,
                reason,
                exact_hash,
                kept_record,
            )

            continue

        if exact_hash in existing_cat_test_hashes:
            reason = "exact_duplicate_cat_test"

            stats["rejected"] += 1
            stats["rejection_reasons"][reason] += 1

            save_duplicate_group_record(
                image_path,
                source_root,
                split_name,
                reason,
                exact_hash,
                existing_cat_test_hashes[
                    exact_hash
                ],
            )

            continue

        if exact_hash in seen_dog_test_hashes:
            reason = "exact_duplicate"

            stats["rejected"] += 1
            stats["rejection_reasons"][reason] += 1

            save_duplicate_group_record(
                image_path,
                source_root,
                split_name,
                reason,
                exact_hash,
                seen_dog_test_hashes[
                    exact_hash
                ],
            )

            continue

        try:
            cleaned_path = copy_preserving_structure(
                image_path,
                source_root,
                dog_auto_cleaned_test_dir,
            )

        except Exception:
            reason = "copy_failed"

            stats["rejected"] += 1
            stats["rejection_reasons"][reason] += 1

            save_rejected_dog_image(
                image_path,
                source_root,
                split_name,
                reason,
            )

            continue

        seen_dog_test_hashes[
            exact_hash
        ] = {
            "source_name": (
                f"Open Images {display_name} Dog"
            ),
            "source_root": source_root,
            "original_path": image_path,
            "cleaned_path": cleaned_path,
        }

        accepted_image_ids.add(
            image_id
        )

        stats["accepted"] += 1
        cleaning_state["cleaned_count"] += 1

    return stats


candidate_manifest_rows = []

for image_path in existing_dog_test_images:
    candidate_manifest_rows.append(
        {
            "split": "existing",
            "image_id": image_path.stem,
            "candidate_source": (
                "existing_auto_cleaned"
            ),
            "confidence": 1.0,
        }
    )


split_results = {}


for split_name in [
    "validation",
    "test",
    "train",
]:
    if (
        cleaning_state["cleaned_count"]
        >= DOG_TEST_AUTOMATIC_TARGET
    ):
        break

    split_info = split_metadata[
        split_name
    ]

    print(
        "Preparing Open Images "
        f"{split_info['display_name']} "
        "Dog candidates..."
    )

    download_file(
        split_info["url"],
        split_info["path"],
    )

    candidate_ids = load_positive_dog_ids(
        split_info["path"],
        split_info["display_name"],
    )

    print()

    print(
        "Human-verified positive Dog "
        f"images in {split_info['display_name']}: "
        f"{len(candidate_ids):,}"
    )

    print()

    for image_id in candidate_ids:
        candidate_manifest_rows.append(
            {
                "split": split_name,
                "image_id": image_id,
                "candidate_source": (
                    "human_verified"
                ),
                "confidence": 1.0,
            }
        )

    split_results[split_name] = (
        process_dog_split(
            split_name,
            candidate_ids,
        )
    )

    print()


with dog_candidate_manifest_path.open(
    "w",
    encoding="utf-8",
    newline="",
) as file:
    writer = csv.DictWriter(
        file,
        fieldnames=[
            "split",
            "image_id",
            "candidate_source",
            "confidence",
        ],
    )

    writer.writeheader()
    writer.writerows(
        candidate_manifest_rows
    )


final_cleaned_count = (
    cleaning_state["cleaned_count"]
)

newly_accepted_total = sum(
    result["accepted"]
    for result in split_results.values()
)

newly_rejected_total = sum(
    result["rejected"]
    for result in split_results.values()
)


print(
    "Dog Test automatic cleaning is complete."
)

print()


for split_name, result in (
    split_results.items()
):
    display_name = split_metadata[
        split_name
    ]["display_name"]

    print(
        f"Open Images {display_name}"
    )

    print(
        "  Candidate image IDs:    "
        f"{result['candidate_ids']:,}"
    )

    print(
        "  Processed images:       "
        f"{result['processed']:,}"
    )

    print(
        "  Newly downloaded:       "
        f"{result['newly_downloaded']:,}"
    )

    print(
        "  Auto Cleaned images:    "
        f"{result['accepted']:,}"
    )

    print(
        "  Automatically rejected: "
        f"{result['rejected']:,}"
    )

    if result["rejection_reasons"]:
        print("  Rejection reasons:")

        for reason, count in sorted(
            result[
                "rejection_reasons"
            ].items()
        ):
            print(
                f"    {reason}: "
                f"{count:,}"
            )

    print()


print("Total")

print(
    "  Existing Auto Cleaned images: "
    f"{existing_cleaned_count:,}"
)

print(
    "  Newly Auto Cleaned images:    "
    f"{newly_accepted_total:,}"
)

print(
    "  Automatically rejected:       "
    f"{newly_rejected_total:,}"
)

print(
    "  Current Auto Cleaned images:   "
    f"{final_cleaned_count:,}"
)

print(
    "  Automatic target:             "
    f"{DOG_TEST_AUTOMATIC_TARGET:,}"
)

print(
    "  Final target:                 "
    f"{DOG_TEST_FINAL_TARGET:,}"
)

print(
    "  Manual-cleaning buffer:       "
    f"{DOG_TEST_BUFFER:,}"
)

print()


if (
    final_cleaned_count
    >= DOG_TEST_AUTOMATIC_TARGET
):
    print(
        "The Dog Test automatic-cleaning "
        "target was reached."
    )

else:
    print(
        "The Dog Test automatic-cleaning "
        "target was not reached."
    )

    print(
        "Automatically cleaned images "
        "still needed: "
        f"{DOG_TEST_AUTOMATIC_TARGET - final_cleaned_count:,}"
    )


print()

print(
    "Auto Cleaned Dog Test images saved in: "
    f"{show_path(dog_auto_cleaned_test_dir)}"
)

print(
    "Automatically rejected Dog Test images "
    "saved in: "
    f"{show_path(dog_rejected_test_dir)}"
)

print(
    "Dog Test candidate manifest saved in: "
    f"{show_path(dog_candidate_manifest_path)}"
)

Dog Test paths are ready.
Original Open Images directory:   Image Classifier Project\Model Building\Datasets\Original\Test\Open Images V7
Auto Cleaned Dog Test directory:  Image Classifier Project\Model Building\Datasets\Auto Cleaned\Test\Dog
Automatically rejected directory: Image Classifier Project\Model Building\Datasets\Rejected\Automatic\Test\Dog
Dog Test automatic target:        3,000
Dog Test final target:            2,500
Manual-cleaning buffer:           500

Open Images Dog label: /m/0bt9lr

Loaded cached Train/Validation exact hashes.
Train/Validation unique hashes loaded: 147,337

Loading existing Auto Cleaned Cat Test hashes...


Hashing existing Cat Test: 100%|██████████| 3000/3000 [00:58<00:00, 51.62image/s]


Existing unique Cat Test hashes: 3,000

Loading existing Auto Cleaned Dog Test hashes...


Hashing existing Dog Test: 0image [00:00, ?image/s]


Existing Auto Cleaned Dog Test images: 0
Existing unique Dog Test hashes:       0

Preparing Open Images Validation Dog candidates...
Already downloaded: oidv7-val-annotations-human-imagelabels.csv


...


Preparing Open Images Test Dog candidates...
Already downloaded: test-annotations-human-imagelabels-boxable.csv


Reading Test Dog labels: 772776annotation [00:00, 1015727.15annotation/s]



Human-verified positive Dog images in Test: 4,835



Open Images Test Dog:  29%|██▉       | 1411/4835 [46:27<1:52:45,  1.98s/image]


Dog Test automatic cleaning is complete.

Open Images Validation
  Candidate image IDs:    1,593
  Processed images:       1,593
  Newly downloaded:       1,593
  Auto Cleaned images:    1,590
  Automatically rejected: 3
  Rejection reasons:
    exact_duplicate_cat_test: 3

Open Images Test
  Candidate image IDs:    4,835
  Processed images:       1,411
  Newly downloaded:       1,411
  Auto Cleaned images:    1,410
  Automatically rejected: 1
  Rejection reasons:
    exact_duplicate_cat_test: 1

Total
  Existing Auto Cleaned images: 0
  Newly Auto Cleaned images:    3,000
  Automatically rejected:       4
  Current Auto Cleaned images:   3,000
  Automatic target:             3,000
  Final target:                 2,500
  Manual-cleaning buffer:       500

The Dog Test automatic-cleaning target was reached.

Auto Cleaned Dog Test images saved in: Image Classifier Project\Model Building\Datasets\Auto Cleaned\Test\Dog
Automatically rejected Dog Test images saved in: Image Classifier Proj

### 2.3 Downloading and Automatically Cleaning Bird Test Images

This section downloads and automatically cleans the Bird images intended for the official Test set.

Raw Bird Test images from Open Images V7 are stored under:

`Model Building/Datasets/Original/Test/Open Images V7/Bird`

Automatically cleaned Bird Test images are stored under:

`Model Building/Datasets/Auto Cleaned/Test/Bird`

Automatically rejected Bird images are copied into reason-based folders under:

`Model Building/Datasets/Rejected/Automatic/Test/Bird`

Open Images V7 is used as the source for the official Bird Test images. Bird candidates are identified using human-verified image labels and suitable bounding-box annotations.

Candidates are collected from the Open Images validation split first, followed by the test split and then the train split until the automatic-cleaning target is reached.

The Bird Test source link is:

```text
Open Images V7:
https://storage.googleapis.com/openimages/web/index.html

The automatic-cleaning process retains up to 3,000 Bird images. This provides a 500-image buffer for the later manual-cleaning stage, after which the best 2,500 Bird images will be retained.

The same universal automatic-cleaning rules used for the Train/Validation images are applied to the Bird Test images.

In [6]:
import csv
import pickle
import random
import time

try:
    import boto3

    from botocore import UNSIGNED
    from botocore.config import Config

except ModuleNotFoundError:
    install_package("boto3")

    import boto3

    from botocore import UNSIGNED
    from botocore.config import Config


BIRD_TEST_FINAL_TARGET = 2_500
BIRD_TEST_AUTOMATIC_TARGET = 3_000
BIRD_TEST_BUFFER = (
    BIRD_TEST_AUTOMATIC_TARGET
    - BIRD_TEST_FINAL_TARGET
)

OPEN_IMAGES_RANDOM_SEED = 42
DOWNLOAD_RETRIES = 3

RESET_BIRD_TEST_OUTPUTS = False


original_test_dir = (
    datasets_dir
    / "Original"
    / "Test"
)

open_images_original_dir = (
    original_test_dir
    / "Open Images V7"
)

open_images_metadata_dir = (
    open_images_original_dir
    / "Metadata"
)

bird_original_test_dirs = {
    "validation": (
        open_images_original_dir
        / "Validation"
        / "Bird"
    ),
    "test": (
        open_images_original_dir
        / "Test"
        / "Bird"
    ),
    "train": (
        open_images_original_dir
        / "Train"
        / "Bird"
    ),
}

bird_auto_cleaned_test_dir = (
    datasets_dir
    / "Auto Cleaned"
    / "Test"
    / "Bird"
)

cat_auto_cleaned_test_dir = (
    datasets_dir
    / "Auto Cleaned"
    / "Test"
    / "Cat"
)

dog_auto_cleaned_test_dir = (
    datasets_dir
    / "Auto Cleaned"
    / "Test"
    / "Dog"
)

bird_rejected_test_dir = (
    datasets_dir
    / "Rejected"
    / "Automatic"
    / "Test"
    / "Bird"
)

auto_cleaned_train_val_dir = (
    datasets_dir
    / "Auto Cleaned"
    / "Train and Validation"
)

train_val_hash_cache_path = (
    auto_cleaned_train_val_dir
    / ".exact_hash_cache.pkl"
)

class_descriptions_path = (
    open_images_metadata_dir
    / "oidv7-class-descriptions.csv"
)

validation_labels_path = (
    open_images_metadata_dir
    / "oidv7-val-annotations-human-imagelabels.csv"
)

test_labels_path = (
    open_images_metadata_dir
    / "test-annotations-human-imagelabels-boxable.csv"
)

train_labels_path = (
    open_images_metadata_dir
    / "train-annotations-human-imagelabels-boxable.csv"
)

bird_candidate_manifest_path = (
    open_images_metadata_dir
    / "bird_test_candidate_manifest.csv"
)


old_validation_metadata_dir = (
    original_test_dir
    / "Open Images V7 Validation"
    / "Metadata"
)

old_class_descriptions_path = (
    old_validation_metadata_dir
    / "oidv7-class-descriptions.csv"
)

old_validation_labels_path = (
    old_validation_metadata_dir
    / "oidv7-val-annotations-human-imagelabels.csv"
)


split_metadata = {
    "validation": {
        "display_name": "Validation",
        "url": (
            "https://storage.googleapis.com/openimages/v7/"
            "oidv7-val-annotations-human-imagelabels.csv"
        ),
        "path": validation_labels_path,
        "random_seed": OPEN_IMAGES_RANDOM_SEED,
    },
    "test": {
        "display_name": "Test",
        "url": (
            "https://storage.googleapis.com/openimages/v5/"
            "test-annotations-human-imagelabels-boxable.csv"
        ),
        "path": test_labels_path,
        "random_seed": OPEN_IMAGES_RANDOM_SEED + 1,
    },
    "train": {
        "display_name": "Train",
        "url": (
            "https://storage.googleapis.com/openimages/v5/"
            "train-annotations-human-imagelabels-boxable.csv"
        ),
        "path": train_labels_path,
        "random_seed": OPEN_IMAGES_RANDOM_SEED + 2,
    },
}


if RESET_BIRD_TEST_OUTPUTS:
    reset_folder(bird_auto_cleaned_test_dir)
    reset_folder(bird_rejected_test_dir)


for folder in [
    original_test_dir,
    open_images_original_dir,
    open_images_metadata_dir,
    bird_auto_cleaned_test_dir,
    bird_rejected_test_dir,
    *bird_original_test_dirs.values(),
]:
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )


if (
    not class_descriptions_path.exists()
    and old_class_descriptions_path.exists()
):
    shutil.copy2(
        old_class_descriptions_path,
        class_descriptions_path,
    )


if (
    not validation_labels_path.exists()
    and old_validation_labels_path.exists()
):
    shutil.copy2(
        old_validation_labels_path,
        validation_labels_path,
    )


if not class_descriptions_path.exists():
    download_file(
        (
            "https://storage.googleapis.com/openimages/v7/"
            "oidv7-class-descriptions.csv"
        ),
        class_descriptions_path,
    )


print("Bird Test paths are ready.")

print(
    "Original Open Images directory:    "
    f"{show_path(open_images_original_dir)}"
)

print(
    "Auto Cleaned Bird Test directory:  "
    f"{show_path(bird_auto_cleaned_test_dir)}"
)

print(
    "Automatically rejected directory: "
    f"{show_path(bird_rejected_test_dir)}"
)

print(
    "Bird Test automatic target:        "
    f"{BIRD_TEST_AUTOMATIC_TARGET:,}"
)

print(
    "Bird Test final target:            "
    f"{BIRD_TEST_FINAL_TARGET:,}"
)

print(
    "Manual-cleaning buffer:            "
    f"{BIRD_TEST_BUFFER:,}"
)

print()


def find_open_images_label_id(
    class_descriptions_file,
    class_name,
):
    with class_descriptions_file.open(
        "r",
        encoding="utf-8-sig",
        newline="",
    ) as file:
        reader = csv.reader(file)

        for row in reader:
            if len(row) < 2:
                continue

            label_id = row[0].strip()
            display_name = row[1].strip()

            if (
                display_name.casefold()
                == class_name.casefold()
            ):
                return label_id

    raise ValueError(
        f"Open Images label was not found: "
        f"{class_name}"
    )


bird_label_id = find_open_images_label_id(
    class_descriptions_path,
    "Bird",
)

print(
    f"Open Images Bird label: "
    f"{bird_label_id}"
)

print()


if not train_val_hash_cache_path.exists():
    raise FileNotFoundError(
        "The Train/Validation exact-hash cache was not found:\n"
        f"{show_path(train_val_hash_cache_path)}"
    )


with train_val_hash_cache_path.open(
    "rb"
) as file:
    cached_data = pickle.load(file)


train_val_hash_records = cached_data.get(
    "hash_records"
)

if not isinstance(
    train_val_hash_records,
    dict,
):
    raise RuntimeError(
        "The Train/Validation exact-hash cache "
        "does not contain valid hash records."
    )


print(
    "Loaded cached Train/Validation exact hashes."
)

print(
    "Train/Validation unique hashes loaded: "
    f"{len(train_val_hash_records):,}"
)

print()


def load_existing_test_hashes(
    folder,
    source_name,
    description,
):
    hash_records = {}

    print(
        f"Loading existing Auto Cleaned "
        f"{source_name} hashes..."
    )

    for image_path in tqdm(
        get_image_files(folder),
        desc=description,
        unit="image",
    ):
        try:
            exact_hash = get_exact_image_hash(
                image_path
            )

        except Exception:
            continue

        if exact_hash not in hash_records:
            hash_records[exact_hash] = {
                "source_name": source_name,
                "source_root": folder,
                "original_path": image_path,
                "cleaned_path": image_path,
            }

    print(
        f"Existing unique {source_name} hashes: "
        f"{len(hash_records):,}"
    )

    print()

    return hash_records


existing_cat_test_hashes = (
    load_existing_test_hashes(
        cat_auto_cleaned_test_dir,
        "Cat Test",
        "Hashing existing Cat Test",
    )
)

existing_dog_test_hashes = (
    load_existing_test_hashes(
        dog_auto_cleaned_test_dir,
        "Dog Test",
        "Hashing existing Dog Test",
    )
)


existing_bird_test_images = get_image_files(
    bird_auto_cleaned_test_dir
)

seen_bird_test_hashes = {}
accepted_image_ids = set()


print(
    "Loading existing Auto Cleaned Bird Test hashes..."
)

for image_path in tqdm(
    existing_bird_test_images,
    desc="Hashing existing Bird Test",
    unit="image",
):
    try:
        exact_hash = get_exact_image_hash(
            image_path
        )

    except Exception:
        continue

    if exact_hash not in seen_bird_test_hashes:
        seen_bird_test_hashes[exact_hash] = {
            "source_name": "Bird Test",
            "source_root": bird_auto_cleaned_test_dir,
            "original_path": image_path,
            "cleaned_path": image_path,
        }

    accepted_image_ids.add(
        image_path.stem
    )


existing_cleaned_count = len(
    existing_bird_test_images
)

cleaning_state = {
    "cleaned_count": existing_cleaned_count
}


print(
    "Existing Auto Cleaned Bird Test images: "
    f"{existing_cleaned_count:,}"
)

print(
    "Existing unique Bird Test hashes:       "
    f"{len(seen_bird_test_hashes):,}"
)

print()


s3_resource = boto3.resource(
    "s3",
    config=Config(
        signature_version=UNSIGNED,
        retries={
            "max_attempts": 5,
            "mode": "standard",
        },
    ),
)

open_images_bucket = s3_resource.Bucket(
    "open-images-dataset"
)


def download_open_images_image(
    split_name,
    image_id,
    output_path,
):
    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if (
        output_path.exists()
        and output_path.stat().st_size > 0
    ):
        return True, False, None

    if output_path.exists():
        output_path.unlink()

    temporary_path = output_path.with_suffix(
        output_path.suffix + ".part"
    )

    if temporary_path.exists():
        temporary_path.unlink()

    last_error = None

    for attempt in range(
        1,
        DOWNLOAD_RETRIES + 1,
    ):
        try:
            open_images_bucket.download_file(
                f"{split_name}/{image_id}.jpg",
                str(temporary_path),
            )

            temporary_path.replace(
                output_path
            )

            return True, True, None

        except Exception as error:
            last_error = error

            if temporary_path.exists():
                temporary_path.unlink()

            if attempt < DOWNLOAD_RETRIES:
                time.sleep(attempt)

    return False, False, str(last_error)


def save_download_failure(
    split_name,
    image_id,
    error_message,
):
    failure_dir = (
        bird_rejected_test_dir
        / "download_failed"
        / split_name.title()
    )

    failure_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    failure_path = (
        failure_dir
        / f"{image_id}.txt"
    )

    failure_path.write_text(
        (
            f"Image ID: {image_id}\n"
            f"Split: {split_name}\n"
            f"Error: {error_message}\n"
        ),
        encoding="utf-8",
    )


def load_positive_bird_ids(
    annotation_path,
    display_name,
):
    positive_ids = set()

    with annotation_path.open(
        "r",
        encoding="utf-8-sig",
        newline="",
    ) as file:
        reader = csv.DictReader(file)

        for row in tqdm(
            reader,
            desc=(
                f"Reading {display_name} "
                "Bird labels"
            ),
            unit="annotation",
        ):
            if (
                row.get("LabelName")
                != bird_label_id
            ):
                continue

            try:
                confidence = float(
                    row.get(
                        "Confidence",
                        0,
                    )
                )

            except (
                TypeError,
                ValueError,
            ):
                continue

            if confidence < 1.0:
                continue

            image_id = row.get(
                "ImageID",
                "",
            ).strip()

            if image_id:
                positive_ids.add(
                    image_id
                )

    return sorted(
        positive_ids
    )


def save_rejected_bird_image(
    image_path,
    split_root,
    split_name,
    reason,
):
    return copy_rejected_image(
        image_path,
        split_root,
        (
            bird_rejected_test_dir
            / reason
            / split_name.title()
        ),
    )


def save_duplicate_group_record(
    image_path,
    split_root,
    split_name,
    reason,
    exact_hash,
    kept_record,
):
    group_dir = (
        bird_rejected_test_dir
        / reason
        / split_name.title()
        / (
            "duplicate_group_"
            f"{exact_hash[:16]}"
        )
    )

    copy_duplicate_group(
        duplicate_path=image_path,
        duplicate_source_root=split_root,
        duplicate_source_name=(
            f"Open Images {split_name.title()} Bird"
        ),
        kept_record=kept_record,
        group_dir=group_dir,
    )


def process_bird_split(
    split_name,
    candidate_ids,
):
    split_info = split_metadata[
        split_name
    ]

    display_name = split_info[
        "display_name"
    ]

    source_root = (
        bird_original_test_dirs[
            split_name
        ]
    )

    random_generator = random.Random(
        split_info["random_seed"]
    )

    candidate_ids = list(
        candidate_ids
    )

    random_generator.shuffle(
        candidate_ids
    )

    stats = {
        "candidate_ids": len(candidate_ids),
        "processed": 0,
        "newly_downloaded": 0,
        "accepted": 0,
        "rejected": 0,
        "already_accepted": 0,
        "rejection_reasons": defaultdict(int),
    }

    print(
        "Downloading and automatically cleaning "
        f"Open Images {display_name} Bird images..."
    )

    for image_id in tqdm(
        candidate_ids,
        desc=(
            f"Open Images {display_name} Bird"
        ),
        unit="image",
    ):
        if (
            cleaning_state["cleaned_count"]
            >= BIRD_TEST_AUTOMATIC_TARGET
        ):
            break

        if image_id in accepted_image_ids:
            stats["already_accepted"] += 1
            continue

        image_path = (
            source_root
            / f"{image_id}.jpg"
        )

        (
            download_successful,
            newly_downloaded,
            download_error,
        ) = download_open_images_image(
            split_name,
            image_id,
            image_path,
        )

        stats["processed"] += 1

        if newly_downloaded:
            stats["newly_downloaded"] += 1

        if not download_successful:
            reason = "download_failed"

            stats["rejected"] += 1
            stats["rejection_reasons"][reason] += 1

            save_download_failure(
                split_name,
                image_id,
                download_error,
            )

            continue

        is_valid, reason = check_image(
            image_path
        )

        if not is_valid:
            stats["rejected"] += 1
            stats["rejection_reasons"][reason] += 1

            save_rejected_bird_image(
                image_path,
                source_root,
                split_name,
                reason,
            )

            continue

        try:
            exact_hash = get_exact_image_hash(
                image_path
            )

        except Exception:
            reason = "hash_failed"

            stats["rejected"] += 1
            stats["rejection_reasons"][reason] += 1

            save_rejected_bird_image(
                image_path,
                source_root,
                split_name,
                reason,
            )

            continue

        if exact_hash in train_val_hash_records:
            reason = (
                "exact_duplicate_train_validation"
            )

            kept_relative_path = Path(
                train_val_hash_records[
                    exact_hash
                ]
            )

            kept_path = (
                auto_cleaned_train_val_dir
                / kept_relative_path
            )

            kept_record = {
                "source_name": "Train and Validation",
                "source_root": auto_cleaned_train_val_dir,
                "original_path": kept_path,
                "cleaned_path": kept_path,
            }

            stats["rejected"] += 1
            stats["rejection_reasons"][reason] += 1

            save_duplicate_group_record(
                image_path,
                source_root,
                split_name,
                reason,
                exact_hash,
                kept_record,
            )

            continue

        if exact_hash in existing_cat_test_hashes:
            reason = "exact_duplicate_cat_test"

            stats["rejected"] += 1
            stats["rejection_reasons"][reason] += 1

            save_duplicate_group_record(
                image_path,
                source_root,
                split_name,
                reason,
                exact_hash,
                existing_cat_test_hashes[
                    exact_hash
                ],
            )

            continue

        if exact_hash in existing_dog_test_hashes:
            reason = "exact_duplicate_dog_test"

            stats["rejected"] += 1
            stats["rejection_reasons"][reason] += 1

            save_duplicate_group_record(
                image_path,
                source_root,
                split_name,
                reason,
                exact_hash,
                existing_dog_test_hashes[
                    exact_hash
                ],
            )

            continue

        if exact_hash in seen_bird_test_hashes:
            reason = "exact_duplicate"

            stats["rejected"] += 1
            stats["rejection_reasons"][reason] += 1

            save_duplicate_group_record(
                image_path,
                source_root,
                split_name,
                reason,
                exact_hash,
                seen_bird_test_hashes[
                    exact_hash
                ],
            )

            continue

        try:
            cleaned_path = copy_preserving_structure(
                image_path,
                source_root,
                bird_auto_cleaned_test_dir,
            )

        except Exception:
            reason = "copy_failed"

            stats["rejected"] += 1
            stats["rejection_reasons"][reason] += 1

            save_rejected_bird_image(
                image_path,
                source_root,
                split_name,
                reason,
            )

            continue

        seen_bird_test_hashes[
            exact_hash
        ] = {
            "source_name": (
                f"Open Images {display_name} Bird"
            ),
            "source_root": source_root,
            "original_path": image_path,
            "cleaned_path": cleaned_path,
        }

        accepted_image_ids.add(
            image_id
        )

        stats["accepted"] += 1
        cleaning_state["cleaned_count"] += 1

    return stats


candidate_manifest_rows = []

for image_path in existing_bird_test_images:
    candidate_manifest_rows.append(
        {
            "split": "existing",
            "image_id": image_path.stem,
            "candidate_source": (
                "existing_auto_cleaned"
            ),
            "confidence": 1.0,
        }
    )


split_results = {}


for split_name in [
    "validation",
    "test",
    "train",
]:
    if (
        cleaning_state["cleaned_count"]
        >= BIRD_TEST_AUTOMATIC_TARGET
    ):
        break

    split_info = split_metadata[
        split_name
    ]

    print(
        "Preparing Open Images "
        f"{split_info['display_name']} "
        "Bird candidates..."
    )

    download_file(
        split_info["url"],
        split_info["path"],
    )

    candidate_ids = load_positive_bird_ids(
        split_info["path"],
        split_info["display_name"],
    )

    print()

    print(
        "Human-verified positive Bird "
        f"images in {split_info['display_name']}: "
        f"{len(candidate_ids):,}"
    )

    print()

    for image_id in candidate_ids:
        candidate_manifest_rows.append(
            {
                "split": split_name,
                "image_id": image_id,
                "candidate_source": (
                    "human_verified"
                ),
                "confidence": 1.0,
            }
        )

    split_results[split_name] = (
        process_bird_split(
            split_name,
            candidate_ids,
        )
    )

    print()


with bird_candidate_manifest_path.open(
    "w",
    encoding="utf-8",
    newline="",
) as file:
    writer = csv.DictWriter(
        file,
        fieldnames=[
            "split",
            "image_id",
            "candidate_source",
            "confidence",
        ],
    )

    writer.writeheader()
    writer.writerows(
        candidate_manifest_rows
    )


final_cleaned_count = (
    cleaning_state["cleaned_count"]
)

newly_accepted_total = sum(
    result["accepted"]
    for result in split_results.values()
)

newly_rejected_total = sum(
    result["rejected"]
    for result in split_results.values()
)


print(
    "Bird Test automatic cleaning is complete."
)

print()


for split_name, result in (
    split_results.items()
):
    display_name = split_metadata[
        split_name
    ]["display_name"]

    print(
        f"Open Images {display_name}"
    )

    print(
        "  Candidate image IDs:    "
        f"{result['candidate_ids']:,}"
    )

    print(
        "  Processed images:       "
        f"{result['processed']:,}"
    )

    print(
        "  Newly downloaded:       "
        f"{result['newly_downloaded']:,}"
    )

    print(
        "  Auto Cleaned images:    "
        f"{result['accepted']:,}"
    )

    print(
        "  Automatically rejected: "
        f"{result['rejected']:,}"
    )

    if result["rejection_reasons"]:
        print("  Rejection reasons:")

        for reason, count in sorted(
            result[
                "rejection_reasons"
            ].items()
        ):
            print(
                f"    {reason}: "
                f"{count:,}"
            )

    print()


print("Total")

print(
    "  Existing Auto Cleaned images: "
    f"{existing_cleaned_count:,}"
)

print(
    "  Newly Auto Cleaned images:    "
    f"{newly_accepted_total:,}"
)

print(
    "  Automatically rejected:       "
    f"{newly_rejected_total:,}"
)

print(
    "  Current Auto Cleaned images:   "
    f"{final_cleaned_count:,}"
)

print(
    "  Automatic target:             "
    f"{BIRD_TEST_AUTOMATIC_TARGET:,}"
)

print(
    "  Final target:                 "
    f"{BIRD_TEST_FINAL_TARGET:,}"
)

print(
    "  Manual-cleaning buffer:       "
    f"{BIRD_TEST_BUFFER:,}"
)

print()


if (
    final_cleaned_count
    >= BIRD_TEST_AUTOMATIC_TARGET
):
    print(
        "The Bird Test automatic-cleaning "
        "target was reached."
    )

else:
    print(
        "The Bird Test automatic-cleaning "
        "target was not reached."
    )

    print(
        "Automatically cleaned images "
        "still needed: "
        f"{BIRD_TEST_AUTOMATIC_TARGET - final_cleaned_count:,}"
    )


print()

print(
    "Auto Cleaned Bird Test images saved in: "
    f"{show_path(bird_auto_cleaned_test_dir)}"
)

print(
    "Automatically rejected Bird Test images "
    "saved in: "
    f"{show_path(bird_rejected_test_dir)}"
)

print(
    "Bird Test candidate manifest saved in: "
    f"{show_path(bird_candidate_manifest_path)}"
)

Bird Test paths are ready.
Original Open Images directory:    Image Classifier Project\Model Building\Datasets\Original\Test\Open Images V7
Auto Cleaned Bird Test directory:  Image Classifier Project\Model Building\Datasets\Auto Cleaned\Test\Bird
Automatically rejected directory: Image Classifier Project\Model Building\Datasets\Rejected\Automatic\Test\Bird
Bird Test automatic target:        3,000
Bird Test final target:            2,500
Manual-cleaning buffer:            500

Open Images Bird label: /m/015p6

Loaded cached Train/Validation exact hashes.
Train/Validation unique hashes loaded: 147,337

Loading existing Auto Cleaned Cat Test hashes...


Hashing existing Cat Test: 100%|██████████| 3000/3000 [00:21<00:00, 141.04image/s]


Existing unique Cat Test hashes: 3,000

Loading existing Auto Cleaned Dog Test hashes...


Hashing existing Dog Test: 100%|██████████| 3000/3000 [00:54<00:00, 55.06image/s]


Existing unique Dog Test hashes: 3,000

Loading existing Auto Cleaned Bird Test hashes...


...


Preparing Open Images Train Bird candidates...
Already downloaded: train-annotations-human-imagelabels-boxable.csv


Reading Train Bird labels: 8996795annotation [00:08, 1029941.47annotation/s]



Human-verified positive Bird images in Train: 35,586



Open Images Train Bird:   1%|          | 411/35586 [13:05<18:40:18,  1.91s/image]


Bird Test automatic cleaning is complete.

Open Images Validation
  Candidate image IDs:    698
  Processed images:       698
  Newly downloaded:       698
  Auto Cleaned images:    697
  Automatically rejected: 1
  Rejection reasons:
    exact_duplicate_cat_test: 1

Open Images Test
  Candidate image IDs:    1,895
  Processed images:       1,895
  Newly downloaded:       1,895
  Auto Cleaned images:    1,892
  Automatically rejected: 3
  Rejection reasons:
    exact_duplicate_dog_test: 3

Open Images Train
  Candidate image IDs:    35,586
  Processed images:       411
  Newly downloaded:       411
  Auto Cleaned images:    411
  Automatically rejected: 0

Total
  Existing Auto Cleaned images: 0
  Newly Auto Cleaned images:    3,000
  Automatically rejected:       4
  Current Auto Cleaned images:   3,000
  Automatic target:             3,000
  Final target:                 2,500
  Manual-cleaning buffer:       500

The Bird Test automatic-cleaning target was reached.

Auto Cleaned Bir

### 2.4 Downloading and Automatically Cleaning Unknown Test Images

This section downloads and automatically cleans the Unknown images intended for the official Test set.

Raw Unknown Test images from Open Images V7 are stored under:

`Model Building/Datasets/Original/Test/Open Images V7/Unknown`

Automatically cleaned Unknown Test images are stored under:

`Model Building/Datasets/Auto Cleaned/Test/Unknown`

Automatically rejected Unknown images are copied into reason-based folders under:

`Model Building/Datasets/Rejected/Automatic/Test/Unknown`

Open Images V7 is used as the source for the official Unknown Test images. Unknown candidates are selected from images with positive non-target labels and no positive Cat, Dog, or Bird image labels.

Candidates are collected from the Open Images validation split first, followed by the test split and then the train split until the automatic-cleaning target is reached.

The Unknown Test source link is:

```text
Open Images V7:
https://storage.googleapis.com/openimages/web/index.html
````

The automatic-cleaning process retains up to **3,000 Unknown images**. This provides a 500-image buffer for the later manual-cleaning stage, after which the best **2,500 Unknown images** will be retained.

The same universal automatic-cleaning rules used for the Train/Validation images are applied to the Unknown Test images.


In [7]:
import csv
import pickle
import random
import time

try:
    import boto3

    from botocore import UNSIGNED
    from botocore.config import Config

except ModuleNotFoundError:
    install_package("boto3")

    import boto3

    from botocore import UNSIGNED
    from botocore.config import Config


UNKNOWN_TEST_FINAL_TARGET = 2_500
UNKNOWN_TEST_AUTOMATIC_TARGET = 3_000
UNKNOWN_TEST_BUFFER = (
    UNKNOWN_TEST_AUTOMATIC_TARGET
    - UNKNOWN_TEST_FINAL_TARGET
)

OPEN_IMAGES_RANDOM_SEED = 42
DOWNLOAD_RETRIES = 3

RESET_UNKNOWN_TEST_OUTPUTS = False


original_test_dir = (
    datasets_dir
    / "Original"
    / "Test"
)

open_images_original_dir = (
    original_test_dir
    / "Open Images V7"
)

open_images_metadata_dir = (
    open_images_original_dir
    / "Metadata"
)

unknown_original_test_dirs = {
    "validation": (
        open_images_original_dir
        / "Validation"
        / "Unknown"
    ),
    "test": (
        open_images_original_dir
        / "Test"
        / "Unknown"
    ),
    "train": (
        open_images_original_dir
        / "Train"
        / "Unknown"
    ),
}

unknown_auto_cleaned_test_dir = (
    datasets_dir
    / "Auto Cleaned"
    / "Test"
    / "Unknown"
)

cat_auto_cleaned_test_dir = (
    datasets_dir
    / "Auto Cleaned"
    / "Test"
    / "Cat"
)

dog_auto_cleaned_test_dir = (
    datasets_dir
    / "Auto Cleaned"
    / "Test"
    / "Dog"
)

bird_auto_cleaned_test_dir = (
    datasets_dir
    / "Auto Cleaned"
    / "Test"
    / "Bird"
)

unknown_rejected_test_dir = (
    datasets_dir
    / "Rejected"
    / "Automatic"
    / "Test"
    / "Unknown"
)

auto_cleaned_train_val_dir = (
    datasets_dir
    / "Auto Cleaned"
    / "Train and Validation"
)

train_val_hash_cache_path = (
    auto_cleaned_train_val_dir
    / ".exact_hash_cache.pkl"
)

class_descriptions_path = (
    open_images_metadata_dir
    / "oidv7-class-descriptions.csv"
)

validation_labels_path = (
    open_images_metadata_dir
    / "oidv7-val-annotations-human-imagelabels.csv"
)

test_labels_path = (
    open_images_metadata_dir
    / "test-annotations-human-imagelabels-boxable.csv"
)

train_labels_path = (
    open_images_metadata_dir
    / "train-annotations-human-imagelabels-boxable.csv"
)

unknown_candidate_manifest_path = (
    open_images_metadata_dir
    / "unknown_test_candidate_manifest.csv"
)


old_validation_metadata_dir = (
    original_test_dir
    / "Open Images V7 Validation"
    / "Metadata"
)

old_class_descriptions_path = (
    old_validation_metadata_dir
    / "oidv7-class-descriptions.csv"
)

old_validation_labels_path = (
    old_validation_metadata_dir
    / "oidv7-val-annotations-human-imagelabels.csv"
)


split_metadata = {
    "validation": {
        "display_name": "Validation",
        "url": (
            "https://storage.googleapis.com/openimages/v7/"
            "oidv7-val-annotations-human-imagelabels.csv"
        ),
        "path": validation_labels_path,
        "random_seed": OPEN_IMAGES_RANDOM_SEED,
    },
    "test": {
        "display_name": "Test",
        "url": (
            "https://storage.googleapis.com/openimages/v5/"
            "test-annotations-human-imagelabels-boxable.csv"
        ),
        "path": test_labels_path,
        "random_seed": OPEN_IMAGES_RANDOM_SEED + 1,
    },
    "train": {
        "display_name": "Train",
        "url": (
            "https://storage.googleapis.com/openimages/v5/"
            "train-annotations-human-imagelabels-boxable.csv"
        ),
        "path": train_labels_path,
        "random_seed": OPEN_IMAGES_RANDOM_SEED + 2,
    },
}


if RESET_UNKNOWN_TEST_OUTPUTS:
    reset_folder(
        unknown_auto_cleaned_test_dir
    )

    reset_folder(
        unknown_rejected_test_dir
    )


for folder in [
    original_test_dir,
    open_images_original_dir,
    open_images_metadata_dir,
    unknown_auto_cleaned_test_dir,
    unknown_rejected_test_dir,
    *unknown_original_test_dirs.values(),
]:
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )


if (
    not class_descriptions_path.exists()
    and old_class_descriptions_path.exists()
):
    shutil.copy2(
        old_class_descriptions_path,
        class_descriptions_path,
    )


if (
    not validation_labels_path.exists()
    and old_validation_labels_path.exists()
):
    shutil.copy2(
        old_validation_labels_path,
        validation_labels_path,
    )


if not class_descriptions_path.exists():
    download_file(
        (
            "https://storage.googleapis.com/openimages/v7/"
            "oidv7-class-descriptions.csv"
        ),
        class_descriptions_path,
    )


print("Unknown Test paths are ready.")

print(
    "Original Open Images directory:       "
    f"{show_path(open_images_original_dir)}"
)

print(
    "Auto Cleaned Unknown Test directory:  "
    f"{show_path(unknown_auto_cleaned_test_dir)}"
)

print(
    "Automatically rejected directory:    "
    f"{show_path(unknown_rejected_test_dir)}"
)

print(
    "Unknown Test automatic target:        "
    f"{UNKNOWN_TEST_AUTOMATIC_TARGET:,}"
)

print(
    "Unknown Test final target:            "
    f"{UNKNOWN_TEST_FINAL_TARGET:,}"
)

print(
    "Manual-cleaning buffer:               "
    f"{UNKNOWN_TEST_BUFFER:,}"
)

print()


def find_open_images_label_id(
    class_descriptions_file,
    class_name,
):
    with class_descriptions_file.open(
        "r",
        encoding="utf-8-sig",
        newline="",
    ) as file:
        reader = csv.reader(file)

        for row in reader:
            if len(row) < 2:
                continue

            label_id = row[0].strip()
            display_name = row[1].strip()

            if (
                display_name.casefold()
                == class_name.casefold()
            ):
                return label_id

    raise ValueError(
        "Open Images label was not found: "
        f"{class_name}"
    )


cat_label_id = find_open_images_label_id(
    class_descriptions_path,
    "Cat",
)

dog_label_id = find_open_images_label_id(
    class_descriptions_path,
    "Dog",
)

bird_label_id = find_open_images_label_id(
    class_descriptions_path,
    "Bird",
)

target_label_ids = {
    cat_label_id,
    dog_label_id,
    bird_label_id,
}


print(
    f"Open Images Cat label:  {cat_label_id}"
)

print(
    f"Open Images Dog label:  {dog_label_id}"
)

print(
    f"Open Images Bird label: {bird_label_id}"
)

print()


if not train_val_hash_cache_path.exists():
    raise FileNotFoundError(
        "The Train/Validation exact-hash cache "
        "was not found:\n"
        f"{show_path(train_val_hash_cache_path)}"
    )


with train_val_hash_cache_path.open(
    "rb"
) as file:
    cached_data = pickle.load(file)


train_val_hash_records = cached_data.get(
    "hash_records"
)

if not isinstance(
    train_val_hash_records,
    dict,
):
    raise RuntimeError(
        "The Train/Validation exact-hash cache "
        "does not contain valid hash records."
    )


print(
    "Loaded cached Train/Validation exact hashes."
)

print(
    "Train/Validation unique hashes loaded: "
    f"{len(train_val_hash_records):,}"
)

print()


def load_existing_test_hashes(
    folder,
    source_name,
    description,
):
    hash_records = {}

    print(
        "Loading existing Auto Cleaned "
        f"{source_name} hashes..."
    )

    image_paths = get_image_files(
        folder
    )

    for image_path in tqdm(
        image_paths,
        desc=description,
        unit="image",
    ):
        try:
            exact_hash = get_exact_image_hash(
                image_path
            )

        except Exception:
            continue

        if exact_hash not in hash_records:
            hash_records[exact_hash] = {
                "source_name": source_name,
                "source_root": folder,
                "original_path": image_path,
                "cleaned_path": image_path,
            }

    print(
        f"Existing {source_name} images:       "
        f"{len(image_paths):,}"
    )

    print(
        f"Existing unique {source_name} hashes: "
        f"{len(hash_records):,}"
    )

    print()

    return hash_records


existing_cat_test_hashes = (
    load_existing_test_hashes(
        cat_auto_cleaned_test_dir,
        "Cat Test",
        "Hashing existing Cat Test",
    )
)

existing_dog_test_hashes = (
    load_existing_test_hashes(
        dog_auto_cleaned_test_dir,
        "Dog Test",
        "Hashing existing Dog Test",
    )
)

existing_bird_test_hashes = (
    load_existing_test_hashes(
        bird_auto_cleaned_test_dir,
        "Bird Test",
        "Hashing existing Bird Test",
    )
)


existing_unknown_test_images = (
    get_image_files(
        unknown_auto_cleaned_test_dir
    )
)

seen_unknown_test_hashes = {}
accepted_image_ids = set()


print(
    "Loading existing Auto Cleaned "
    "Unknown Test hashes..."
)

for image_path in tqdm(
    existing_unknown_test_images,
    desc="Hashing existing Unknown Test",
    unit="image",
):
    try:
        exact_hash = get_exact_image_hash(
            image_path
        )

    except Exception:
        continue

    if exact_hash not in seen_unknown_test_hashes:
        seen_unknown_test_hashes[
            exact_hash
        ] = {
            "source_name": "Unknown Test",
            "source_root": (
                unknown_auto_cleaned_test_dir
            ),
            "original_path": image_path,
            "cleaned_path": image_path,
        }

    accepted_image_ids.add(
        image_path.stem
    )


existing_cleaned_count = len(
    existing_unknown_test_images
)

cleaning_state = {
    "cleaned_count": existing_cleaned_count
}


print(
    "Existing Auto Cleaned Unknown Test images: "
    f"{existing_cleaned_count:,}"
)

print(
    "Existing unique Unknown Test hashes:       "
    f"{len(seen_unknown_test_hashes):,}"
)

print()


s3_resource = boto3.resource(
    "s3",
    config=Config(
        signature_version=UNSIGNED,
        retries={
            "max_attempts": 5,
            "mode": "standard",
        },
    ),
)

open_images_bucket = s3_resource.Bucket(
    "open-images-dataset"
)


def download_open_images_image(
    split_name,
    image_id,
    output_path,
):
    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if (
        output_path.exists()
        and output_path.stat().st_size > 0
    ):
        return True, False, None

    if output_path.exists():
        output_path.unlink()

    temporary_path = output_path.with_suffix(
        output_path.suffix + ".part"
    )

    if temporary_path.exists():
        temporary_path.unlink()

    last_error = None

    for attempt in range(
        1,
        DOWNLOAD_RETRIES + 1,
    ):
        try:
            open_images_bucket.download_file(
                f"{split_name}/{image_id}.jpg",
                str(temporary_path),
            )

            temporary_path.replace(
                output_path
            )

            return True, True, None

        except Exception as error:
            last_error = error

            if temporary_path.exists():
                temporary_path.unlink()

            if attempt < DOWNLOAD_RETRIES:
                time.sleep(attempt)

    return False, False, str(last_error)


def save_download_failure(
    split_name,
    image_id,
    error_message,
):
    failure_dir = (
        unknown_rejected_test_dir
        / "download_failed"
        / split_name.title()
    )

    failure_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    failure_path = (
        failure_dir
        / f"{image_id}.txt"
    )

    failure_path.write_text(
        (
            f"Image ID: {image_id}\n"
            f"Split: {split_name}\n"
            f"Error: {error_message}\n"
        ),
        encoding="utf-8",
    )


def load_unknown_candidate_ids(
    annotation_path,
    display_name,
):
    non_target_positive_ids = set()
    target_positive_ids = set()

    with annotation_path.open(
        "r",
        encoding="utf-8-sig",
        newline="",
    ) as file:
        reader = csv.DictReader(file)

        for row in tqdm(
            reader,
            desc=(
                f"Reading {display_name} "
                "Unknown candidate labels"
            ),
            unit="annotation",
        ):
            try:
                confidence = float(
                    row.get(
                        "Confidence",
                        0,
                    )
                )

            except (
                TypeError,
                ValueError,
            ):
                continue

            if confidence < 1.0:
                continue

            image_id = row.get(
                "ImageID",
                "",
            ).strip()

            label_id = row.get(
                "LabelName",
                "",
            ).strip()

            if not image_id or not label_id:
                continue

            if label_id in target_label_ids:
                target_positive_ids.add(
                    image_id
                )

            else:
                non_target_positive_ids.add(
                    image_id
                )

    candidate_ids = sorted(
        non_target_positive_ids
        - target_positive_ids
    )

    return {
        "candidate_ids": candidate_ids,
        "non_target_positive_images": len(
            non_target_positive_ids
        ),
        "target_positive_images": len(
            target_positive_ids
        ),
        "excluded_target_images": len(
            non_target_positive_ids
            & target_positive_ids
        ),
    }


def save_rejected_unknown_image(
    image_path,
    split_root,
    split_name,
    reason,
):
    return copy_rejected_image(
        image_path,
        split_root,
        (
            unknown_rejected_test_dir
            / reason
            / split_name.title()
        ),
    )


def save_duplicate_group_record(
    image_path,
    split_root,
    split_name,
    reason,
    exact_hash,
    kept_record,
):
    group_dir = (
        unknown_rejected_test_dir
        / reason
        / split_name.title()
        / (
            "duplicate_group_"
            f"{exact_hash[:16]}"
        )
    )

    copy_duplicate_group(
        duplicate_path=image_path,
        duplicate_source_root=split_root,
        duplicate_source_name=(
            "Open Images "
            f"{split_name.title()} Unknown"
        ),
        kept_record=kept_record,
        group_dir=group_dir,
    )


def process_unknown_split(
    split_name,
    candidate_ids,
):
    split_info = split_metadata[
        split_name
    ]

    display_name = split_info[
        "display_name"
    ]

    source_root = (
        unknown_original_test_dirs[
            split_name
        ]
    )

    random_generator = random.Random(
        split_info["random_seed"]
    )

    candidate_ids = list(
        candidate_ids
    )

    random_generator.shuffle(
        candidate_ids
    )

    stats = {
        "candidate_ids": len(candidate_ids),
        "processed": 0,
        "newly_downloaded": 0,
        "accepted": 0,
        "rejected": 0,
        "already_accepted": 0,
        "rejection_reasons": defaultdict(int),
    }

    print(
        "Downloading and automatically cleaning "
        f"Open Images {display_name} Unknown images..."
    )

    for image_id in tqdm(
        candidate_ids,
        desc=(
            "Open Images "
            f"{display_name} Unknown"
        ),
        unit="image",
    ):
        if (
            cleaning_state["cleaned_count"]
            >= UNKNOWN_TEST_AUTOMATIC_TARGET
        ):
            break

        if image_id in accepted_image_ids:
            stats["already_accepted"] += 1
            continue

        image_path = (
            source_root
            / f"{image_id}.jpg"
        )

        (
            download_successful,
            newly_downloaded,
            download_error,
        ) = download_open_images_image(
            split_name,
            image_id,
            image_path,
        )

        stats["processed"] += 1

        if newly_downloaded:
            stats["newly_downloaded"] += 1

        if not download_successful:
            reason = "download_failed"

            stats["rejected"] += 1
            stats["rejection_reasons"][
                reason
            ] += 1

            save_download_failure(
                split_name,
                image_id,
                download_error,
            )

            continue

        is_valid, reason = check_image(
            image_path
        )

        if not is_valid:
            stats["rejected"] += 1
            stats["rejection_reasons"][
                reason
            ] += 1

            save_rejected_unknown_image(
                image_path,
                source_root,
                split_name,
                reason,
            )

            continue

        try:
            exact_hash = get_exact_image_hash(
                image_path
            )

        except Exception:
            reason = "hash_failed"

            stats["rejected"] += 1
            stats["rejection_reasons"][
                reason
            ] += 1

            save_rejected_unknown_image(
                image_path,
                source_root,
                split_name,
                reason,
            )

            continue

        if exact_hash in train_val_hash_records:
            reason = (
                "exact_duplicate_train_validation"
            )

            kept_relative_path = Path(
                train_val_hash_records[
                    exact_hash
                ]
            )

            kept_path = (
                auto_cleaned_train_val_dir
                / kept_relative_path
            )

            kept_record = {
                "source_name": (
                    "Train and Validation"
                ),
                "source_root": (
                    auto_cleaned_train_val_dir
                ),
                "original_path": kept_path,
                "cleaned_path": kept_path,
            }

            stats["rejected"] += 1
            stats["rejection_reasons"][
                reason
            ] += 1

            save_duplicate_group_record(
                image_path,
                source_root,
                split_name,
                reason,
                exact_hash,
                kept_record,
            )

            continue

        if exact_hash in existing_cat_test_hashes:
            reason = "exact_duplicate_cat_test"

            stats["rejected"] += 1
            stats["rejection_reasons"][
                reason
            ] += 1

            save_duplicate_group_record(
                image_path,
                source_root,
                split_name,
                reason,
                exact_hash,
                existing_cat_test_hashes[
                    exact_hash
                ],
            )

            continue

        if exact_hash in existing_dog_test_hashes:
            reason = "exact_duplicate_dog_test"

            stats["rejected"] += 1
            stats["rejection_reasons"][
                reason
            ] += 1

            save_duplicate_group_record(
                image_path,
                source_root,
                split_name,
                reason,
                exact_hash,
                existing_dog_test_hashes[
                    exact_hash
                ],
            )

            continue

        if exact_hash in existing_bird_test_hashes:
            reason = "exact_duplicate_bird_test"

            stats["rejected"] += 1
            stats["rejection_reasons"][
                reason
            ] += 1

            save_duplicate_group_record(
                image_path,
                source_root,
                split_name,
                reason,
                exact_hash,
                existing_bird_test_hashes[
                    exact_hash
                ],
            )

            continue

        if exact_hash in seen_unknown_test_hashes:
            reason = "exact_duplicate"

            stats["rejected"] += 1
            stats["rejection_reasons"][
                reason
            ] += 1

            save_duplicate_group_record(
                image_path,
                source_root,
                split_name,
                reason,
                exact_hash,
                seen_unknown_test_hashes[
                    exact_hash
                ],
            )

            continue

        try:
            cleaned_path = copy_preserving_structure(
                image_path,
                source_root,
                unknown_auto_cleaned_test_dir,
            )

        except Exception:
            reason = "copy_failed"

            stats["rejected"] += 1
            stats["rejection_reasons"][
                reason
            ] += 1

            save_rejected_unknown_image(
                image_path,
                source_root,
                split_name,
                reason,
            )

            continue

        seen_unknown_test_hashes[
            exact_hash
        ] = {
            "source_name": (
                "Open Images "
                f"{display_name} Unknown"
            ),
            "source_root": source_root,
            "original_path": image_path,
            "cleaned_path": cleaned_path,
        }

        accepted_image_ids.add(
            image_id
        )

        stats["accepted"] += 1
        cleaning_state["cleaned_count"] += 1

    return stats


candidate_manifest_rows = []

for image_path in existing_unknown_test_images:
    candidate_manifest_rows.append(
        {
            "split": "existing",
            "image_id": image_path.stem,
            "candidate_source": (
                "existing_auto_cleaned"
            ),
            "confidence": 1.0,
        }
    )


split_results = {}
split_candidate_summaries = {}


for split_name in [
    "validation",
    "test",
    "train",
]:
    if (
        cleaning_state["cleaned_count"]
        >= UNKNOWN_TEST_AUTOMATIC_TARGET
    ):
        break

    split_info = split_metadata[
        split_name
    ]

    print(
        "Preparing Open Images "
        f"{split_info['display_name']} "
        "Unknown candidates..."
    )

    download_file(
        split_info["url"],
        split_info["path"],
    )

    candidate_summary = (
        load_unknown_candidate_ids(
            split_info["path"],
            split_info["display_name"],
        )
    )

    split_candidate_summaries[
        split_name
    ] = candidate_summary

    candidate_ids = candidate_summary[
        "candidate_ids"
    ]

    print()

    print(
        "Images with positive non-target labels: "
        f"{candidate_summary['non_target_positive_images']:,}"
    )

    print(
        "Images with positive Cat, Dog, or Bird labels: "
        f"{candidate_summary['target_positive_images']:,}"
    )

    print(
        "Non-target images excluded because they also "
        "have a target label: "
        f"{candidate_summary['excluded_target_images']:,}"
    )

    print(
        "Eligible Unknown candidate images: "
        f"{len(candidate_ids):,}"
    )

    print()

    for image_id in candidate_ids:
        candidate_manifest_rows.append(
            {
                "split": split_name,
                "image_id": image_id,
                "candidate_source": (
                    "human_verified_non_target"
                ),
                "confidence": 1.0,
            }
        )

    split_results[split_name] = (
        process_unknown_split(
            split_name,
            candidate_ids,
        )
    )

    print()


with unknown_candidate_manifest_path.open(
    "w",
    encoding="utf-8",
    newline="",
) as file:
    writer = csv.DictWriter(
        file,
        fieldnames=[
            "split",
            "image_id",
            "candidate_source",
            "confidence",
        ],
    )

    writer.writeheader()

    writer.writerows(
        candidate_manifest_rows
    )


final_cleaned_count = (
    cleaning_state["cleaned_count"]
)

newly_accepted_total = sum(
    result["accepted"]
    for result in split_results.values()
)

newly_rejected_total = sum(
    result["rejected"]
    for result in split_results.values()
)


print(
    "Unknown Test automatic cleaning is complete."
)

print()


for split_name, result in (
    split_results.items()
):
    display_name = split_metadata[
        split_name
    ]["display_name"]

    print(
        f"Open Images {display_name}"
    )

    print(
        "  Candidate image IDs:    "
        f"{result['candidate_ids']:,}"
    )

    print(
        "  Processed images:       "
        f"{result['processed']:,}"
    )

    print(
        "  Newly downloaded:       "
        f"{result['newly_downloaded']:,}"
    )

    print(
        "  Auto Cleaned images:    "
        f"{result['accepted']:,}"
    )

    print(
        "  Automatically rejected: "
        f"{result['rejected']:,}"
    )

    if result["rejection_reasons"]:
        print("  Rejection reasons:")

        for reason, count in sorted(
            result[
                "rejection_reasons"
            ].items()
        ):
            print(
                f"    {reason}: "
                f"{count:,}"
            )

    print()


print("Total")

print(
    "  Existing Auto Cleaned images: "
    f"{existing_cleaned_count:,}"
)

print(
    "  Newly Auto Cleaned images:    "
    f"{newly_accepted_total:,}"
)

print(
    "  Automatically rejected:       "
    f"{newly_rejected_total:,}"
)

print(
    "  Current Auto Cleaned images:   "
    f"{final_cleaned_count:,}"
)

print(
    "  Automatic target:             "
    f"{UNKNOWN_TEST_AUTOMATIC_TARGET:,}"
)

print(
    "  Final target:                 "
    f"{UNKNOWN_TEST_FINAL_TARGET:,}"
)

print(
    "  Manual-cleaning buffer:       "
    f"{UNKNOWN_TEST_BUFFER:,}"
)

print()


if (
    final_cleaned_count
    >= UNKNOWN_TEST_AUTOMATIC_TARGET
):
    print(
        "The Unknown Test automatic-cleaning "
        "target was reached."
    )

else:
    print(
        "The Unknown Test automatic-cleaning "
        "target was not reached."
    )

    print(
        "Automatically cleaned images "
        "still needed: "
        f"{UNKNOWN_TEST_AUTOMATIC_TARGET - final_cleaned_count:,}"
    )


print()

print(
    "Auto Cleaned Unknown Test images saved in: "
    f"{show_path(unknown_auto_cleaned_test_dir)}"
)

print(
    "Automatically rejected Unknown Test images "
    "saved in: "
    f"{show_path(unknown_rejected_test_dir)}"
)

print(
    "Unknown Test candidate manifest saved in: "
    f"{show_path(unknown_candidate_manifest_path)}"
)

Unknown Test paths are ready.
Original Open Images directory:       Image Classifier Project\Model Building\Datasets\Original\Test\Open Images V7
Auto Cleaned Unknown Test directory:  Image Classifier Project\Model Building\Datasets\Auto Cleaned\Test\Unknown
Automatically rejected directory:    Image Classifier Project\Model Building\Datasets\Rejected\Automatic\Test\Unknown
Unknown Test automatic target:        3,000
Unknown Test final target:            2,500
Manual-cleaning buffer:               500

Open Images Cat label:  /m/01yrx
Open Images Dog label:  /m/0bt9lr
Open Images Bird label: /m/015p6

Loaded cached Train/Validation exact hashes.
Train/Validation unique hashes loaded: 147,337

Loading existing Auto Cleaned Cat Test hashes...


Hashing existing Cat Test: 100%|██████████| 3000/3000 [00:22<00:00, 133.02image/s]


Existing Cat Test images:       3,000
Existing unique Cat Test hashes: 3,000

Loading existing Auto Cleaned Dog Test hashes...


Hashing existing Dog Test: 100%|██████████| 3000/3000 [00:25<00:00, 117.23image/s]


Existing Dog Test images:       3,000
Existing unique Dog Test hashes: 3,000

Loading existing Auto Cleaned Bird Test hashes...


...

Existing Auto Cleaned Unknown Test images: 0
Existing unique Unknown Test hashes:       0

Preparing Open Images Validation Unknown candidates...
Already downloaded: oidv7-val-annotations-human-imagelabels.csv


Reading Validation Unknown candidate labels: 618184annotation [00:00, 745946.43annotation/s]



Images with positive non-target labels: 41,421
Images with positive Cat, Dog, or Bird labels: 2,637
Non-target images excluded because they also have a target label: 2,636
Eligible Unknown candidate images: 38,785



Open Images Validation Unknown:   8%|▊         | 3000/38785 [1:41:04<20:05:36,  2.02s/image]



Unknown Test automatic cleaning is complete.

Open Images Validation
  Candidate image IDs:    38,785
  Processed images:       3,000
  Newly downloaded:       3,000
  Auto Cleaned images:    3,000
  Automatically rejected: 0

Total
  Existing Auto Cleaned images: 0
  Newly Auto Cleaned images:    3,000
  Automatically rejected:       0
  Current Auto Cleaned images:   3,000
  Automatic target:             3,000
  Final target:                 2,500
  Manual-cleaning buffer:       500

The Unknown Test automatic-cleaning target was reached.

Auto Cleaned Unknown Test images saved in: Image Classifier Project\Model Building\Datasets\Auto Cleaned\Test\Unknown
Automatically rejected Unknown Test images saved in: Image Classifier Project\Model Building\Datasets\Rejected\Automatic\Test\Unknown
Unknown Test candidate manifest saved in: Image Classifier Project\Model Building\Datasets\Original\Test\Open Images V7\Metadata\unknown_test_candidate_manifest.csv
